# MiniMax H3 Remote Colab Notebook ：Ref2V / I2V / T2V / Fast H3 + VOSR2  

### 透過Quick Tunnel 讓agents (codex、claude code)連接

### H3 生成尺寸（16:9，multiple=32）

| megapixels | Output |
|---:|---:|
| 0.2 | 608 × 352 |
| 0.3 | 736 × 416 |
| 0.4 | 864 × 480 |
| 0.5 | 960 × 544 |
| 0.6 | 1056 × 608 |
| **0.7（預設）** | **1152 × 640** |
| 0.8 | 1216 × 672 |
| 0.9 | 1280 × 736 |
| 0.98 | 1344 × 768 |

手動 ：Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8。  
Remote API：Cell 1 → 2 → 3 → 6 → 10。  
只做既有影片 VOSR2：Cell 1 → 2 → 6 → 9A。


## Cell 1 — 安裝 / 更新 ComfyUI、VOSR2、VideoHelperSuite

第一次會 clone / 更新 ComfyUI，並安裝：

- `ylchen333/ComfyUI-VOSR2`
- `Kosinkadink/ComfyUI-VideoHelperSuite`

VOSR2 的約 7GB model bundle 不會在這格預先下載；第一次真正執行 `VOSR2ModelLoader` 時才由節點自動準備。若 custom node 是在 ComfyUI 已經啟動後才安裝，請重啟 runtime 再從 Cell 1 開始。


In [ ]:
import os, sys, math, json, time, shutil, subprocess, secrets, re
from pathlib import Path

import torch

assert torch.cuda.is_available(), "請先在 Colab：執行階段 → 變更執行階段類型 → 選 GPU。"

COMFY = Path("/content/ComfyUI")
LOG_PATH = Path("/content/comfyui_h3.log")
CUSTOM_NODES = COMFY / "custom_nodes"

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("PyTorch CUDA:", torch.version.cuda)
print("GPU capability:", torch.cuda.get_device_capability(0))
print("Free disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))

if not COMFY.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Comfy-Org/ComfyUI.git",
        str(COMFY),
    ], check=True)
else:
    subprocess.run([
        "git", "-C", str(COMFY), "fetch", "--depth", "1", "origin", "master"
    ], check=True)
    subprocess.run([
        "git", "-C", str(COMFY), "reset", "--hard", "FETCH_HEAD"
    ], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(COMFY / "requirements.txt"),
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "hf_xet", "ipywidgets", "pillow", "requests",
    "fastapi", "uvicorn", "python-multipart",
], check=True)

CUSTOM_NODES.mkdir(parents=True, exist_ok=True)


def _sync_custom_node(repo_url, folder_name):
    target = CUSTOM_NODES / folder_name
    if (target / ".git").is_dir():
        subprocess.run([
            "git", "-C", str(target), "fetch", "--depth", "1", "origin", "main"
        ], check=True)
        subprocess.run([
            "git", "-C", str(target), "reset", "--hard", "FETCH_HEAD"
        ], check=True)
        print("更新 custom node:", folder_name)
    elif target.exists():
        raise RuntimeError(
            f"{target} 已存在但不是 git repo。請先移除/改名後再執行 Cell 1。"
        )
    else:
        subprocess.run([
            "git", "clone", "--depth", "1", repo_url, str(target)
        ], check=True)
        print("安裝 custom node:", folder_name)
    return target


VOSR2_NODE_DIR = _sync_custom_node(
    "https://github.com/ylchen333/ComfyUI-VOSR2.git",
    "ComfyUI-VOSR2",
)
VHS_NODE_DIR = _sync_custom_node(
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "ComfyUI-VideoHelperSuite",
)

# VideoHelperSuite 有自己的 requirements；VOSR2 目前主要依賴 ComfyUI 已有套件。
if (VHS_NODE_DIR / "requirements.txt").is_file():
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(VHS_NODE_DIR / "requirements.txt"),
    ], check=True)

assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "需要 ffmpeg / ffprobe。"

(COMFY / "input").mkdir(parents=True, exist_ok=True)
(COMFY / "output").mkdir(parents=True, exist_ok=True)


def _git_commit(path):
    try:
        return subprocess.run(
            ["git", "-C", str(path), "rev-parse", "HEAD"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
    except Exception:
        return "unknown"


COMFY_COMMIT = _git_commit(COMFY)
VOSR2_NODE_COMMIT = _git_commit(VOSR2_NODE_DIR)
VHS_NODE_COMMIT = _git_commit(VHS_NODE_DIR)

try:
    import importlib.metadata as importlib_metadata
    COMFY_KITCHEN_VERSION = importlib_metadata.version("comfy-kitchen")
except Exception:
    COMFY_KITCHEN_VERSION = "unknown"

SPARSE_KERNEL_AVAILABLE = False
SPARSE_KERNEL_CHECK_ERROR = ""
try:
    import comfy_kitchen as ck
    SPARSE_KERNEL_AVAILABLE = bool(ck.sol_attn_is_available(torch.device("cuda")))
except Exception as exc:
    SPARSE_KERNEL_CHECK_ERROR = repr(exc)

print("ComfyUI:", COMFY)
print("ComfyUI commit:", COMFY_COMMIT)
print("ComfyUI-VOSR2 commit:", VOSR2_NODE_COMMIT)
print("VideoHelperSuite commit:", VHS_NODE_COMMIT)
print("comfy-kitchen:", COMFY_KITCHEN_VERSION)
print("VSA/Sol sparse kernel available:", SPARSE_KERNEL_AVAILABLE)
if not SPARSE_KERNEL_AVAILABLE:
    print("提醒：目前 GPU/runtime 沒有可用 sparse kernel；Fast H3 的 VSA 節點會走 dense fallback。")
    if SPARSE_KERNEL_CHECK_ERROR:
        print("kernel check:", SPARSE_KERNEL_CHECK_ERROR)

print("VOSR2 權重不在 Cell 1 預先下載；第一次真正跑 VOSR2 時，VOSR2ModelLoader 會自動準備模型 bundle。")
print("環境準備完成。")


## Cell 2 — Workflow、模型 / LoRA、Sampling、VOSR2、Prompt

先設定再按 **「套用設定」**。

- Workflow 只決定 Ref2V / I2V / T2V conditioning；非 Fast workflow 的主模型與 LoRA 可以自由組合。
- `Comfy-Org/MiniMax-H3` 的官方 `diffusion_models` / `loras` 會嘗試即時讀取；若 Hugging Face 清單讀取失敗會使用 notebook 內建 fallback 清單。
- Sampler 預設 `res_multistep`，常用選項還有 `euler / er_sde`；Scheduler 預設 `simple`，可選 `beta / normal`，兩者都仍可手動輸入。
- Inline VOSR2 預設關閉。開啟後，目標尺寸預設 1920×1080，保持比例、不裁切，可選長邊或短邊對齊。


In [ ]:
import math
import secrets
from pathlib import Path
from urllib.parse import urlparse, unquote

import ipywidgets as widgets
from IPython.display import display
from huggingface_hub import list_repo_files

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass


# ============================================================
# Repo / model constants
# ============================================================

OFFICIAL_REPO = "Comfy-Org/MiniMax-H3"

TENSTRIP_REPO = "TenStrip/10Eros-Max"
TENSTRIP_MODEL = (
    "10Eros_Max_h3_TURBO-hybrid_beta5_int8.safetensors"
)

FAST_REPO = "FastVideo/FastVideo-FastH3-Comfy"
FAST_MODEL = (
    "fastvideo_fasth3_8step_v2_pruned_int8_convrot.safetensors"
)


FALLBACK_OFFICIAL_DIFFUSION_MODELS = [
    "minimax_h3_fl2va_bf16.safetensors",
    "minimax_h3_fl2va_int8_convrot.safetensors",
    "minimax_h3_fl2va_pruned_bf16.safetensors",
    "minimax_h3_fl2va_pruned_fp8_scaled.safetensors",
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors",
    "minimax_h3_fl2va_pruned_w6a8.safetensors",

    "minimax_h3_ref2va_bf16.safetensors",
    "minimax_h3_ref2va_int8_convrot.safetensors",
    "minimax_h3_ref2va_pruned_bf16.safetensors",
    "minimax_h3_ref2va_pruned_fp8_scaled.safetensors",
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors",
    "minimax_h3_ref2va_pruned_w6a8.safetensors",
]


FALLBACK_OFFICIAL_LORAS = [
    "minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors",
    "minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors",
    "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors",
]


def _repo_model_choices():
    try:
        files = list_repo_files(
            OFFICIAL_REPO,
            repo_type="model",
        )

        diffusion = sorted(
            Path(name).name
            for name in files
            if (
                name.startswith("diffusion_models/")
                and name.lower().endswith(".safetensors")
            )
        )

        loras = sorted(
            Path(name).name
            for name in files
            if (
                name.startswith("loras/")
                and name.lower().endswith(".safetensors")
            )
        )

        if not diffusion:
            diffusion = list(
                FALLBACK_OFFICIAL_DIFFUSION_MODELS
            )

        if not loras:
            loras = list(
                FALLBACK_OFFICIAL_LORAS
            )

        return diffusion, loras

    except Exception as exc:
        print(
            "提醒：無法即時讀取 Comfy-Org/MiniMax-H3 "
            "檔案清單，改用 notebook 內建清單：",
            repr(exc),
        )

        return (
            list(FALLBACK_OFFICIAL_DIFFUSION_MODELS),
            list(FALLBACK_OFFICIAL_LORAS),
        )


(
    OFFICIAL_DIFFUSION_MODELS,
    OFFICIAL_LORAS,
) = _repo_model_choices()


DEFAULT_REF_MODEL = (
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors"
)

DEFAULT_FL_MODEL = (
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors"
)


if DEFAULT_REF_MODEL not in OFFICIAL_DIFFUSION_MODELS:
    DEFAULT_REF_MODEL = OFFICIAL_DIFFUSION_MODELS[0]

if DEFAULT_FL_MODEL not in OFFICIAL_DIFFUSION_MODELS:
    DEFAULT_FL_MODEL = OFFICIAL_DIFFUSION_MODELS[0]


# ============================================================
# Resolution presets
# ============================================================

RESOLUTION_PRESETS = {
    "0.2": (608, 352),
    "0.3": (736, 416),
    "0.4": (864, 480),
    "0.5": (960, 544),
    "0.6": (1056, 608),
    "0.7": (1152, 640),
    "0.8": (1216, 672),
    "0.9": (1280, 736),
    "0.98": (1344, 768),
}


FINAL_SIZE_PRESETS = {
    "720p": (1280, 720),
    "1080p": (1920, 1080),
    "1440p": (2560, 1440),
    "2160p": (3840, 2160),
}


# ============================================================
# Workflow
# ============================================================

workflow_widget = widgets.Dropdown(
    options=[
        (
            "Ref2V（Reference conditioning，預設）",
            "ref2v_native",
        ),
        (
            "I2V / First-Last Frame",
            "i2v_native",
        ),
        (
            "T2V",
            "t2v_native",
        ),
        (
            "Fast H3 I2V / First-Last Frame",
            "fast_i2v",
        ),
        (
            "Fast H3 T2V",
            "fast_t2v",
        ),
    ],
    value="ref2v_native",
    description="Workflow",
    layout=widgets.Layout(width="760px"),
)


# ============================================================
# H3 generation size
# ============================================================

resolution_widget = widgets.Dropdown(
    options=[
        ("0.2 MP — 608×352", "0.2"),
        ("0.3 MP — 736×416", "0.3"),
        ("0.4 MP — 864×480", "0.4"),
        ("0.5 MP — 960×544", "0.5"),
        ("0.6 MP — 1056×608", "0.6"),
        ("0.7 MP — 1152×640（預設）", "0.7"),
        ("0.8 MP — 1216×672", "0.8"),
        ("0.9 MP — 1280×736", "0.9"),
        (
            "0.98 MP — 1344×768（官方 768p）",
            "0.98",
        ),
        (
            "Custom",
            "custom",
        ),
    ],
    value="0.7",
    description="生成尺寸",
    layout=widgets.Layout(width="720px"),
)


width_widget = widgets.IntText(
    value=1152,
    description="Width",
)

height_widget = widgets.IntText(
    value=640,
    description="Height",
)


duration_widget = widgets.FloatSlider(
    value=10.0,
    min=4.0,
    max=15.0,
    step=0.125,
    description="Seconds",
    readout_format=".3f",
    layout=widgets.Layout(width="620px"),
)


# ============================================================
# Sampling
# ============================================================

steps_widget = widgets.IntSlider(
    value=20,
    min=1,
    max=40,
    step=1,
    description="Steps",
    layout=widgets.Layout(width="620px"),
)


seed_mode_widget = widgets.Dropdown(
    options=[
        ("Random（預設）", "random"),
        ("Fixed", "fixed"),
    ],
    value="random",
    description="Seed mode",
)


fixed_seed_widget = widgets.Text(
    value="768914106098754",
    description="Fixed seed",
    layout=widgets.Layout(width="430px"),
)


# Combobox 保留手動輸入。
sampler_widget = widgets.Combobox(
    options=[
        "res_multistep",
        "euler",
        "er_sde",
    ],
    value="res_multistep",
    description="Sampler",
    ensure_option=False,
    layout=widgets.Layout(width="430px"),
)


scheduler_widget = widgets.Combobox(
    options=[
        "simple",
        "beta",
        "normal",
    ],
    value="simple",
    description="Scheduler",
    ensure_option=False,
    layout=widgets.Layout(width="430px"),
)


ref_image_size_widget = widgets.Dropdown(
    options=[
        (
            "match（較快 / Turbo 建議）",
            "match",
        ),
        (
            "max（較強參照細節）",
            "max",
        ),
    ],
    value="match",
    description="Ref image",
)


# ============================================================
# Fast H3 only
# ============================================================

fast_vsa_widget = widgets.Checkbox(
    value=True,
    description=(
        "Fast H3 使用原生 VSA Sparse Attention"
    ),
    indent=False,
    layout=widgets.Layout(width="420px"),
)


fast_vsa_verbose_widget = widgets.Checkbox(
    value=False,
    description="VSA verbose log",
    indent=False,
    layout=widgets.Layout(width="220px"),
)


fast_shift_video_widget = widgets.FloatText(
    value=10.0,
    description="Fast shift V",
)


fast_shift_audio_widget = widgets.FloatText(
    value=3.0,
    description="Fast shift A",
)


low_vram_widget = widgets.Dropdown(
    options=[
        ("auto", "auto"),
        ("on", "on"),
        ("off", "off"),
    ],
    value="auto",
    description="Low VRAM",
)


output_prefix_widget = widgets.Text(
    value="MiniMax_H3_Colab",
    description="Output",
    layout=widgets.Layout(width="520px"),
)


# ============================================================
# Prompt
# ============================================================

prompt_widget = widgets.Textarea(
    value=(
        "Use <Picture 1> as the main identity reference. "
        "Create a cinematic 10-second video with coherent "
        "motion, natural lighting, stable character identity, "
        "and synchronized environmental audio."
    ),
    description="Prompt",
    layout=widgets.Layout(
        width="95%",
        height="280px",
    ),
)


# ============================================================
# Main model
# ============================================================

model_source_widget = widgets.Dropdown(
    options=[
        (
            "Comfy-Org/MiniMax-H3 官方 diffusion_models",
            "official",
        ),
        (
            f"{TENSTRIP_REPO} / {TENSTRIP_MODEL}",
            "tenstrip",
        ),
        (
            "Hugging Face repo（自訂）",
            "hf",
        ),
        (
            "直接下載網址 URL",
            "url",
        ),
        (
            "Local / Google Drive 已存在檔案",
            "local",
        ),
    ],
    value="official",
    description="主模型來源",
    layout=widgets.Layout(width="900px"),
)


official_model_widget = widgets.Dropdown(
    options=OFFICIAL_DIFFUSION_MODELS,
    value=DEFAULT_REF_MODEL,
    description="官方模型",
    layout=widgets.Layout(width="900px"),
)


model_hf_repo_widget = widgets.Text(
    value="",
    placeholder="例如：owner/repo",
    description="HF repo",
    layout=widgets.Layout(width="720px"),
)


model_hf_file_widget = widgets.Text(
    value="",
    placeholder="例如：model.safetensors",
    description="HF file",
    layout=widgets.Layout(width="720px"),
)


model_url_widget = widgets.Text(
    value="",
    placeholder="https://.../model.safetensors",
    description="Model URL",
    layout=widgets.Layout(width="95%"),
)


model_local_path_widget = widgets.Text(
    value="",
    placeholder=(
        "/content/drive/MyDrive/models/model.safetensors"
    ),
    description="Model path",
    layout=widgets.Layout(width="95%"),
)


model_save_name_widget = widgets.Text(
    value="",
    placeholder="留空則自動使用來源檔名",
    description="儲存名稱",
    layout=widgets.Layout(width="720px"),
)


# ============================================================
# LoRA
# ============================================================

lora_source_widget = widgets.Dropdown(
    options=[
        (
            "不使用 LoRA（預設）",
            "none",
        ),
        (
            "Comfy-Org/MiniMax-H3 官方 loras",
            "official",
        ),
        (
            "Hugging Face repo（自訂）",
            "hf",
        ),
        (
            "直接下載網址 URL",
            "url",
        ),
        (
            "Local / Google Drive 已存在檔案",
            "local",
        ),
    ],
    value="none",
    description="LoRA 來源",
    layout=widgets.Layout(width="900px"),
)


official_lora_widget = widgets.Dropdown(
    options=OFFICIAL_LORAS,
    value=OFFICIAL_LORAS[0],
    description="官方 LoRA",
    layout=widgets.Layout(width="900px"),
)


lora_hf_repo_widget = widgets.Text(
    value="",
    placeholder="例如：owner/repo",
    description="HF repo",
    layout=widgets.Layout(width="720px"),
)


lora_hf_file_widget = widgets.Text(
    value="",
    placeholder="例如：lora.safetensors",
    description="HF file",
    layout=widgets.Layout(width="720px"),
)


lora_url_widget = widgets.Text(
    value="",
    placeholder="https://.../lora.safetensors",
    description="LoRA URL",
    layout=widgets.Layout(width="95%"),
)


lora_local_path_widget = widgets.Text(
    value="",
    placeholder=(
        "/content/drive/MyDrive/models/lora.safetensors"
    ),
    description="LoRA path",
    layout=widgets.Layout(width="95%"),
)


lora_save_name_widget = widgets.Text(
    value="",
    placeholder="留空則自動使用來源檔名",
    description="儲存名稱",
    layout=widgets.Layout(width="720px"),
)


lora_strength_widget = widgets.FloatSlider(
    value=1.0,
    min=0.0,
    max=2.0,
    step=0.05,
    description="LoRA strength",
    layout=widgets.Layout(width="520px"),
)


# ============================================================
# ModelSamplingMiniMaxH3 / MiniMaxH3SigmaShift
# ============================================================

h3_shift_enabled_widget = widgets.Checkbox(
    value=True,
    description="套用 ModelSamplingMiniMaxH3",
    indent=False,
    layout=widgets.Layout(width="320px"),
)



h3_shift_video_widget = widgets.FloatText(
    value=6.0,
    description="Video shift",
    layout=widgets.Layout(width="260px"),
)


h3_shift_audio_widget = widgets.FloatText(
    value=3.0,
    description="Audio shift",
    layout=widgets.Layout(width="260px"),
)


h3_shift_note = widgets.HTML(
    value=(
        "<span style='color:#888'>"
        "768p FL2VA Turbo 常用 Video 6 / Audio 3。"
        "Ref2VA Turbo 4-step v0.1 使用 12 / 3。"
        "自訂 LoRA 請依其訓練設定手動調整。"
        "</span>"
    )
)


# ============================================================
# VOSR2
# ============================================================

vosr_mode_widget = widgets.Dropdown(
    options=[
        (
            "不放大（預設）",
            "none",
        ),
        (
            "Inline VOSR2：H3 decode 後直接放大，只輸出放大版",
            "inline",
        ),
    ],
    value="none",
    description="VOSR2",
    layout=widgets.Layout(width="900px"),
)


final_resolution_widget = widgets.Dropdown(
    options=[
        ("1280×720", "720p"),
        (
            "1920×1080（常用）",
            "1080p",
        ),
        ("2560×1440", "1440p"),
        ("3840×2160", "2160p"),
        (
            "Custom",
            "custom",
        ),
    ],
    value="1080p",
    description="目標尺寸",
    layout=widgets.Layout(width="620px"),
)


final_width_widget = widgets.IntText(
    value=1920,
    description="Target W",
)


final_height_widget = widgets.IntText(
    value=1080,
    description="Target H",
)


vosr_fit_side_widget = widgets.Dropdown(
    options=[
        (
            "長邊對齊目標值（保持比例、不裁切）",
            "long",
        ),
        (
            "短邊對齊目標值（保持比例、不裁切）",
            "short",
        ),
    ],
    value="long",
    description="尺寸對齊",
    layout=widgets.Layout(width="720px"),
)


vosr_dtype_widget = widgets.Dropdown(
    options=[
        "default",
        "fp16",
        "bf16",
    ],
    value="bf16",
    description="VOSR dtype",
)


vosr_seed_widget = widgets.IntText(
    value=42,
    description="VOSR seed",
)


vosr_color_widget = widgets.Dropdown(
    options=[
        "wavelet",
        "adain",
        "none",
    ],
    value="wavelet",
    description="Color",
)


vosr_tile_size_widget = widgets.IntText(
    value=512,
    description="Tile",
)


vosr_tile_overlap_widget = widgets.IntText(
    value=64,
    description="Overlap",
)


vosr_vae_tile_size_widget = widgets.IntText(
    value=1024,
    description="VAE tile",
)


vosr_vae_tile_overlap_widget = widgets.IntText(
    value=128,
    description="VAE overlap",
)


vosr_advanced_box = widgets.VBox([
    widgets.HTML(
        "<b>VOSR2 進階設定</b>"
    ),
    widgets.HBox([
        vosr_dtype_widget,
        vosr_seed_widget,
        vosr_color_widget,
    ]),
    widgets.HBox([
        vosr_tile_size_widget,
        vosr_tile_overlap_widget,
        vosr_vae_tile_size_widget,
        vosr_vae_tile_overlap_widget,
    ]),
])


# ============================================================
# Buttons
# ============================================================

apply_button = widgets.Button(
    description="套用設定",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="180px"),
)


settings_output = widgets.Output()


# ============================================================
# Helpers
# ============================================================

def _sync_resolution(change=None):
    if resolution_widget.value in RESOLUTION_PRESETS:
        (
            width_widget.value,
            height_widget.value,
        ) = RESOLUTION_PRESETS[
            resolution_widget.value
        ]


def _sync_final_resolution(change=None):
    if (
        final_resolution_widget.value
        in FINAL_SIZE_PRESETS
    ):
        (
            final_width_widget.value,
            final_height_widget.value,
        ) = FINAL_SIZE_PRESETS[
            final_resolution_widget.value
        ]


def plan_vosr_output(
    source_width,
    source_height,
    target_width,
    target_height,
    fit_side="long",
):
    """
    VOSR2 最終輸出尺寸規劃。

    原則：
    1. 保持原始比例
    2. 不裁切
    3. 不拉伸
    4. long  = 長邊盡量命中使用者設定值
    5. short = 短邊盡量命中使用者設定值
    6. 最終寬高強制為偶數，避免 H.264 / yuv420p / libx264
       在 SaveVideo 時因奇數尺寸失敗
    """

    sw = int(source_width)
    sh = int(source_height)

    tw = int(target_width)
    th = int(target_height)

    if min(
        sw,
        sh,
        tw,
        th,
    ) <= 0:
        raise ValueError(
            "VOSR source/target width/height 必須 > 0。"
        )

    if fit_side not in {
        "long",
        "short",
    }:
        raise ValueError(
            "fit_side 必須是 long 或 short。"
        )

    def even_floor(value):
        """
        轉成不超過原值的最近偶數。

        例如：
        1067 -> 1066
        1920 -> 1920
        """
        value = int(
            math.floor(
                float(value)
            )
        )

        if value < 2:
            return 2

        if value % 2:
            value -= 1

        return max(
            2,
            value,
        )

    def even_target(value):
        """
        使用者若輸入奇數 custom target，
        H.264 無法精確保留奇數，因此往下修成偶數。
        """
        return even_floor(
            value
        )

    target_long = even_target(
        max(
            tw,
            th,
        )
    )

    target_short = even_target(
        min(
            tw,
            th,
        )
    )

    landscape = (
        sw >= sh
    )

    if fit_side == "long":

        if landscape:

            # 長邊 Width 固定
            final_w = (
                target_long
            )

            proportional_h = (
                sh
                * final_w
                / sw
            )

            final_h = even_floor(
                proportional_h
            )

            # ImageScale:

            scale_width = (
                final_w
            )

            scale_height = (
                final_h
            )

        else:

            # 長邊 Height 固定
            final_h = (
                target_long
            )

            proportional_w = (
                sw
                * final_h
                / sh
            )

            final_w = even_floor(
                proportional_w
            )

            scale_width = (
                final_w
            )

            scale_height = (
                final_h
            )

    else:

        if landscape:

            # 短邊 Height 固定
            final_h = (
                target_short
            )

            proportional_w = (
                sw
                * final_h
                / sh
            )

            final_w = even_floor(
                proportional_w
            )

            scale_width = (
                final_w
            )

            scale_height = (
                final_h
            )

        else:

            # 短邊 Width 固定
            final_w = (
                target_short
            )

            proportional_h = (
                sh
                * final_w
                / sw
            )

            final_h = even_floor(
                proportional_h
            )

            scale_width = (
                final_w
            )

            scale_height = (
                final_h
            )


    final_w = even_floor(
        final_w
    )

    final_h = even_floor(
        final_h
    )

    scale_width = final_w
    scale_height = final_h

    # VOSR2 本身只能用整數倍率，
    # 所以先放大到足夠尺寸，
    # 再用 ImageScale 縮回精確的偶數輸出尺寸。
    upscale_factor = max(
        1,
        math.ceil(
            max(
                final_w / sw,
                final_h / sh,
            )
        ),
    )

    return {
        "source_width":
            sw,

        "source_height":
            sh,

        "target_width":
            tw,

        "target_height":
            th,

        "fit_side":
            fit_side,

        "final_width":
            final_w,

        "final_height":
            final_h,

        "scale_width":
            scale_width,

        "scale_height":
            scale_height,

        "upscale_factor":
            upscale_factor,

        "codec_even_dimensions":
            True,
    }


def _set_official_model_for_workflow():
    mode = workflow_widget.value

    if mode == "ref2v_native":
        wanted = DEFAULT_REF_MODEL

    elif mode in {
        "i2v_native",
        "t2v_native",
    }:
        wanted = DEFAULT_FL_MODEL

    else:
        return

    if wanted in OFFICIAL_DIFFUSION_MODELS:
        official_model_widget.value = wanted


def _apply_known_lora_profile():
    """
    已知 LightX2V / Comfy-Org Turbo LoRA profile。

    注意：
    - notebook 的 Sampler 預設仍遵照使用者要求保留 res_multistep。
    - 若要完全照 LightX2V 推薦，可以手動選 euler。
    """

    if (
        workflow_widget.value.startswith("fast_")
        or lora_source_widget.value != "official"
    ):
        return

    name = (
        official_lora_widget.value
        or ""
    )

    if "4step" in name:
        steps_widget.value = 4

    elif "8step" in name:
        steps_widget.value = 8

    scheduler_widget.value = "simple"

    # 使用者要求 notebook 預設仍為 res_multistep。
    sampler_widget.value = "res_multistep"

    # Ref2VA Turbo v0.1 是 12 / 3。
    if (
        "ref2v_turbo_4step_v0.1"
        in name
    ):
        h3_shift_video_widget.value = 12.0
        h3_shift_audio_widget.value = 3.0

    else:
        # 768p FL2VA Turbo profile。
        h3_shift_video_widget.value = 6.0
        h3_shift_audio_widget.value = 3.0

    h3_shift_enabled_widget.value = True


def _refresh_ui(change=None):
    mode = workflow_widget.value

    is_fast = mode.startswith(
        "fast_"
    )

    ms = model_source_widget.value
    ls = lora_source_widget.value

    inline = (
        vosr_mode_widget.value
        == "inline"
    )

    ref_image_size_widget.layout.display = (
        ""
        if mode.startswith(
            "ref2v_"
        )
        else "none"
    )

    for w in [
        fast_vsa_widget,
        fast_vsa_verbose_widget,
        fast_shift_video_widget,
        fast_shift_audio_widget,
    ]:
        w.layout.display = (
            ""
            if is_fast
            else "none"
        )

    # Fast H3 使用自己的 checkpoint / shift / VSA。
    model_source_widget.disabled = is_fast
    official_model_widget.disabled = is_fast
    lora_source_widget.disabled = is_fast
    official_lora_widget.disabled = is_fast

    official_model_widget.layout.display = (
        ""
        if (
            ms == "official"
            and not is_fast
        )
        else "none"
    )

    model_hf_repo_widget.layout.display = (
        ""
        if (
            ms == "hf"
            and not is_fast
        )
        else "none"
    )

    model_hf_file_widget.layout.display = (
        ""
        if (
            ms == "hf"
            and not is_fast
        )
        else "none"
    )

    model_url_widget.layout.display = (
        ""
        if (
            ms == "url"
            and not is_fast
        )
        else "none"
    )

    model_local_path_widget.layout.display = (
        ""
        if (
            ms == "local"
            and not is_fast
        )
        else "none"
    )

    model_save_name_widget.layout.display = (
        ""
        if (
            ms in {
                "hf",
                "url",
                "local",
            }
            and not is_fast
        )
        else "none"
    )

    official_lora_widget.layout.display = (
        ""
        if (
            ls == "official"
            and not is_fast
        )
        else "none"
    )

    lora_hf_repo_widget.layout.display = (
        ""
        if (
            ls == "hf"
            and not is_fast
        )
        else "none"
    )

    lora_hf_file_widget.layout.display = (
        ""
        if (
            ls == "hf"
            and not is_fast
        )
        else "none"
    )

    lora_url_widget.layout.display = (
        ""
        if (
            ls == "url"
            and not is_fast
        )
        else "none"
    )

    lora_local_path_widget.layout.display = (
        ""
        if (
            ls == "local"
            and not is_fast
        )
        else "none"
    )

    lora_save_name_widget.layout.display = (
        ""
        if (
            ls in {
                "hf",
                "url",
                "local",
            }
            and not is_fast
        )
        else "none"
    )

    show_lora = (
        ls != "none"
        and not is_fast
    )

    lora_strength_widget.layout.display = (
        ""
        if show_lora
        else "none"
    )

    for w in [
        h3_shift_enabled_widget,
        h3_shift_video_widget,
        h3_shift_audio_widget,
        h3_shift_note,
    ]:
        w.layout.display = (
            ""
            if show_lora
            else "none"
        )

    show_shift_values = (
        show_lora
        and h3_shift_enabled_widget.value
    )

    h3_shift_video_widget.layout.display = (
        ""
        if show_shift_values
        else "none"
    )

    h3_shift_audio_widget.layout.display = (
        ""
        if show_shift_values
        else "none"
    )

    h3_shift_note.layout.display = (
        ""
        if show_shift_values
        else "none"
    )

    for w in [
        final_resolution_widget,
        final_width_widget,
        final_height_widget,
        vosr_fit_side_widget,
        vosr_advanced_box,
    ]:
        w.layout.display = (
            ""
            if inline
            else "none"
        )

    custom_final = (
        inline
        and final_resolution_widget.value
        == "custom"
    )

    final_width_widget.layout.display = (
        ""
        if custom_final
        else "none"
    )

    final_height_widget.layout.display = (
        ""
        if custom_final
        else "none"
    )


def _workflow_defaults(change=None):
    is_fast = (
        workflow_widget.value.startswith(
            "fast_"
        )
    )

    # 保留之前指定的 Notebook 預設。
    sampler_widget.value = (
        "res_multistep"
    )

    scheduler_widget.value = (
        "simple"
    )

    steps_widget.value = (
        8
        if is_fast
        else 20
    )

    if is_fast:
        fast_shift_video_widget.value = 10.0
        fast_shift_audio_widget.value = 3.0

    else:
        _set_official_model_for_workflow()

        # 一般 LoRA shift 初始值。
        if (
            lora_source_widget.value
            != "official"
        ):
            h3_shift_video_widget.value = 6.0
            h3_shift_audio_widget.value = 3.0

    _refresh_ui()


def _lora_changed(change=None):
    if workflow_widget.value.startswith(
        "fast_"
    ):
        _refresh_ui()
        return

    if (
        lora_source_widget.value
        == "official"
    ):
        _apply_known_lora_profile()

    elif (
        lora_source_widget.value
        == "none"
    ):
        h3_shift_enabled_widget.value = False

    else:
        # 自訂 LoRA：預設開啟 6 / 3，
        # 使用者仍可自行取消或修改。
        h3_shift_enabled_widget.value = True

    _refresh_ui()


# ============================================================
# Observers
# ============================================================

resolution_widget.observe(
    _sync_resolution,
    names="value",
)

final_resolution_widget.observe(
    _sync_final_resolution,
    names="value",
)

workflow_widget.observe(
    _workflow_defaults,
    names="value",
)

model_source_widget.observe(
    _refresh_ui,
    names="value",
)

lora_source_widget.observe(
    _lora_changed,
    names="value",
)

official_lora_widget.observe(
    _lora_changed,
    names="value",
)

h3_shift_enabled_widget.observe(
    _refresh_ui,
    names="value",
)

vosr_mode_widget.observe(
    _refresh_ui,
    names="value",
)


# ============================================================
# Apply
# ============================================================

def apply_settings(_=None):

    global INFERENCE_MODE
    global MODE_LABEL

    global WIDTH
    global HEIGHT
    global DURATION_SECONDS
    global STEPS
    global FRAMES
    global ACTUAL_SECONDS

    global SEED
    global SEED_SOURCE
    global SAMPLER_NAME
    global SCHEDULER_NAME
    global DENOISE

    global REFERENCE_IMAGE_SIZE

    global H3_SHIFT_ENABLED
    global H3_SHIFT_VIDEO
    global H3_SHIFT_AUDIO

    global FAST_SHIFT_VIDEO
    global FAST_SHIFT_AUDIO
    global LOW_VRAM_MODE

    global FAST_USE_VSA
    global FAST_ATTENTION_BACKEND
    global FAST_VSA_KEEP_PERCENT
    global FAST_VSA_START_PERCENT
    global FAST_VSA_END_PERCENT
    global FAST_VSA_DENSE_BLOCKS
    global FAST_VSA_MIN_TOKENS
    global FAST_VSA_EXTRA_TOKENS
    global FAST_VSA_SINK_CONDITIONING
    global FAST_VSA_VERBOSE

    global OUTPUT_PREFIX_BASE
    global OUTPUT_PREFIX
    global PROMPT

    global TEXT_ENCODER
    global VIDEO_VAE
    global AUDIO_VAE

    global DIFFUSION
    global LORA
    global USE_TURBO_LORA
    global USE_LORA
    global LORA_STRENGTH

    global MODEL_SOURCE
    global MODEL_HF_REPO
    global MODEL_HF_FILE
    global MODEL_URL
    global MODEL_LOCAL_PATH
    global MODEL_SAVE_NAME

    global LORA_SOURCE
    global LORA_HF_REPO
    global LORA_HF_FILE
    global LORA_URL
    global LORA_LOCAL_PATH
    global LORA_SAVE_NAME

    global VOSR_MODE
    global VOSR_MODEL_BUNDLE
    global VOSR_DTYPE
    global VOSR_SEED
    global VOSR_COLOR_ALIGNMENT
    global VOSR_TILE_SIZE
    global VOSR_TILE_OVERLAP
    global VOSR_VAE_TILE_SIZE
    global VOSR_VAE_TILE_OVERLAP
    global VOSR_TARGET_WIDTH
    global VOSR_TARGET_HEIGHT
    global VOSR_FIT_SIDE
    global VOSR_FINAL_WIDTH
    global VOSR_FINAL_HEIGHT
    global VOSR_SCALE_WIDTH
    global VOSR_SCALE_HEIGHT
    global VOSR_UPSCALE_FACTOR

    # --------------------------------------------------------
    # Basic
    # --------------------------------------------------------

    INFERENCE_MODE = (
        workflow_widget.value
    )

    WIDTH = int(
        width_widget.value
    )

    HEIGHT = int(
        height_widget.value
    )

    DURATION_SECONDS = float(
        duration_widget.value
    )

    STEPS = int(
        steps_widget.value
    )

    SAMPLER_NAME = (
        sampler_widget.value.strip()
    )

    SCHEDULER_NAME = (
        scheduler_widget.value.strip()
    )

    DENOISE = 1.0

    REFERENCE_IMAGE_SIZE = (
        ref_image_size_widget.value
    )

    LOW_VRAM_MODE = (
        low_vram_widget.value
    )

    OUTPUT_PREFIX_BASE = (
        output_prefix_widget.value.strip()
    )

    PROMPT = (
        prompt_widget.value.strip()
    )

    # --------------------------------------------------------
    # H3 shift
    # --------------------------------------------------------

    H3_SHIFT_VIDEO = float(
        h3_shift_video_widget.value
    )

    H3_SHIFT_AUDIO = float(
        h3_shift_audio_widget.value
    )

    assert (
        H3_SHIFT_VIDEO > 0
        and H3_SHIFT_AUDIO > 0
    ), (
        "MiniMax H3 shift 必須 > 0。"
    )

    # --------------------------------------------------------
    # Fast
    # --------------------------------------------------------

    FAST_SHIFT_VIDEO = float(
        fast_shift_video_widget.value
    )

    FAST_SHIFT_AUDIO = float(
        fast_shift_audio_widget.value
    )

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    assert (
        WIDTH > 0
        and HEIGHT > 0
        and WIDTH % 32 == 0
        and HEIGHT % 32 == 0
    ), (
        "生成寬高必須為正數且是 32 的倍數。"
    )

    assert (
        4
        <= DURATION_SECONDS
        <= 15
    ), (
        "影片長度必須是 4–15 秒。"
    )

    assert STEPS > 0, (
        "Steps 必須 > 0。"
    )

    assert SAMPLER_NAME, (
        "Sampler 不能空白。"
    )

    assert SCHEDULER_NAME, (
        "Scheduler 不能空白。"
    )

    assert PROMPT, (
        "Prompt 不能空白。"
    )

    assert OUTPUT_PREFIX_BASE, (
        "Output prefix 不能空白。"
    )

    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    if (
        seed_mode_widget.value
        == "random"
    ):
        SEED = secrets.randbits(
            64
        )

        SEED_SOURCE = "random"

    else:
        SEED = int(
            fixed_seed_widget.value.strip()
        )

        assert (
            0
            <= SEED
            < 2**64
        ), (
            "Fixed seed 必須介於 0 到 2**64-1。"
        )

        SEED_SOURCE = "fixed"

    # --------------------------------------------------------
    # Frame grid
    # --------------------------------------------------------

    requested_frames = max(
        5,
        round(
            DURATION_SECONDS
            * 24
        ),
    )

    FRAMES = (
        requested_frames
        + (
            5
            - requested_frames % 17
        )
        % 17
    )

    ACTUAL_SECONDS = (
        FRAMES / 24
    )

    # --------------------------------------------------------
    # Shared models
    # --------------------------------------------------------

    TEXT_ENCODER = (
        "qwen3vl_32b_"
        "minimax_h3_nvfp4_awq.safetensors"
    )

    VIDEO_VAE = (
        "minimax_h3_video_vae_"
        "int8_convrot.safetensors"
    )

    AUDIO_VAE = (
        "minimax_h3_audio_vae_fp32.safetensors"
    )

    # --------------------------------------------------------
    # Mode label
    # --------------------------------------------------------

    if (
        INFERENCE_MODE
        == "ref2v_native"
    ):
        MODE_LABEL = (
            "Ref2V（Reference conditioning）"
        )

    elif (
        INFERENCE_MODE
        == "i2v_native"
    ):
        MODE_LABEL = (
            "I2V / First-Last Frame"
        )

    elif (
        INFERENCE_MODE
        == "t2v_native"
    ):
        MODE_LABEL = "T2V"

    elif (
        INFERENCE_MODE
        == "fast_i2v"
    ):
        MODE_LABEL = (
            "Fast H3 I2V"
        )

    else:
        MODE_LABEL = (
            "Fast H3 T2V"
        )

    # --------------------------------------------------------
    # Diffusion model
    # --------------------------------------------------------

    if INFERENCE_MODE.startswith(
        "fast_"
    ):
        MODEL_SOURCE = (
            "fastvideo"
        )

        MODEL_HF_REPO = (
            FAST_REPO
        )

        MODEL_HF_FILE = (
            f"diffusion_models/{FAST_MODEL}"
        )

        MODEL_URL = ""
        MODEL_LOCAL_PATH = ""
        MODEL_SAVE_NAME = ""

        DIFFUSION = (
            FAST_MODEL
        )

    else:
        MODEL_SOURCE = (
            model_source_widget.value
        )

        MODEL_HF_REPO = (
            model_hf_repo_widget.value.strip()
        )

        MODEL_HF_FILE = (
            model_hf_file_widget.value.strip()
        )

        MODEL_URL = (
            model_url_widget.value.strip()
        )

        MODEL_LOCAL_PATH = (
            model_local_path_widget.value.strip()
        )

        MODEL_SAVE_NAME = (
            model_save_name_widget.value.strip()
        )

        if (
            MODEL_SOURCE
            == "official"
        ):
            DIFFUSION = (
                official_model_widget.value
            )

            MODEL_HF_REPO = (
                OFFICIAL_REPO
            )

            MODEL_HF_FILE = (
                f"diffusion_models/{DIFFUSION}"
            )

        elif (
            MODEL_SOURCE
            == "tenstrip"
        ):
            DIFFUSION = (
                TENSTRIP_MODEL
            )

            MODEL_HF_REPO = (
                TENSTRIP_REPO
            )

            MODEL_HF_FILE = (
                TENSTRIP_MODEL
            )

        elif (
            MODEL_SOURCE
            == "hf"
        ):
            assert (
                MODEL_HF_REPO
                and MODEL_HF_FILE
            ), (
                "請填寫主模型 HF repo 與 filename。"
            )

            DIFFUSION = (
                MODEL_SAVE_NAME
                or Path(
                    MODEL_HF_FILE
                ).name
            )

        elif (
            MODEL_SOURCE
            == "url"
        ):
            assert MODEL_URL, (
                "請填寫主模型下載 URL。"
            )

            DIFFUSION = (
                MODEL_SAVE_NAME
                or Path(
                    unquote(
                        urlparse(
                            MODEL_URL
                        ).path
                    )
                ).name
            )

        elif (
            MODEL_SOURCE
            == "local"
        ):
            assert MODEL_LOCAL_PATH, (
                "請填寫主模型本地 / Google Drive 路徑。"
            )

            DIFFUSION = (
                MODEL_SAVE_NAME
                or Path(
                    MODEL_LOCAL_PATH
                ).name
            )

        else:
            raise ValueError(
                "未知主模型來源："
                f"{MODEL_SOURCE}"
            )

        DIFFUSION = Path(
            DIFFUSION.strip()
        ).name

    # --------------------------------------------------------
    # LoRA
    # --------------------------------------------------------

    LORA_STRENGTH = float(
        lora_strength_widget.value
    )

    LORA_SOURCE = (
        "none"
        if INFERENCE_MODE.startswith(
            "fast_"
        )
        else lora_source_widget.value
    )

    LORA_HF_REPO = (
        lora_hf_repo_widget.value.strip()
    )

    LORA_HF_FILE = (
        lora_hf_file_widget.value.strip()
    )

    LORA_URL = (
        lora_url_widget.value.strip()
    )

    LORA_LOCAL_PATH = (
        lora_local_path_widget.value.strip()
    )

    LORA_SAVE_NAME = (
        lora_save_name_widget.value.strip()
    )

    if (
        LORA_SOURCE
        == "none"
    ):
        LORA = "N/A"
        USE_LORA = False

    elif (
        LORA_SOURCE
        == "official"
    ):
        LORA = (
            official_lora_widget.value
        )

        LORA_HF_REPO = (
            OFFICIAL_REPO
        )

        LORA_HF_FILE = (
            f"loras/{LORA}"
        )

        USE_LORA = True

    elif (
        LORA_SOURCE
        == "hf"
    ):
        assert (
            LORA_HF_REPO
            and LORA_HF_FILE
        ), (
            "請填寫 LoRA HF repo 與 filename。"
        )

        LORA = (
            LORA_SAVE_NAME
            or Path(
                LORA_HF_FILE
            ).name
        )

        USE_LORA = True

    elif (
        LORA_SOURCE
        == "url"
    ):
        assert LORA_URL, (
            "請填寫 LoRA 下載 URL。"
        )

        LORA = (
            LORA_SAVE_NAME
            or Path(
                unquote(
                    urlparse(
                        LORA_URL
                    ).path
                )
            ).name
        )

        USE_LORA = True

    elif (
        LORA_SOURCE
        == "local"
    ):
        assert LORA_LOCAL_PATH, (
            "請填寫 LoRA 本地 / Google Drive 路徑。"
        )

        LORA = (
            LORA_SAVE_NAME
            or Path(
                LORA_LOCAL_PATH
            ).name
        )

        USE_LORA = True

    else:
        raise ValueError(
            "未知 LoRA 來源："
            f"{LORA_SOURCE}"
        )

    if USE_LORA:
        LORA = Path(
            LORA.strip()
        ).name

    # 舊 Cell 相容。
    USE_TURBO_LORA = (
        USE_LORA
    )

    H3_SHIFT_ENABLED = bool(
        USE_LORA
        and not INFERENCE_MODE.startswith(
            "fast_"
        )
        and h3_shift_enabled_widget.value
    )

    # --------------------------------------------------------
    # Fast H3
    # --------------------------------------------------------

    FAST_USE_VSA = (
        bool(
            fast_vsa_widget.value
        )
        if INFERENCE_MODE.startswith(
            "fast_"
        )
        else False
    )

    FAST_ATTENTION_BACKEND = (
        "comfy kitchen attention"
    )

    FAST_VSA_KEEP_PERCENT = 10.0
    FAST_VSA_START_PERCENT = 0.2
    FAST_VSA_END_PERCENT = 1.0
    FAST_VSA_DENSE_BLOCKS = ""
    FAST_VSA_MIN_TOKENS = 12288
    FAST_VSA_EXTRA_TOKENS = 256

    FAST_VSA_SINK_CONDITIONING = (
        "exact_kv_and_rows"
    )

    FAST_VSA_VERBOSE = bool(
        fast_vsa_verbose_widget.value
    )

    # --------------------------------------------------------
    # VOSR2
    # --------------------------------------------------------

    VOSR_MODE = (
        vosr_mode_widget.value
    )

    VOSR_MODEL_BUNDLE = (
        "VOSR2"
    )

    VOSR_DTYPE = (
        vosr_dtype_widget.value
    )

    VOSR_SEED = int(
        vosr_seed_widget.value
    )

    VOSR_COLOR_ALIGNMENT = (
        vosr_color_widget.value
    )

    VOSR_TILE_SIZE = int(
        vosr_tile_size_widget.value
    )

    VOSR_TILE_OVERLAP = int(
        vosr_tile_overlap_widget.value
    )

    VOSR_VAE_TILE_SIZE = int(
        vosr_vae_tile_size_widget.value
    )

    VOSR_VAE_TILE_OVERLAP = int(
        vosr_vae_tile_overlap_widget.value
    )

    VOSR_TARGET_WIDTH = int(
        final_width_widget.value
    )

    VOSR_TARGET_HEIGHT = int(
        final_height_widget.value
    )

    VOSR_FIT_SIDE = (
        vosr_fit_side_widget.value
    )

    assert VOSR_SEED >= 0

    assert (
        VOSR_TILE_SIZE >= 0
        and VOSR_TILE_OVERLAP >= 0
    )

    assert (
        VOSR_VAE_TILE_SIZE >= 0
        and VOSR_VAE_TILE_OVERLAP >= 0
    )

    if VOSR_TILE_SIZE > 0:
        assert (
            VOSR_TILE_OVERLAP
            < VOSR_TILE_SIZE
        ), (
            "VOSR tile overlap 必須小於 tile size。"
        )

    if VOSR_VAE_TILE_SIZE > 0:
        assert (
            VOSR_VAE_TILE_OVERLAP
            < VOSR_VAE_TILE_SIZE
        ), (
            "VOSR VAE overlap 必須小於 VAE tile size。"
        )

    plan = plan_vosr_output(
        WIDTH,
        HEIGHT,
        VOSR_TARGET_WIDTH,
        VOSR_TARGET_HEIGHT,
        VOSR_FIT_SIDE,
    )

    VOSR_FINAL_WIDTH = (
        plan[
            "final_width"
        ]
    )

    VOSR_FINAL_HEIGHT = (
        plan[
            "final_height"
        ]
    )

    VOSR_SCALE_WIDTH = (
        plan[
            "scale_width"
        ]
    )

    VOSR_SCALE_HEIGHT = (
        plan[
            "scale_height"
        ]
    )

    VOSR_UPSCALE_FACTOR = (
        plan[
            "upscale_factor"
        ]
    )

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    OUTPUT_PREFIX = (
        f"video/"
        f"{OUTPUT_PREFIX_BASE}_"
        f"{INFERENCE_MODE}_"
        f"seed{SEED}"
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    with settings_output:

        settings_output.clear_output()

        print(
            "設定已套用"
        )

        print(
            "模式:",
            MODE_LABEL,
        )

        print(
            "生成尺寸:",
            f"{WIDTH}x{HEIGHT}",
        )

        print(
            "實際排程:",
            f"{FRAMES} frames / "
            f"{ACTUAL_SECONDS:.3f} sec @ 24fps",
        )

        print(
            "Sampling:",
            f"{SAMPLER_NAME} / "
            f"{SCHEDULER_NAME} / "
            f"{STEPS} steps",
        )

        print(
            "Seed:",
            SEED,
            f"({SEED_SOURCE})",
        )

        print(
            "主模型來源:",
            MODEL_SOURCE,
        )

        print(
            "Diffusion:",
            DIFFUSION,
        )

        print(
            "LoRA:",
            (
                LORA
                if USE_LORA
                else "None"
            ),
        )

        if USE_LORA:
            print(
                "LoRA strength:",
                LORA_STRENGTH,
            )

            print(
                "ModelSamplingMiniMaxH3:",
                (
                    "enabled"
                    if H3_SHIFT_ENABLED
                    else "disabled"
                ),
            )

            if H3_SHIFT_ENABLED:
                print(
                    "H3 shift:",
                    H3_SHIFT_VIDEO,
                    H3_SHIFT_AUDIO,
                )

        if INFERENCE_MODE.startswith(
            "ref2v_"
        ):
            print(
                "Ref image size:",
                REFERENCE_IMAGE_SIZE,
            )

        if INFERENCE_MODE.startswith(
            "fast_"
        ):
            print(
                "Fast sigma shift:",
                FAST_SHIFT_VIDEO,
                FAST_SHIFT_AUDIO,
            )

            print(
                "Native VSA:",
                FAST_USE_VSA,
            )

        print(
            "VOSR2 mode:",
            VOSR_MODE,
        )

        if (
            VOSR_MODE
            == "inline"
        ):
            print(
                "VOSR2 final:",
                (
                    f"{VOSR_FINAL_WIDTH}"
                    f"x"
                    f"{VOSR_FINAL_HEIGHT}"
                ),
                (
                    f"(target "
                    f"{VOSR_TARGET_WIDTH}"
                    f"x"
                    f"{VOSR_TARGET_HEIGHT}, "
                    f"fit={VOSR_FIT_SIDE}, "
                    f"internal "
                    f"{VOSR_UPSCALE_FACTOR}x)"
                ),
            )

            print(
                "提醒：Inline VOSR2 "
                "只輸出放大版，"
                "不另外保留原始解析度 MP4。"
            )


apply_button.on_click(
    apply_settings
)


# ============================================================
# Initial UI state
# ============================================================

_sync_resolution()
_sync_final_resolution()
_workflow_defaults()
_refresh_ui()


ui = widgets.VBox([
    workflow_widget,

    resolution_widget,

    widgets.HBox([
        width_widget,
        height_widget,
    ]),

    duration_widget,

    steps_widget,

    widgets.HBox([
        seed_mode_widget,
        fixed_seed_widget,
    ]),

    widgets.HBox([
        sampler_widget,
        scheduler_widget,
    ]),

    ref_image_size_widget,

    widgets.HBox([
        fast_vsa_widget,
        fast_vsa_verbose_widget,
    ]),

    widgets.HBox([
        fast_shift_video_widget,
        fast_shift_audio_widget,
    ]),

    widgets.HBox([
        low_vram_widget,
        output_prefix_widget,
    ]),

    widgets.HTML(
        "<hr><b>"
        "主模型（與 Ref2V / I2V / T2V conditioning 分離）"
        "</b>"
    ),

    model_source_widget,
    official_model_widget,
    model_hf_repo_widget,
    model_hf_file_widget,
    model_url_widget,
    model_local_path_widget,
    model_save_name_widget,

    widgets.HTML(
        "<hr><b>"
        "LoRA（非 Fast workflow 可任意搭配）"
        "</b>"
    ),

    lora_source_widget,
    official_lora_widget,
    lora_hf_repo_widget,
    lora_hf_file_widget,
    lora_url_widget,
    lora_local_path_widget,
    lora_save_name_widget,

    lora_strength_widget,

    h3_shift_enabled_widget,

    widgets.HBox([
        h3_shift_video_widget,
        h3_shift_audio_widget,
    ]),

    h3_shift_note,

    widgets.HTML(
        "<hr><b>VOSR2 最終輸出</b>"
    ),

    vosr_mode_widget,
    final_resolution_widget,

    widgets.HBox([
        final_width_widget,
        final_height_widget,
    ]),

    vosr_fit_side_widget,
    vosr_advanced_box,

    widgets.HTML("<hr>"),

    prompt_widget,
    apply_button,
    settings_output,
])


display(
    ui
)

apply_settings()

## Cell 3 — 準備目前選定的主模型與 LoRA

只下載 / 連結目前 Cell 2 真正選中的模型，並重用共用的 text encoder / video VAE / audio VAE。

官方 LoRA 直接從 `Comfy-Org/MiniMax-H3/loras` 下載。  
`10Eros_Max_h3_TURBO-hybrid_beta5_int8.safetensors`由`TenStrip/10Eros-Max`下載。

VOSR2 權重由 `VOSR2ModelLoader` 第一次執行時自動準備，不在這格下載。


In [ ]:
from pathlib import Path
from urllib.parse import urlparse, unquote
import os
import shutil
import subprocess

from huggingface_hub import hf_hub_download

assert "INFERENCE_MODE" in globals(), "請先執行 Cell 2 並按『套用設定』。"

MODELS_ROOT = COMFY / "models"
DIFFUSION_DIR = MODELS_ROOT / "diffusion_models"
LORA_DIR = MODELS_ROOT / "loras"
TEXT_ENCODER_DIR = MODELS_ROOT / "text_encoders"
VAE_DIR = MODELS_ROOT / "vae"

for p in [DIFFUSION_DIR, LORA_DIR, TEXT_ENCODER_DIR, VAE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

BASE_REPO = "Comfy-Org/MiniMax-H3"
FAST_REPO = "FastVideo/FastVideo-FastH3-Comfy"
TENSTRIP_REPO = "TenStrip/10Eros-Max"
TENSTRIP_MODEL = "10Eros_Max_h3_TURBO-hybrid_beta5_int8.safetensors"


def readable_size(path):
    size = Path(path).stat().st_size
    if size >= 2**30:
        return f"{size / 2**30:.2f} GiB"
    return f"{size / 2**20:.1f} MiB"


def safe_filename(name):
    name = Path(name).name.strip()
    if not name:
        raise ValueError("無法判斷檔名。")
    if not name.lower().endswith((".safetensors", ".ckpt", ".pt", ".pth", ".bin")):
        print("提醒：檔名沒有常見模型副檔名：", name)
    return name


def filename_from_url(url):
    return safe_filename(unquote(Path(urlparse(url).path).name))


def link_existing_file(source, target):
    source = Path(source).expanduser()
    target = Path(target)
    assert source.is_file() and source.stat().st_size > 0, f"來源檔案不存在或是空的：{source}"
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.is_file() and target.stat().st_size > 0:
        print("重用：", target.name, readable_size(target))
        return target

    if target.exists() or target.is_symlink():
        target.unlink()

    try:
        target.symlink_to(source.resolve())
        print("建立 symlink：", source, "->", target)
    except Exception:
        try:
            os.link(source, target)
            print("建立 hard link：", target)
        except Exception:
            print("無法 link，改用複製：", target)
            shutil.copy2(source, target)
    return target


def download_hf_to_target(repo_id, filename, target):
    target = Path(target)
    if target.is_file() and target.stat().st_size > 0:
        print("重用：", target.name, readable_size(target))
        return target

    print("\nHugging Face 下載：")
    print(" repo:", repo_id)
    print(" file:", filename)
    cached = Path(hf_hub_download(repo_id=repo_id, filename=filename))
    return link_existing_file(cached, target)


def download_url_to_target(url, target):
    target = Path(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and target.stat().st_size > 0:
        print("重用：", target.name, readable_size(target))
        return target

    print("\nDirect URL 下載：", url, "->", target)
    subprocess.run([
        "curl", "-L", "--fail", "--retry", "5", "--retry-delay", "5",
        "-C", "-", "-o", str(target), url,
    ], check=True)
    assert target.is_file() and target.stat().st_size > 0, f"下載完成後找不到檔案：{target}"
    return target


# 共用 text encoder / VAEs
download_hf_to_target(
    BASE_REPO,
    "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    TEXT_ENCODER_DIR / TEXT_ENCODER,
)
download_hf_to_target(
    BASE_REPO,
    "vae/minimax_h3_video_vae_int8_convrot.safetensors",
    VAE_DIR / VIDEO_VAE,
)
download_hf_to_target(
    BASE_REPO,
    "vae/minimax_h3_audio_vae_fp32.safetensors",
    VAE_DIR / AUDIO_VAE,
)


def install_main_model():
    global DIFFUSION

    if MODEL_SOURCE in {"official", "fastvideo", "tenstrip", "hf"}:
        repo_id = MODEL_HF_REPO
        filename = MODEL_HF_FILE
        if not repo_id or not filename:
            raise ValueError(f"{MODEL_SOURCE} 主模型缺少 repo/filename。請重新執行 Cell 2。")
        DIFFUSION = safe_filename(MODEL_SAVE_NAME or Path(filename).name)
        return download_hf_to_target(repo_id, filename, DIFFUSION_DIR / DIFFUSION)

    if MODEL_SOURCE == "url":
        DIFFUSION = safe_filename(MODEL_SAVE_NAME or filename_from_url(MODEL_URL))
        return download_url_to_target(MODEL_URL, DIFFUSION_DIR / DIFFUSION)

    if MODEL_SOURCE == "local":
        source = Path(MODEL_LOCAL_PATH).expanduser()
        DIFFUSION = safe_filename(MODEL_SAVE_NAME or source.name)
        return link_existing_file(source, DIFFUSION_DIR / DIFFUSION)

    raise ValueError(f"未知主模型來源：{MODEL_SOURCE}")


def install_lora():
    global LORA, USE_LORA, USE_TURBO_LORA

    if not USE_LORA or LORA_SOURCE == "none":
        LORA = "N/A"
        USE_LORA = False
        USE_TURBO_LORA = False
        return None

    if INFERENCE_MODE.startswith("fast_"):
        raise ValueError("Fast H3 維持 FastVSA 專用 graph，目前不套用 LoraLoaderModelOnly。")

    if LORA_SOURCE in {"official", "hf"}:
        if not LORA_HF_REPO or not LORA_HF_FILE:
            raise ValueError("LoRA 缺少 repo/filename。請重新執行 Cell 2。")
        LORA = safe_filename(LORA_SAVE_NAME or Path(LORA_HF_FILE).name)
        result = download_hf_to_target(LORA_HF_REPO, LORA_HF_FILE, LORA_DIR / LORA)
    elif LORA_SOURCE == "url":
        LORA = safe_filename(LORA_SAVE_NAME or filename_from_url(LORA_URL))
        result = download_url_to_target(LORA_URL, LORA_DIR / LORA)
    elif LORA_SOURCE == "local":
        source = Path(LORA_LOCAL_PATH).expanduser()
        LORA = safe_filename(LORA_SAVE_NAME or source.name)
        result = link_existing_file(source, LORA_DIR / LORA)
    else:
        raise ValueError(f"未知 LoRA 來源：{LORA_SOURCE}")

    USE_LORA = True
    USE_TURBO_LORA = True
    return result


main_model_path = install_main_model()
lora_path = install_lora()

assert Path(main_model_path).name == DIFFUSION, (
    f"主模型狀態不一致：{DIFFUSION} / {Path(main_model_path).name}"
)
if USE_LORA:
    assert lora_path is not None and Path(lora_path).name == LORA, "LoRA 狀態與下載結果不一致。"

print("\n" + "=" * 72)
print("模型準備完成")
print("=" * 72)
print("主模型來源：", MODEL_SOURCE)
print("主模型：", DIFFUSION)
print("主模型位置：", main_model_path)
print("LoRA 來源：", LORA_SOURCE)
print("LoRA：", LORA if USE_LORA else "None")
if lora_path is not None:
    print("LoRA 位置：", lora_path)
    print("LoRA strength：", LORA_STRENGTH)
print("Sampling：", f"{SAMPLER_NAME} / {SCHEDULER_NAME} / {STEPS} steps")
if VOSR_MODE == "inline":
    print(
        "VOSR2：inline；第一次執行 VOSR2ModelLoader 時會自動下載/準備 VOSR2 bundle，"
        "此下載不在 Cell 3 預先執行。"
    )
print("=" * 72)


## Cell 4 — 上傳參照 / 首尾幀素材

- **Ref2V**：Picture 1–9、Video 1–3、Audio 1–3。Prompt 可用 `<Picture 1>`、`<Video 1>`、`<Audio 1>`。
- **I2V**：First frame 必填，Last frame 選填。
- **T2V**：不需要素材。

Ref2V 同類素材請從 1 開始連續使用，不要中間跳號。


In [ ]:

from google.colab import files as colab_files
from io import BytesIO
from PIL import Image
from pathlib import Path

MEDIA_READY = False

PICTURE_MAP = {}
VIDEO_MAP = {}
AUDIO_MAP = {}

FIRST_FRAME_INFO = None
LAST_FRAME_INFO = None

input_dir = COMFY / "input"

input_dir.mkdir(
    parents=True,
    exist_ok=True,
)


def _upload_one(label):
    """
    使用 Colab 原生上傳器。
    一次只收一個檔案。
    """

    print(
        f"\n請上傳 {label}"
        "（一次只選 1 個檔案）"
    )

    uploaded = colab_files.upload()

    if not uploaded:
        raise RuntimeError(
            f"{label} 沒有收到檔案，"
            "請重新執行 Cell 4。"
        )

    if len(uploaded) != 1:
        raise RuntimeError(
            f"{label} 請一次只選 1 個檔案；"
            f"目前收到 {len(uploaded)} 個。"
        )

    original_name, payload = (
        next(iter(uploaded.items()))
    )

    payload = bytes(payload)

    if not payload:
        raise RuntimeError(
            f"{label} 上傳內容是空的。"
        )

    return (
        original_name,
        payload,
    )


def _save_image(
    item,
    out_name,
):

    original_name, payload = item

    out_path = (
        input_dir
        / out_name
    )

    with Image.open(
        BytesIO(payload)
    ) as im:

        rgb = im.convert("RGB")

        rgb.save(
            out_path,
            format="PNG",
        )

        size = rgb.size

    if (
        not out_path.is_file()
        or out_path.stat().st_size == 0
    ):
        raise RuntimeError(
            f"圖片寫入失敗：{out_path}"
        )

    return {
        "filename": out_name,
        "original_name": original_name,
        "path": str(out_path),
        "size": size,
    }


def _save_binary(
    item,
    out_name,
):

    original_name, payload = item

    out_path = (
        input_dir
        / out_name
    )

    out_path.write_bytes(
        payload
    )

    if (
        not out_path.is_file()
        or out_path.stat().st_size == 0
    ):
        raise RuntimeError(
            f"檔案寫入失敗：{out_path}"
        )

    return {
        "filename": out_name,
        "original_name": original_name,
        "path": str(out_path),
        "bytes": out_path.stat().st_size,
    }


def _ask_count(
    label,
    default,
    maximum,
):

    while True:

        raw = input(
            f"{label} 要上傳幾個？"
            f"0–{maximum} "
            f"[預設 {default}]："
        ).strip()

        if raw == "":
            return default

        try:
            value = int(raw)

        except ValueError:
            print(
                "請輸入整數。"
            )
            continue

        if 0 <= value <= maximum:
            return value

        print(
            f"請輸入 0–{maximum}。"
        )


# =========================================================
# T2V
# =========================================================

if INFERENCE_MODE in {
    "t2v_native",
    "fast_t2v",
}:

    print(
        "目前是 T2V；"
        "不需要上傳素材。"
    )

    MEDIA_READY = True


# =========================================================
# I2V
# =========================================================

elif INFERENCE_MODE in {
    "i2v_native",
    "fast_i2v",
}:

    first_item = _upload_one(
        "First frame"
    )

    FIRST_FRAME_INFO = _save_image(
        first_item,
        "h3_first_frame.png",
    )

    print(
        "First frame:",
        FIRST_FRAME_INFO[
            "original_name"
        ],
        "->",
        FIRST_FRAME_INFO[
            "filename"
        ],
        FIRST_FRAME_INFO[
            "size"
        ],
    )

    use_last = (
        input(
            "要上傳 Last frame 嗎？"
            "[y/N]："
        )
        .strip()
        .lower()
        in {"y", "yes"}
    )

    if use_last:

        last_item = _upload_one(
            "Last frame"
        )

        LAST_FRAME_INFO = _save_image(
            last_item,
            "h3_last_frame.png",
        )

        print(
            "Last frame:",
            LAST_FRAME_INFO[
                "original_name"
            ],
            "->",
            LAST_FRAME_INFO[
                "filename"
            ],
            LAST_FRAME_INFO[
                "size"
            ],
        )

    print(
        "\n素材已實際寫入 "
        "ComfyUI/input。"
        "接著執行 Cell 5 驗證。"
    )


# =========================================================
# Ref2V
# =========================================================

else:

    # 預設 1 張 Picture，
    # 因為預設 Prompt 會用 <Picture 1>。
    picture_count = _ask_count(
        "Picture",
        default=1,
        maximum=9,
    )

    video_count = _ask_count(
        "Video",
        default=0,
        maximum=3,
    )

    audio_count = _ask_count(
        "Audio",
        default=0,
        maximum=3,
    )

    if (
        picture_count
        + video_count
        + audio_count
        == 0
    ):
        raise RuntimeError(
            "Ref2V 至少需要一個 "
            "Picture、Video 或 Audio 參照。"
        )


    # -----------------------------------------------------
    # Picture
    # -----------------------------------------------------

    for i in range(
        1,
        picture_count + 1,
    ):

        item = _upload_one(
            f"Picture {i}"
        )

        info = _save_image(
            item,
            f"h3_picture_{i:02d}.png",
        )

        PICTURE_MAP[i] = info

        print(
            f"<Picture {i}>:",
            info["original_name"],
            "->",
            info["filename"],
            info["size"],
        )


    # -----------------------------------------------------
    # Video
    # -----------------------------------------------------

    for i in range(
        1,
        video_count + 1,
    ):

        item = _upload_one(
            f"Video {i}"
        )

        original_name = item[0]

        suffix = (
            Path(original_name)
            .suffix
            .lower()
            or ".mp4"
        )

        info = _save_binary(
            item,
            f"h3_video_{i:02d}{suffix}",
        )

        VIDEO_MAP[i] = info

        print(
            f"<Video {i}>:",
            info["original_name"],
            "->",
            info["filename"],
            f"({info['bytes'] / 2**20:.1f} MiB)",
        )


    # -----------------------------------------------------
    # Audio
    # -----------------------------------------------------

    for i in range(
        1,
        audio_count + 1,
    ):

        item = _upload_one(
            f"Audio {i}"
        )

        original_name = item[0]

        suffix = (
            Path(original_name)
            .suffix
            .lower()
            or ".wav"
        )

        info = _save_binary(
            item,
            f"h3_audio_{i:02d}{suffix}",
        )

        AUDIO_MAP[i] = info

        print(
            f"<Audio {i}>:",
            info["original_name"],
            "->",
            info["filename"],
            f"({info['bytes'] / 2**20:.1f} MiB)",
        )


    print(
        "\n素材已實際寫入 "
        "ComfyUI/input。"
        "接著執行 Cell 5 驗證。"
    )


## Cell 5 — 整理素材並檢查 Prompt

把素材存入 `/content/ComfyUI/input` 並做基本驗證。只換素材時通常重跑 **Cell 4 → 5 → 7 → 8**。


In [ ]:
from pathlib import Path
import json
import re
import subprocess

from PIL import Image


def _assert_file(
    info,
    label,
):

    path = Path(
        info["path"]
    )

    assert path.is_file(), \
        f"{label} 找不到檔案：{path}"

    assert path.stat().st_size > 0, \
        f"{label} 檔案是空的：{path}"

    return path


def _probe_has_stream(
    path,
    stream_type,
):

    p = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",

            "-show_entries",
            "stream=codec_type",

            "-of",
            "json",

            str(path),
        ],
        capture_output=True,
        text=True,
    )

    if p.returncode != 0:
        return (
            False,
            p.stderr.strip(),
        )

    try:

        stream_types = {
            s.get("codec_type")
            for s
            in json.loads(
                p.stdout
            ).get(
                "streams",
                [],
            )
        }

    except Exception as exc:

        return (
            False,
            repr(exc),
        )

    return (
        stream_type in stream_types,
        stream_types,
    )


MEDIA_READY = False


# =========================================================
# T2V
# =========================================================

if INFERENCE_MODE in {
    "t2v_native",
    "fast_t2v",
}:

    MEDIA_READY = True

    print(
        "T2V：不需要素材，"
        "檢查完成。"
    )


# =========================================================
# I2V
# =========================================================

elif INFERENCE_MODE in {
    "i2v_native",
    "fast_i2v",
}:

    assert FIRST_FRAME_INFO is not None, \
        "First frame 尚未上傳；請重新執行 Cell 4。"

    first_path = _assert_file(
        FIRST_FRAME_INFO,
        "First frame",
    )

    with Image.open(
        first_path
    ) as im:
        im.verify()

    print(
        "First frame OK:",
        first_path.name,
    )

    if LAST_FRAME_INFO is not None:

        last_path = _assert_file(
            LAST_FRAME_INFO,
            "Last frame",
        )

        with Image.open(
            last_path
        ) as im:
            im.verify()

        print(
            "Last frame OK:",
            last_path.name,
        )

    MEDIA_READY = True

    print(
        "I2V 素材驗證完成。"
    )


# =========================================================
# Ref2V
# =========================================================

else:

    assert (
        PICTURE_MAP
        or VIDEO_MAP
        or AUDIO_MAP
    ), (
        "Ref2V 沒有收到任何素材；"
        "請重新執行 Cell 4。"
    )


    # -----------------------------------------------------
    # Picture
    # -----------------------------------------------------

    for i, info in sorted(
        PICTURE_MAP.items()
    ):

        path = _assert_file(
            info,
            f"Picture {i}",
        )

        with Image.open(
            path
        ) as im:
            im.verify()

        print(
            f"Picture {i} OK:",
            path.name,
        )


    # -----------------------------------------------------
    # Video
    # -----------------------------------------------------

    for i, info in sorted(
        VIDEO_MAP.items()
    ):

        path = _assert_file(
            info,
            f"Video {i}",
        )

        ok, detail = _probe_has_stream(
            path,
            "video",
        )

        assert ok, (
            f"Video {i} "
            f"無法辨識出 video stream："
            f"{detail}"
        )

        print(
            f"Video {i} OK:",
            path.name,
        )


    # -----------------------------------------------------
    # Audio
    # -----------------------------------------------------

    for i, info in sorted(
        AUDIO_MAP.items()
    ):

        path = _assert_file(
            info,
            f"Audio {i}",
        )

        ok, detail = _probe_has_stream(
            path,
            "audio",
        )

        assert ok, (
            f"Audio {i} "
            f"無法辨識出 audio stream："
            f"{detail}"
        )

        print(
            f"Audio {i} OK:",
            path.name,
        )


    # -----------------------------------------------------
    # Prompt tag 檢查
    # -----------------------------------------------------

    for kind, mapping in [
        (
            "Picture",
            PICTURE_MAP,
        ),
        (
            "Video",
            VIDEO_MAP,
        ),
        (
            "Audio",
            AUDIO_MAP,
        ),
    ]:

        refs = sorted({
            int(x)
            for x in re.findall(
                fr"<{kind}\s+(\d+)>",
                PROMPT,
            )
        })

        missing = [
            n
            for n in refs
            if n not in mapping
        ]

        assert not missing, (
            f"Prompt 使用不存在的 "
            f"{kind} 槽位："
            f"{missing}"
        )

        unused = [
            n
            for n in mapping
            if n not in refs
        ]

        if unused:
            print(
                f"提醒：已上傳但 Prompt "
                f"尚未引用的 {kind}："
                f"{unused}"
            )


    MEDIA_READY = True

    print(
        "Ref2V 素材整理與驗證完成。"
    )


## Cell 6 — 啟動 ComfyUI 並檢查 H3 / VOSR2 / VHS 節點

除了 H3 原生節點之外，這格也會確認 `VOSR2ModelLoader`、`VOSR2Upscale`、`VHS_LoadVideo`、`ImageScale` 已可用。Remote API 模式不需要先跑 Cell 4 / 5。


In [ ]:
import requests


# 手動模式會由 Cell 4 / 5 建立；
# Remote API 不必先有素材。
MEDIA_READY = bool(
    globals().get(
        "MEDIA_READY",
        False,
    )
)

PICTURE_MAP = globals().get(
    "PICTURE_MAP",
    {},
)

VIDEO_MAP = globals().get(
    "VIDEO_MAP",
    {},
)

AUDIO_MAP = globals().get(
    "AUDIO_MAP",
    {},
)

FIRST_FRAME_INFO = globals().get(
    "FIRST_FRAME_INFO",
    None,
)

LAST_FRAME_INFO = globals().get(
    "LAST_FRAME_INFO",
    None,
)


PORT = 8188

BASE_URL = (
    f"http://127.0.0.1:{PORT}"
)


try:
    requests.get(
        BASE_URL
        + "/system_stats",
        timeout=2,
    ).raise_for_status()

    print(
        "使用已啟動的 ComfyUI"
    )

except requests.RequestException:

    LOG_FILE = open(
        LOG_PATH,
        "a",
    )

    launch = [
        sys.executable,
        str(
            COMFY / "main.py"
        ),
        "--listen",
        "127.0.0.1",
        "--port",
        str(PORT),
        "--disable-auto-launch",
    ]

    if (
        LOW_VRAM_MODE
        == "on"
        or (
            LOW_VRAM_MODE
            == "auto"
            and torch.cuda
            .get_device_properties(0)
            .total_memory
            < 48 * 2**30
        )
    ):
        launch.append(
            "--lowvram"
        )

    COMFY_PROCESS = subprocess.Popen(
        launch,
        cwd=COMFY,
        stdout=LOG_FILE,
        stderr=subprocess.STDOUT,
    )

    for _ in range(
        180
    ):
        if (
            COMFY_PROCESS.poll()
            is not None
        ):
            tail = (
                LOG_PATH.read_text(
                    errors="replace"
                )[-8000:]
                if LOG_PATH.exists()
                else ""
            )

            raise RuntimeError(
                "ComfyUI 啟動失敗：\n"
                + tail
            )

        try:
            requests.get(
                BASE_URL
                + "/system_stats",
                timeout=2,
            ).raise_for_status()

            break

        except requests.RequestException:
            time.sleep(
                2
            )

    else:
        raise TimeoutError(
            "ComfyUI 啟動逾時；"
            f"查看 {LOG_PATH}"
        )


object_info = requests.get(
    BASE_URL
    + "/object_info",
    timeout=60,
).json()


required = {
    "UNETLoader",
    "CLIPLoader",
    "VAELoader",

    "ModelAttentionBackend",

    "BasicGuider",
    "RandomNoise",
    "KSamplerSelect",
    "BasicScheduler",
    "SamplerCustomAdvanced",

    "VAEDecode",
    "VAEDecodeAudio",

    "CreateVideo",
    "SaveVideo",
    "ImageScale",

    "VOSR2ModelLoader",
    "VOSR2Upscale",

    "VHS_LoadVideo",
}


if INFERENCE_MODE.startswith(
    "ref2v_"
):
    required |= {
        "MiniMaxH3ReferenceToVideo",
        "LoadImage",
    }

    if VIDEO_MAP:
        required |= {
            "LoadVideo",
            "GetVideoComponents",
        }

    if AUDIO_MAP:
        required.add(
            "LoadAudio"
        )

else:
    required.add(
        "MiniMaxH3ImageToVideo"
    )

    if INFERENCE_MODE in {
        "i2v_native",
        "fast_i2v",
    }:
        required.add(
            "LoadImage"
        )


if USE_LORA:
    required.add(
        "LoraLoaderModelOnly"
    )


# UI 顯示名為 ModelSamplingMiniMaxH3；
# API / object_info 的 class key 是 MiniMaxH3SigmaShift。
if H3_SHIFT_ENABLED:
    required.add(
        "MiniMaxH3SigmaShift"
    )


if INFERENCE_MODE.startswith(
    "fast_"
):
    required |= {
        "MiniMaxH3SigmaShift",
        "BlockSparseAttention",
    }


missing = sorted(
    required
    - object_info.keys()
)


assert not missing, (
    f"ComfyUI 缺少必要節點：{missing}。"
    "請重新執行 Cell 1；"
    "若 ComfyUI 在安裝 VOSR2 / VideoHelperSuite 前已啟動，"
    "請重啟 runtime 後從 Cell 1 開始。"
)


def _extract_combo_options(
    spec,
):
    if isinstance(
        spec,
        (
            list,
            tuple,
        ),
    ):
        if (
            spec
            and isinstance(
                spec[0],
                (
                    list,
                    tuple,
                ),
            )
        ):
            return list(
                spec[0]
            )

        for item in spec[1:]:
            if isinstance(
                item,
                dict,
            ):
                for key in (
                    "options",
                    "values",
                    "choices",
                ):
                    values = item.get(
                        key
                    )

                    if isinstance(
                        values,
                        (
                            list,
                            tuple,
                        ),
                    ):
                        return list(
                            values
                        )

    if isinstance(
        spec,
        dict,
    ):
        for key in (
            "options",
            "values",
            "choices",
        ):
            values = spec.get(
                key
            )

            if isinstance(
                values,
                (
                    list,
                    tuple,
                ),
            ):
                return list(
                    values
                )

    return None


ks_spec = (
    object_info[
        "KSamplerSelect"
    ][
        "input"
    ][
        "required"
    ][
        "sampler_name"
    ]
)


available_samplers = (
    _extract_combo_options(
        ks_spec
    )
)


if (
    available_samplers
    is not None
    and SAMPLER_NAME
    not in available_samplers
):
    raise ValueError(
        f"Sampler '{SAMPLER_NAME}' 不存在。"
        f"可用值包含："
        f"{available_samplers[:50]}"
    )


sched_spec = (
    object_info[
        "BasicScheduler"
    ][
        "input"
    ][
        "required"
    ][
        "scheduler"
    ]
)


available_schedulers = (
    _extract_combo_options(
        sched_spec
    )
)


if (
    available_schedulers
    is not None
    and SCHEDULER_NAME
    not in available_schedulers
):
    raise ValueError(
        f"Scheduler '{SCHEDULER_NAME}' 不存在。"
        f"可用值：{available_schedulers}"
    )


if H3_SHIFT_ENABLED:
    print(
        "ModelSamplingMiniMaxH3:",
        f"Video {H3_SHIFT_VIDEO} / "
        f"Audio {H3_SHIFT_AUDIO}",
    )


if INFERENCE_MODE.startswith(
    "fast_"
):
    print(
        "FastH3 attention backend:",
        FAST_ATTENTION_BACKEND,
    )

    print(
        "FastH3 native VSA enabled:",
        FAST_USE_VSA,
    )

    print(
        "comfy-kitchen version:",
        COMFY_KITCHEN_VERSION,
    )

    print(
        "Sparse kernel available:",
        SPARSE_KERNEL_AVAILABLE,
    )

    if (
        FAST_USE_VSA
        and not SPARSE_KERNEL_AVAILABLE
    ):
        print(
            "⚠️ BlockSparseAttention 會存在，"
            "但目前會走 dense fallback。"
        )


print(
    "VOSR2 / VideoHelperSuite 節點檢查通過。"
)

print(
    "ComfyUI 必要節點檢查通過。"
)

## Cell 7 — 建立 H3 API Graph 並送出生成

In [35]:
assert (
    "object_info"
    in globals()
), (
    "請先執行 Cell 6。"
)


assert MEDIA_READY, (
    "素材尚未準備完成。"
)


# ============================================================
# Model file checks
# ============================================================

_manual_model_path = (
    COMFY
    / "models"
    / "diffusion_models"
    / DIFFUSION
)


if not _manual_model_path.is_file():
    raise FileNotFoundError(
        "Cell 7 找不到目前選定主模型："
        f"{_manual_model_path}\n"
        "若你剛更換主模型，請先執行 Cell 3；"
        "若只改 Prompt / sampling / shift，"
        "Cell 2 → 7 可直接使用已存在模型。"
    )


if USE_LORA:

    _manual_lora_path = (
        COMFY
        / "models"
        / "loras"
        / LORA
    )

    if not _manual_lora_path.is_file():
        raise FileNotFoundError(
            "Cell 7 找不到目前選定 LoRA："
            f"{_manual_lora_path}\n"
            "請先執行 Cell 3 準備 LoRA。"
        )


# ============================================================
# Graph helpers
# ============================================================

graph = {}

_node_counter = 0


def add_node(
    class_type,
    inputs,
):
    global _node_counter

    _node_counter += 1

    node_id = str(
        _node_counter
    )

    graph[
        node_id
    ] = {
        "class_type":
            class_type,

        "inputs":
            inputs,
    }

    return node_id


# ============================================================
# Model / CLIP / VAEs
# ============================================================

unet_id = add_node(
    "UNETLoader",
    {
        "unet_name":
            DIFFUSION,

        "weight_dtype":
            "default",
    },
)


clip_id = add_node(
    "CLIPLoader",
    {
        "clip_name":
            TEXT_ENCODER,

        "type":
            "minimax",

        "device":
            "default",
    },
)


video_vae_id = add_node(
    "VAELoader",
    {
        "vae_name":
            VIDEO_VAE,
    },
)


audio_vae_id = add_node(
    "VAELoader",
    {
        "vae_name":
            AUDIO_VAE,
    },
)


# ============================================================
# Sampling model
# ============================================================

if INFERENCE_MODE.startswith(
    "fast_"
):
    # FastVideo 自己的 shift 設定。
    sigma_id = add_node(
        "MiniMaxH3SigmaShift",
        {
            "model":
                [
                    unet_id,
                    0,
                ],

            "shift_video":
                FAST_SHIFT_VIDEO,

            "shift_audio":
                FAST_SHIFT_AUDIO,
        },
    )

    backend_id = add_node(
        "ModelAttentionBackend",
        {
            "model":
                [
                    sigma_id,
                    0,
                ],

            "attention":
                FAST_ATTENTION_BACKEND,
        },
    )

    if FAST_USE_VSA:

        sparse_id = add_node(
            "BlockSparseAttention",
            {
                "model":
                    [
                        backend_id,
                        0,
                    ],

                "selection":
                    "vsa",

                "selection.keep_percent":
                    FAST_VSA_KEEP_PERCENT,

                "start_percent":
                    FAST_VSA_START_PERCENT,

                "end_percent":
                    FAST_VSA_END_PERCENT,

                "dense_blocks":
                    FAST_VSA_DENSE_BLOCKS,

                "min_tokens":
                    FAST_VSA_MIN_TOKENS,

                "extra_tokens":
                    FAST_VSA_EXTRA_TOKENS,

                "sink_conditioning":
                    FAST_VSA_SINK_CONDITIONING,

                "verbose":
                    FAST_VSA_VERBOSE,
            },
        )

        sampling_model = [
            sparse_id,
            0,
        ]

    else:
        sampling_model = [
            backend_id,
            0,
        ]


else:

    model_source = [
        unet_id,
        0,
    ]

    # --------------------------------------------------------
    # LoRA
    # --------------------------------------------------------

    if USE_LORA:

        lora_id = add_node(
            "LoraLoaderModelOnly",
            {
                "model":
                    model_source,

                "lora_name":
                    LORA,

                "strength_model":
                    LORA_STRENGTH,
            },
        )

        model_source = [
            lora_id,
            0,
        ]

    # --------------------------------------------------------
    # ModelSamplingMiniMaxH3
    #
    # UI 名稱：
    #   ModelSamplingMiniMaxH3
    #
    # API class_type：
    #   MiniMaxH3SigmaShift
    # --------------------------------------------------------

    if H3_SHIFT_ENABLED:

        shift_id = add_node(
            "MiniMaxH3SigmaShift",
            {
                "model":
                    model_source,

                "shift_video":
                    H3_SHIFT_VIDEO,

                "shift_audio":
                    H3_SHIFT_AUDIO,
            },
        )

        model_source = [
            shift_id,
            0,
        ]

    # --------------------------------------------------------
    # Attention
    # --------------------------------------------------------

    backend_id = add_node(
        "ModelAttentionBackend",
        {
            "model":
                model_source,

            "attention":
                "comfy kitchen attention",
        },
    )

    sampling_model = [
        backend_id,
        0,
    ]


# ============================================================
# Conditioning
# ============================================================

if INFERENCE_MODE.startswith(
    "ref2v_"
):

    ref_inputs = {
        "clip":
            [
                clip_id,
                0,
            ],

        "vae":
            [
                video_vae_id,
                0,
            ],

        "audio_vae":
            [
                audio_vae_id,
                0,
            ],

        "prompt":
            PROMPT,

        "width":
            WIDTH,

        "height":
            HEIGHT,

        "length":
            FRAMES,

        "ref_image_size":
            REFERENCE_IMAGE_SIZE,
    }

    # Picture
    for (
        n,
        item,
    ) in PICTURE_MAP.items():

        load_id = add_node(
            "LoadImage",
            {
                "image":
                    item[
                        "filename"
                    ],
            },
        )

        ref_inputs[
            (
                "ref_images."
                f"ref_image_{n - 1}"
            )
        ] = [
            load_id,
            0,
        ]

    # Video + embedded audio
    for (
        n,
        item,
    ) in VIDEO_MAP.items():

        load_id = add_node(
            "LoadVideo",
            {
                "file":
                    item[
                        "filename"
                    ],
            },
        )

        split_id = add_node(
            "GetVideoComponents",
            {
                "video":
                    [
                        load_id,
                        0,
                    ],
            },
        )

        ref_inputs[
            (
                "ref_videos."
                f"ref_video_{n - 1}"
            )
        ] = [
            split_id,
            0,
        ]

        ref_inputs[
            (
                "ref_video_audios."
                f"ref_video_audio_{n - 1}"
            )
        ] = [
            split_id,
            1,
        ]

    # Standalone audio
    for (
        n,
        item,
    ) in AUDIO_MAP.items():

        load_id = add_node(
            "LoadAudio",
            {
                "audio":
                    item[
                        "filename"
                    ],
            },
        )

        ref_inputs[
            (
                "ref_audios."
                f"ref_audio_{n - 1}"
            )
        ] = [
            load_id,
            0,
        ]

    conditioning_id = add_node(
        "MiniMaxH3ReferenceToVideo",
        ref_inputs,
    )


else:

    i2v_inputs = {
        "clip":
            [
                clip_id,
                0,
            ],

        "vae":
            [
                video_vae_id,
                0,
            ],

        "prompt":
            PROMPT,

        "width":
            WIDTH,

        "height":
            HEIGHT,

        "length":
            FRAMES,
    }

    if INFERENCE_MODE in {
        "i2v_native",
        "fast_i2v",
    }:

        first_id = add_node(
            "LoadImage",
            {
                "image":
                    FIRST_FRAME_INFO[
                        "filename"
                    ],
            },
        )

        i2v_inputs[
            "first_frame"
        ] = [
            first_id,
            0,
        ]

        if (
            LAST_FRAME_INFO
            is not None
        ):

            last_id = add_node(
                "LoadImage",
                {
                    "image":
                        LAST_FRAME_INFO[
                            "filename"
                        ],
                },
            )

            i2v_inputs[
                "last_frame"
            ] = [
                last_id,
                0,
            ]

    conditioning_id = add_node(
        "MiniMaxH3ImageToVideo",
        i2v_inputs,
    )


# ============================================================
# Sampling
# ============================================================

guider_id = add_node(
    "BasicGuider",
    {
        "model":
            sampling_model,

        "conditioning":
            [
                conditioning_id,
                0,
            ],
    },
)


noise_id = add_node(
    "RandomNoise",
    {
        "noise_seed":
            SEED,
    },
)


sampler_id = add_node(
    "KSamplerSelect",
    {
        "sampler_name":
            SAMPLER_NAME,
    },
)


scheduler_id = add_node(
    "BasicScheduler",
    {
        "model":
            sampling_model,

        "scheduler":
            SCHEDULER_NAME,

        "steps":
            STEPS,

        "denoise":
            DENOISE,
    },
)


sample_id = add_node(
    "SamplerCustomAdvanced",
    {
        "noise":
            [
                noise_id,
                0,
            ],

        "guider":
            [
                guider_id,
                0,
            ],

        "sampler":
            [
                sampler_id,
                0,
            ],

        "sigmas":
            [
                scheduler_id,
                0,
            ],

        "latent_image":
            [
                conditioning_id,
                1,
            ],
    },
)


# ============================================================
# Decode
# ============================================================

video_decode_id = add_node(
    "VAEDecode",
    {
        "samples":
            [
                sample_id,
                0,
            ],

        "vae":
            [
                video_vae_id,
                0,
            ],
    },
)


audio_decode_id = add_node(
    "VAEDecodeAudio",
    {
        "samples":
            [
                sample_id,
                0,
            ],

        "vae":
            [
                audio_vae_id,
                0,
            ],
    },
)


# ============================================================
# Inline VOSR2
# ============================================================

video_images_ref = [
    video_decode_id,
    0,
]


if (
    VOSR_MODE
    == "inline"
):

    vosr_loader_id = add_node(
        "VOSR2ModelLoader",
        {
            "model":
                VOSR_MODEL_BUNDLE,

            "dtype":
                VOSR_DTYPE,
        },
    )


    vosr_upscale_id = add_node(
        "VOSR2Upscale",
        {
            "model":
                [
                    vosr_loader_id,
                    0,
                ],

            "image":
                video_images_ref,

            "upscale":
                VOSR_UPSCALE_FACTOR,

            "seed":
                VOSR_SEED,

            "color_alignment":
                VOSR_COLOR_ALIGNMENT,

            "tile_size":
                VOSR_TILE_SIZE,

            "tile_overlap":
                VOSR_TILE_OVERLAP,

            "vae_tile_size":
                VOSR_VAE_TILE_SIZE,

            "vae_tile_overlap":
                VOSR_VAE_TILE_OVERLAP,
        },
    )


    final_scale_id = add_node(
        "ImageScale",
        {
            "image":
                [
                    vosr_upscale_id,
                    0,
                ],

            "upscale_method":
                "lanczos",

            "width":
                VOSR_SCALE_WIDTH,

            "height":
                VOSR_SCALE_HEIGHT,

            "crop":
                "disabled",
        },
    )


    video_images_ref = [
        final_scale_id,
        0,
    ]


# ============================================================
# Create / Save
# ============================================================

create_video_id = add_node(
    "CreateVideo",
    {
        "images":
            video_images_ref,

        "audio":
            [
                audio_decode_id,
                0,
            ],

        "fps":
            24,

        "bit_depth":
            8,

        "color_space":
            "sRGB",

        "codec":
            "none",
    },
)


save_inputs = {
    "video":
        [
            create_video_id,
            0,
        ],

    "filename_prefix":
        OUTPUT_PREFIX,

    "format":
        "auto",

    "codec":
        "auto",
}


OUTPUT_NODE_ID = add_node(
    "SaveVideo",
    save_inputs,
)


# ============================================================
# Graph validation
# ============================================================

for (
    node_id,
    node,
) in graph.items():

    class_type = (
        node[
            "class_type"
        ]
    )

    if class_type not in object_info:
        raise ValueError(
            "ComfyUI 缺少節點："
            f"{class_type}"
        )

    spec = (
        object_info[
            class_type
        ][
            "input"
        ]
    )

    known = (
        set(
            spec.get(
                "required",
                {},
            )
        )
        | set(
            spec.get(
                "optional",
                {},
            )
        )
    )

    dynamic = set()

    if (
        class_type
        == "MiniMaxH3ReferenceToVideo"
    ):
        dynamic |= {
            key
            for key
            in node[
                "inputs"
            ]
            if re.fullmatch(
                (
                    r"ref_(?:"
                    r"images\.ref_image|"
                    r"videos\.ref_video|"
                    r"video_audios\.ref_video_audio|"
                    r"audios\.ref_audio"
                    r")_\d+"
                ),
                key,
            )
        }

    elif (
        class_type
        == "BlockSparseAttention"
    ):
        dynamic |= {
            key
            for key
            in node[
                "inputs"
            ]
            if key.startswith(
                "selection."
            )
        }

    unexpected = (
        set(
            node[
                "inputs"
            ]
        )
        - known
        - dynamic
    )

    if unexpected:
        raise ValueError(
            f"{class_type} 的 input schema "
            "可能已變更；未知參數："
            f"{sorted(unexpected)}"
        )


VSA_EXPECTED_ACTIVE = bool(
    INFERENCE_MODE.startswith(
        "fast_"
    )
    and FAST_USE_VSA
    and SPARSE_KERNEL_AVAILABLE
)


RUN_STARTED_WALLCLOCK = (
    time.time()
)


# ============================================================
# Submit
# ============================================================

result = requests.post(
    BASE_URL
    + "/prompt",
    json={
        "prompt":
            graph,

        "client_id":
            "h3-colab-integrated",
    },
    timeout=60,
)


if not result.ok:
    raise RuntimeError(
        "提交失敗 "
        f"HTTP {result.status_code}: "
        f"{result.text}"
    )


job = result.json()


if job.get(
    "node_errors"
):
    raise RuntimeError(
        "ComfyUI 節點驗證失敗：\n"
        + json.dumps(
            job[
                "node_errors"
            ],
            ensure_ascii=False,
            indent=2,
        )
    )


prompt_id = (
    job[
        "prompt_id"
    ]
)


print(
    "已提交推論"
)

print(
    "Prompt ID:",
    prompt_id,
)

print(
    "SEED:",
    SEED,
)

print(
    "模式:",
    MODE_LABEL,
)

print(
    "Model:",
    DIFFUSION,
)

print(
    "LoRA:",
    (
        LORA
        if USE_LORA
        else "None"
    ),
)


if H3_SHIFT_ENABLED:
    print(
        "ModelSamplingMiniMaxH3:",
        f"Video {H3_SHIFT_VIDEO} / "
        f"Audio {H3_SHIFT_AUDIO}",
    )


if (
    VOSR_MODE
    == "inline"
):
    print(
        "Inline VOSR2:",
        (
            f"{WIDTH}x{HEIGHT}"
            " -> "
            f"{VOSR_FINAL_WIDTH}"
            f"x"
            f"{VOSR_FINAL_HEIGHT}"
        ),
        (
            f"(internal "
            f"{VOSR_UPSCALE_FACTOR}x, "
            "no crop)"
        ),
    )

已提交推論
Prompt ID: db166c89-adaa-40c5-b33b-263dd231e6cf
SEED: 12452964462614543489
模式: Ref2V（Reference conditioning）
Model: minimax_h3_ref2va_pruned_int8_convrot.safetensors
LoRA: minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors
ModelSamplingMiniMaxH3: Video 6.0 / Audio 3.0


## Cell 8 — 等待完成、預覽並下載 MP4 + TXT

等待時間會包含 Inline VOSR2 的逐幀放大。TXT 會另外記錄生成尺寸、實際輸出尺寸、主模型 / LoRA、Sampling 與 VOSR2 設定。


In [ ]:
import json
import math
import os
import subprocess
import sys
import threading
import time
from collections import deque

import requests


# ============================================================
# Preconditions
# ============================================================

assert (
    "prompt_id"
    in globals()
), (
    "請先執行 Cell 7 提交 H3 工作。"
)


assert (
    "graph"
    in globals()
), (
    "找不到 Cell 7 建立的 graph；"
    "請重新執行 Cell 7。"
)


assert (
    "OUTPUT_NODE_ID"
    in globals()
), (
    "找不到 OUTPUT_NODE_ID；"
    "請重新執行 Cell 7。"
)


# ============================================================
# websocket-client
# ============================================================

try:
    import websocket

except ImportError:

    print(
        "第一次使用即時進度顯示，"
        "正在安裝 websocket-client..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "websocket-client",
        ],
        check=True,
    )

    import websocket


# ============================================================
# Timeout / reporting
# ============================================================

job_timeout_seconds = float(
    os.environ.get(
        "H3_JOB_TIMEOUT_SECONDS",
        "10800",
    )
)


assert (
    math.isfinite(
        job_timeout_seconds
    )
    and job_timeout_seconds > 0
), (
    "H3_JOB_TIMEOUT_SECONDS 必須是正數。"
)


# 一般狀況每 15 秒印一次。
STATUS_REPORT_INTERVAL = 15.0


# ComfyUI client id。
#
# 目前 Cell 7：
#
# "client_id": "h3-colab-integrated"
#
# 所以這裡必須使用相同 ID，
# 才能收到正在執行中的 progress events。
H3_COMFY_CLIENT_ID = (
    globals().get(
        "H3_COMFY_CLIENT_ID",
        "h3-colab-integrated",
    )
)


# ============================================================
# Node labels
# ============================================================

PRETTY_NODE_NAMES = {
    "UNETLoader":
        "載入 MiniMax H3 主模型",

    "CLIPLoader":
        "載入文字編碼器",

    "VAELoader":
        "載入 VAE",

    "LoraLoaderModelOnly":
        "套用 LoRA",

    "MiniMaxH3SigmaShift":
        "ModelSamplingMiniMaxH3 Shift",

    "ModelAttentionBackend":
        "設定 Attention Backend",

    "BlockSparseAttention":
        "Fast H3 VSA",

    "MiniMaxH3ReferenceToVideo":
        "建立 Ref2V Conditioning",

    "MiniMaxH3ImageToVideo":
        "建立 I2V / T2V Conditioning",

    "BasicGuider":
        "建立 Guider",

    "RandomNoise":
        "建立 Noise",

    "KSamplerSelect":
        "準備 Sampler",

    "BasicScheduler":
        "建立 Sigma Schedule",

    "SamplerCustomAdvanced":
        "MiniMax H3 Sampling",

    "VAEDecode":
        "MiniMax Video VAE Decode",

    "VAEDecodeAudio":
        "MiniMax Audio VAE Decode",

    "VOSR2ModelLoader":
        "載入 VOSR2 模型",

    "VOSR2Upscale":
        "VOSR2 AI 放大",

    "ImageScale":
        "縮放到最終輸出尺寸",

    "CreateVideo":
        "建立影片 / 合併音訊",

    "SaveVideo":
        "H.264 編碼 / 儲存 MP4",
}


# 從 Cell 7 的 graph 建立：
#
# node id -> class_type
# node id -> readable label

NODE_TYPES = {}

NODE_LABELS = {}


for (
    node_id,
    node,
) in graph.items():

    class_type = (
        node.get(
            "class_type",
            "Unknown",
        )
    )

    NODE_TYPES[
        str(
            node_id
        )
    ] = (
        class_type
    )

    NODE_LABELS[
        str(
            node_id
        )
    ] = (
        PRETTY_NODE_NAMES.get(
            class_type,
            class_type,
        )
    )


SAMPLER_NODE_IDS = {
    node_id
    for (
        node_id,
        class_type,
    ) in NODE_TYPES.items()
    if (
        class_type
        == "SamplerCustomAdvanced"
    )
}


VOSR_NODE_IDS = {
    node_id
    for (
        node_id,
        class_type,
    ) in NODE_TYPES.items()
    if (
        class_type
        == "VOSR2Upscale"
    )
}


# ============================================================
# Formatting helpers
# ============================================================

def _format_seconds(
    seconds,
):
    if (
        seconds is None
        or not math.isfinite(
            float(
                seconds
            )
        )
        or seconds < 0
    ):
        return "--:--"

    seconds = int(
        round(
            seconds
        )
    )

    hours, remainder = divmod(
        seconds,
        3600,
    )

    minutes, secs = divmod(
        remainder,
        60,
    )

    if hours:
        return (
            f"{hours:02d}:"
            f"{minutes:02d}:"
            f"{secs:02d}"
        )

    return (
        f"{minutes:02d}:"
        f"{secs:02d}"
    )


def _gpu_status():
    """
    回傳：
    used MiB
    total MiB
    utilization %
    """

    try:

        p = subprocess.run(
            [
                "nvidia-smi",

                "--query-gpu="
                "memory.used,"
                "memory.total,"
                "utilization.gpu",

                "--format="
                "csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        line = (
            p.stdout
            .strip()
            .splitlines()[0]
        )

        (
            used,
            total,
            util,
        ) = [
            int(
                item.strip()
            )
            for item
            in line.split(",")
        ]

        return {
            "used_mib":
                used,

            "total_mib":
                total,

            "util_percent":
                util,
        }

    except Exception:

        return {
            "used_mib":
                None,

            "total_mib":
                None,

            "util_percent":
                None,
        }


def _stage_prefix(
    class_type,
):
    if (
        class_type
        == "SamplerCustomAdvanced"
    ):
        return "[H3]"

    if (
        class_type
        == "VOSR2Upscale"
    ):
        return "[VOSR2]"

    if (
        class_type
        in {
            "VAEDecode",
            "VAEDecodeAudio",
        }
    ):
        return "[VAE]"

    if (
        class_type
        in {
            "CreateVideo",
            "SaveVideo",
        }
    ):
        return "[VIDEO]"

    return "[ComfyUI]"


def _progress_unit(
    class_type,
):
    if (
        class_type
        == "SamplerCustomAdvanced"
    ):
        return "steps"

    if (
        class_type
        == "VOSR2Upscale"
    ):
        return "tiles"

    return "units"


# ============================================================
# GPU peak tracking
# ============================================================

peak_gpu_memory_mib = None


def _update_peak_gpu():
    global peak_gpu_memory_mib

    status = (
        _gpu_status()
    )

    used = (
        status[
            "used_mib"
        ]
    )

    if (
        used is not None
        and (
            peak_gpu_memory_mib
            is None
            or used
            > peak_gpu_memory_mib
        )
    ):
        peak_gpu_memory_mib = (
            used
        )

    return status


_update_peak_gpu()


# ============================================================
# Progress state
# ============================================================

progress_lock = (
    threading.Lock()
)


progress_stop = (
    threading.Event()
)


progress_data = {
    "ws_connected":
        False,

    "ws_error":
        None,

    "node_id":
        None,

    "node_type":
        None,

    "node_label":
        "等待 ComfyUI 工作",

    "node_started":
        None,

    "value":
        None,

    "max":
        None,

    "percent":
        None,

    # 保存最近進度點，
    # 用移動速度計算 ETA。
    "samples":
        deque(
            maxlen=30
        ),
}


def _change_current_node(
    node_id,
):
    """
    切換目前執行節點。

    若 node 真的改變，
    就重設該階段的 progress samples。
    """

    if node_id is None:
        return

    node_id = str(
        node_id
    )

    now = (
        time.monotonic()
    )

    if (
        progress_data[
            "node_id"
        ]
        != node_id
    ):

        progress_data[
            "node_id"
        ] = (
            node_id
        )

        progress_data[
            "node_type"
        ] = (
            NODE_TYPES.get(
                node_id,
                "Unknown",
            )
        )

        progress_data[
            "node_label"
        ] = (
            NODE_LABELS.get(
                node_id,
                (
                    "節點 "
                    f"{node_id}"
                ),
            )
        )

        progress_data[
            "node_started"
        ] = (
            now
        )

        progress_data[
            "value"
        ] = None

        progress_data[
            "max"
        ] = None

        progress_data[
            "percent"
        ] = None

        progress_data[
            "samples"
        ].clear()


def _update_progress(
    node_id,
    value,
    maximum,
):
    node_id = str(
        node_id
    )

    _change_current_node(
        node_id
    )

    try:
        value = float(
            value
        )

        maximum = float(
            maximum
        )

    except Exception:
        return

    now = (
        time.monotonic()
    )

    progress_data[
        "value"
    ] = (
        value
    )

    progress_data[
        "max"
    ] = (
        maximum
    )

    progress_data[
        "percent"
    ] = (
        (
            value
            / maximum
            * 100.0
        )
        if maximum > 0
        else None
    )

    samples = (
        progress_data[
            "samples"
        ]
    )

    # 同一 value 不一直塞。
    if (
        not samples
        or samples[
            -1
        ][1]
        != value
    ):
        samples.append(
            (
                now,
                value,
            )
        )


def _calculate_eta(
    snapshot,
):
    """
    用最近 progress samples 的實際速度估算 ETA。

    比單純：
        elapsed / percent

    更不容易被模型載入時間影響。
    """

    maximum = (
        snapshot.get(
            "max"
        )
    )

    value = (
        snapshot.get(
            "value"
        )
    )

    samples = list(
        snapshot.get(
            "samples",
            [],
        )
    )

    if (
        maximum is None
        or value is None
        or maximum <= 0
        or value >= maximum
        or len(
            samples
        ) < 2
    ):
        return None

    # 優先使用最近最多 10 個有效 sample。
    samples = (
        samples[
            -10:
        ]
    )

    first_t, first_v = (
        samples[
            0
        ]
    )

    last_t, last_v = (
        samples[
            -1
        ]
    )

    delta_time = (
        last_t
        - first_t
    )

    delta_value = (
        last_v
        - first_v
    )

    if (
        delta_time <= 2
        or delta_value <= 0
    ):
        return None

    rate = (
        delta_value
        / delta_time
    )

    if rate <= 0:
        return None

    remaining = (
        maximum
        - value
    )

    return (
        remaining
        / rate
    )


# ============================================================
# WebSocket reader
# ============================================================

def _progress_ws_worker():

    ws = None

    try:

        ws_url = (
            BASE_URL.replace(
                "http://",
                "ws://",
            )
            + "/ws"
            + (
                "?clientId="
                f"{H3_COMFY_CLIENT_ID}"
            )
        )

        # localhost 不經 proxy。
        ws = websocket.create_connection(
            ws_url,
            timeout=5,
            http_proxy_host=None,
            http_proxy_port=None,
        )

        ws.settimeout(
            2
        )

        with progress_lock:
            progress_data[
                "ws_connected"
            ] = True

        while (
            not progress_stop.is_set()
        ):

            try:

                message = (
                    ws.recv()
                )

            except (
                websocket
                .WebSocketTimeoutException
            ):
                continue

            except Exception as exc:

                if (
                    not progress_stop.is_set()
                ):
                    with progress_lock:
                        progress_data[
                            "ws_error"
                        ] = str(
                            exc
                        )

                break

            # Preview frame 是 binary，
            # ETA 不需要它。
            if isinstance(
                message,
                bytes,
            ):
                continue

            try:
                event = (
                    json.loads(
                        message
                    )
                )

            except Exception:
                continue

            event_type = (
                event.get(
                    "type"
                )
            )

            data = (
                event.get(
                    "data",
                    {},
                )
            )

            event_prompt_id = (
                data.get(
                    "prompt_id"
                )
            )

            # 某些 reconnect 的 executing event
            # 沒 prompt_id；
            # 有 prompt_id 時才做過濾。
            if (
                event_prompt_id
                and event_prompt_id
                != prompt_id
            ):
                continue

            # ------------------------------------------------
            # Current node
            # ------------------------------------------------

            if (
                event_type
                == "executing"
            ):

                node_id = (
                    data.get(
                        "node"
                    )
                )

                if (
                    node_id
                    is not None
                ):

                    with progress_lock:
                        _change_current_node(
                            node_id
                        )

            # ------------------------------------------------
            # Modern ComfyUI:
            # progress_state
            # ------------------------------------------------

            elif (
                event_type
                == "progress_state"
            ):

                nodes = (
                    data.get(
                        "nodes",
                        {}
                    )
                )

                # 優先找 state=running。
                running = []

                for (
                    node_key,
                    state,
                ) in nodes.items():

                    if (
                        state.get(
                            "state"
                        )
                        == "running"
                    ):
                        running.append(
                            (
                                node_key,
                                state,
                            )
                        )

                if running:

                    node_key, state = (
                        running[
                            -1
                        ]
                    )

                    actual_node_id = str(
                        state.get(
                            "display_node_id"
                        )
                        or state.get(
                            "real_node_id"
                        )
                        or state.get(
                            "node_id"
                        )
                        or node_key
                    )

                    with progress_lock:

                        _update_progress(
                            actual_node_id,
                            state.get(
                                "value",
                                0,
                            ),
                            state.get(
                                "max",
                                0,
                            ),
                        )

            # ------------------------------------------------
            # Older ComfyUI fallback:
            # progress
            # ------------------------------------------------

            elif (
                event_type
                == "progress"
            ):

                node_id = (
                    data.get(
                        "node"
                    )
                )

                if (
                    node_id
                    is not None
                ):

                    with progress_lock:

                        _update_progress(
                            node_id,
                            data.get(
                                "value",
                                0,
                            ),
                            data.get(
                                "max",
                                0,
                            ),
                        )

    except Exception as exc:

        with progress_lock:
            progress_data[
                "ws_error"
            ] = str(
                exc
            )

    finally:

        if (
            ws is not None
        ):
            try:
                ws.close()

            except Exception:
                pass


progress_thread = (
    threading.Thread(
        target=_progress_ws_worker,
        daemon=True,
    )
)


progress_thread.start()


# ============================================================
# Wait loop
# ============================================================

start_time = (
    time.monotonic()
)


deadline = (
    start_time
    + job_timeout_seconds
)


last_report = 0.0

last_print_node = None

last_print_percent = None

outputs = None


print()
print(
    "=" * 72
)

print(
    "MiniMax H3 工作進度"
)

print(
    "=" * 72
)

print(
    "Prompt ID:",
    prompt_id,
)

print(
    "Model:",
    DIFFUSION,
)

print(
    "Sampling:",
    (
        f"{SAMPLER_NAME} / "
        f"{SCHEDULER_NAME} / "
        f"{STEPS} steps"
    ),
)


if H3_SHIFT_ENABLED:
    print(
        "ModelSamplingMiniMaxH3:",
        (
            f"Video {H3_SHIFT_VIDEO} / "
            f"Audio {H3_SHIFT_AUDIO}"
        ),
    )


if (
    VOSR_MODE
    == "inline"
):
    print(
        "Inline VOSR2:",
        (
            f"{WIDTH}x{HEIGHT}"
            " → "
            f"{VOSR_FINAL_WIDTH}"
            f"x"
            f"{VOSR_FINAL_HEIGHT}"
        ),
    )

    print(
        "注意：H3 Sampling 完成後"
        "還會進入 VOSR2 AI 放大。"
    )


print(
    "進度資訊約每 15 秒更新。"
)

print(
    "=" * 72
)

print()


while (
    time.monotonic()
    < deadline
):

    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------

    gpu = (
        _update_peak_gpu()
    )

    # --------------------------------------------------------
    # History / errors / completion
    # --------------------------------------------------------

    try:

        history = requests.get(
            BASE_URL
            + f"/history/{prompt_id}",
            timeout=30,
        ).json()

    except requests.RequestException:

        history = {}

    if (
        prompt_id
        in history
    ):

        run = (
            history[
                prompt_id
            ]
        )

        status = run.get(
            "status",
            {},
        )

        if (
            status.get(
                "status_str"
            )
            == "error"
        ):

            progress_stop.set()

            raise RuntimeError(
                "推論失敗：\n"
                + json.dumps(
                    status,
                    ensure_ascii=False,
                    indent=2,
                )
                + f"\n日誌：{LOG_PATH}"
            )

        outputs = (
            run.get(
                "outputs",
                {},
            )
        )

        if (
            OUTPUT_NODE_ID
            in outputs
            or status.get(
                "status_str"
            )
            == "success"
        ):
            break

    # --------------------------------------------------------
    # Snapshot websocket progress
    # --------------------------------------------------------

    with progress_lock:

        snapshot = {
            key:
                (
                    deque(
                        value,
                        maxlen=30,
                    )
                    if key
                    == "samples"
                    else value
                )

            for (
                key,
                value,
            ) in progress_data.items()
        }

    now = (
        time.monotonic()
    )

    total_elapsed = (
        now
        - start_time
    )

    node_id = (
        snapshot.get(
            "node_id"
        )
    )

    node_type = (
        snapshot.get(
            "node_type"
        )
        or (
            NODE_TYPES.get(
                str(
                    node_id
                )
            )
            if node_id
            else None
        )
    )

    node_label = (
        snapshot.get(
            "node_label"
        )
        or "等待 ComfyUI"
    )

    percent = (
        snapshot.get(
            "percent"
        )
    )

    # --------------------------------------------------------
    # Decide when to print
    # --------------------------------------------------------

    node_changed = (
        node_id
        and node_id
        != last_print_node
    )

    percent_changed = False

    if (
        percent is not None
    ):

        if (
            last_print_percent
            is None
            or abs(
                percent
                - last_print_percent
            )
            >= 2.0
        ):
            percent_changed = True

    timed_report = (
        now
        - last_report
        >= STATUS_REPORT_INTERVAL
    )

    if (
        node_changed
        or percent_changed
        or timed_report
    ):

        last_report = (
            now
        )

        last_print_node = (
            node_id
        )

        if (
            percent is not None
        ):
            last_print_percent = (
                percent
            )

        else:
            last_print_percent = (
                None
            )

        stage_prefix = (
            _stage_prefix(
                node_type
            )
        )

        eta = (
            _calculate_eta(
                snapshot
            )
        )

        gpu_text = ""

        if (
            gpu[
                "used_mib"
            ]
            is not None
        ):
            gpu_text = (
                " | GPU "
                f"{gpu['used_mib']}"
                "/"
                f"{gpu['total_mib']} MiB"
                " | util "
                f"{gpu['util_percent']}%"
            )

        # ----------------------------------------------------
        # Progress-capable node
        # ----------------------------------------------------

        value = (
            snapshot.get(
                "value"
            )
        )

        maximum = (
            snapshot.get(
                "max"
            )
        )

        if (
            percent is not None
            and maximum is not None
            and maximum > 0
        ):

            unit = (
                _progress_unit(
                    node_type
                )
            )

            print(
                f"{stage_prefix} "
                f"{node_label}"
                " | "
                f"{percent:6.2f}%"
                " | "
                f"{int(value)}/{int(maximum)} "
                f"{unit}"
                " | total elapsed "
                f"{_format_seconds(total_elapsed)}"
                " | ETA "
                f"{_format_seconds(eta)}"
                f"{gpu_text}"
            )

        # ----------------------------------------------------
        # Node without numeric progress
        # ----------------------------------------------------

        else:

            node_elapsed = None

            if (
                snapshot.get(
                    "node_started"
                )
                is not None
            ):
                node_elapsed = (
                    now
                    - snapshot[
                        "node_started"
                    ]
                )

            print(
                f"{stage_prefix} "
                f"{node_label}"
                " | total elapsed "
                f"{_format_seconds(total_elapsed)}"
                " | node elapsed "
                f"{_format_seconds(node_elapsed)}"
                f"{gpu_text}"
            )

        # ----------------------------------------------------
        # Stage ETA warning
        # ----------------------------------------------------

        if (
            node_type
            == "SamplerCustomAdvanced"
            and VOSR_MODE
            == "inline"
        ):
            print(
                "      ↑ 這個 ETA 只代表 H3 Sampling；"
                "完成後還有 Inline VOSR2。"
            )

        elif (
            node_type
            == "VOSR2Upscale"
        ):
            print(
                "      ↑ VOSR2 ETA 完成後仍有"
                "最終 resize / audio merge / MP4 encode。"
            )

        # ----------------------------------------------------
        # WebSocket fallback warning
        # ----------------------------------------------------

        if (
            snapshot.get(
                "ws_error"
            )
            and not snapshot.get(
                "ws_connected"
            )
        ):
            print(
                "      即時 progress WebSocket 無法使用：",
                snapshot[
                    "ws_error"
                ],
            )

            print(
                "      將繼續用 /history 等待；"
                "影片仍可正常生成，"
                "只是無法計算精確 ETA。"
            )

    time.sleep(
        3
    )


else:

    progress_stop.set()

    raise TimeoutError(
        "等待推論超過 "
        f"{job_timeout_seconds:.0f} 秒；"
        f"日誌：{LOG_PATH}"
    )


# ============================================================
# Job complete
# ============================================================

progress_stop.set()


final_gpu = (
    _update_peak_gpu()
)


generation_wait_seconds = (
    time.monotonic()
    - start_time
)


print()
print(
    "[ComfyUI] 所有 graph 節點已完成。"
)

print(
    "總等待時間:",
    _format_seconds(
        generation_wait_seconds
    ),
)

print(
    "正在確認輸出 MP4..."
)


# ============================================================
# Find MP4
# ============================================================

def _find_mp4_infos(
    obj,
    found=None,
):
    if found is None:
        found = []

    if isinstance(
        obj,
        dict,
    ):

        filename = obj.get(
            "filename"
        )

        if (
            isinstance(
                filename,
                str,
            )
            and filename.lower().endswith(
                ".mp4"
            )
        ):
            found.append(
                obj
            )

        for value in obj.values():

            _find_mp4_infos(
                value,
                found,
            )

    elif isinstance(
        obj,
        (
            list,
            tuple,
        ),
    ):

        for value in obj:

            _find_mp4_infos(
                value,
                found,
            )

    return found


candidates = (
    _find_mp4_infos(
        (
            outputs.get(
                OUTPUT_NODE_ID,
                {},
            )
            if outputs
            else {}
        )
    )
)


video_path = None


for info in candidates:

    p = (
        COMFY
        / "output"
        / info.get(
            "subfolder",
            "",
        )
        / info[
            "filename"
        ]
    )

    if (
        p.is_file()
        and p.stat().st_size > 0
    ):
        video_path = (
            p
        )

        break


if (
    video_path
    is None
):

    recent = [
        p
        for p
        in (
            COMFY
            / "output"
        ).rglob(
            "*.mp4"
        )
        if (
            p.is_file()
            and p.stat().st_mtime
            >= RUN_STARTED_WALLCLOCK - 2
        )
    ]

    if recent:

        video_path = max(
            recent,
            key=lambda p:
                p.stat().st_mtime,
        )


if (
    video_path
    is None
):
    raise RuntimeError(
        "任務已完成，但找不到本次 MP4。\n"
        + json.dumps(
            outputs
            or {},
            ensure_ascii=False,
            indent=2,
        )[:12000]
    )


# ============================================================
# Probe output
# ============================================================

probe = subprocess.run(
    [
        "ffprobe",
        "-v",
        "error",

        "-show_entries",
        "stream=codec_type,width,height",

        "-of",
        "json",

        str(
            video_path
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)


probe_json = (
    json.loads(
        probe.stdout
    )
)


streams = {
    x.get(
        "codec_type"
    )
    for x
    in probe_json.get(
        "streams",
        [],
    )
}


assert (
    "video"
    in streams
), (
    "MP4 缺少影片串流："
    f"{streams}"
)


assert (
    "audio"
    in streams
), (
    "MP4 缺少音訊串流："
    f"{streams}"
)


video_stream = next(
    (
        x
        for x
        in probe_json.get(
            "streams",
            [],
        )
        if (
            x.get(
                "codec_type"
            )
            == "video"
        )
    ),
    {},
)


actual_output_width = (
    video_stream.get(
        "width",
        (
            VOSR_FINAL_WIDTH
            if VOSR_MODE
            == "inline"
            else WIDTH
        ),
    )
)


actual_output_height = (
    video_stream.get(
        "height",
        (
            VOSR_FINAL_HEIGHT
            if VOSR_MODE
            == "inline"
            else HEIGHT
        ),
    )
)


# ============================================================
# TXT
# ============================================================

info_txt_path = (
    video_path.with_suffix(
        ".txt"
    )
)


info_lines = [
    "============================================================",
    "MiniMax H3 Generation Information",
    "============================================================",
    "",

    f"seed={SEED}",
    f"seed_source={SEED_SOURCE}",

    f"inference_mode={INFERENCE_MODE}",
    f"mode_label={MODE_LABEL}",

    "native_comfyui_nodes=True",

    f"comfyui_commit={COMFY_COMMIT}",

    (
        "comfy_kitchen_version="
        f"{COMFY_KITCHEN_VERSION}"
    ),

    (
        "vosr2_node_commit="
        f"{globals().get('VOSR2_NODE_COMMIT', 'unknown')}"
    ),

    (
        "videohelpersuite_commit="
        f"{globals().get('VHS_NODE_COMMIT', 'unknown')}"
    ),

    (
        "sparse_kernel_available="
        f"{SPARSE_KERNEL_AVAILABLE}"
    ),

    (
        "generation_wait_seconds="
        f"{generation_wait_seconds:.3f}"
    ),

    (
        "peak_gpu_memory_used_mib_observed="
        f"{peak_gpu_memory_mib if peak_gpu_memory_mib is not None else 'N/A'}"
    ),

    "",
    "============================================================",
    "Models",
    "============================================================",
    "",

    f"diffusion={DIFFUSION}",

    f"model_source={MODEL_SOURCE}",

    (
        "lora="
        f"{LORA if USE_LORA else 'N/A'}"
    ),

    (
        "lora_enabled="
        f"{USE_LORA}"
    ),

    (
        "lora_strength="
        f"{LORA_STRENGTH if USE_LORA else 'N/A'}"
    ),

    (
        "h3_shift_enabled="
        f"{H3_SHIFT_ENABLED}"
    ),

    (
        "h3_shift_video="
        f"{H3_SHIFT_VIDEO if H3_SHIFT_ENABLED else 'N/A'}"
    ),

    (
        "h3_shift_audio="
        f"{H3_SHIFT_AUDIO if H3_SHIFT_ENABLED else 'N/A'}"
    ),

    f"text_encoder={TEXT_ENCODER}",
    f"video_vae={VIDEO_VAE}",
    f"audio_vae={AUDIO_VAE}",

    "",
    "============================================================",
    "Video / Sampling",
    "============================================================",
    "",

    f"generation_width={WIDTH}",
    f"generation_height={HEIGHT}",

    (
        "output_width="
        f"{actual_output_width}"
    ),

    (
        "output_height="
        f"{actual_output_height}"
    ),

    f"frames={FRAMES}",

    (
        "requested_duration_seconds="
        f"{DURATION_SECONDS}"
    ),

    (
        "actual_duration_seconds="
        f"{ACTUAL_SECONDS}"
    ),

    "fps=24",

    f"output_prefix={OUTPUT_PREFIX}",

    f"prompt_id={prompt_id}",

    f"steps={STEPS}",

    f"sampler={SAMPLER_NAME}",

    f"scheduler={SCHEDULER_NAME}",

    f"denoise={DENOISE}",
]


# ============================================================
# Fast H3 info
# ============================================================

if INFERENCE_MODE.startswith(
    "fast_"
):

    info_lines += [
        (
            "fast_shift_video="
            f"{FAST_SHIFT_VIDEO}"
        ),

        (
            "fast_shift_audio="
            f"{FAST_SHIFT_AUDIO}"
        ),

        (
            "fast_attention_backend="
            f"{FAST_ATTENTION_BACKEND}"
        ),

        (
            "fast_use_native_vsa="
            f"{FAST_USE_VSA}"
        ),

        (
            "vsa_expected_active="
            f"{VSA_EXPECTED_ACTIVE}"
        ),

        (
            "vsa_keep_percent="
            f"{FAST_VSA_KEEP_PERCENT if FAST_USE_VSA else 'N/A'}"
        ),

        (
            "vsa_start_percent="
            f"{FAST_VSA_START_PERCENT if FAST_USE_VSA else 'N/A'}"
        ),

        (
            "vsa_end_percent="
            f"{FAST_VSA_END_PERCENT if FAST_USE_VSA else 'N/A'}"
        ),

        (
            "vsa_min_tokens="
            f"{FAST_VSA_MIN_TOKENS if FAST_USE_VSA else 'N/A'}"
        ),

        (
            "vsa_sink_conditioning="
            f"{FAST_VSA_SINK_CONDITIONING if FAST_USE_VSA else 'N/A'}"
        ),
    ]


# ============================================================
# VOSR2 info
# ============================================================

info_lines += [
    "",
    "============================================================",
    "VOSR2",
    "============================================================",
    "",

    f"vosr_mode={VOSR_MODE}",
]


if (
    VOSR_MODE
    == "inline"
):

    info_lines += [
        (
            "vosr_model_bundle="
            f"{VOSR_MODEL_BUNDLE}"
        ),

        (
            "vosr_dtype="
            f"{VOSR_DTYPE}"
        ),

        (
            "vosr_seed="
            f"{VOSR_SEED}"
        ),

        (
            "vosr_color_alignment="
            f"{VOSR_COLOR_ALIGNMENT}"
        ),

        (
            "vosr_target_width="
            f"{VOSR_TARGET_WIDTH}"
        ),

        (
            "vosr_target_height="
            f"{VOSR_TARGET_HEIGHT}"
        ),

        (
            "vosr_fit_side="
            f"{VOSR_FIT_SIDE}"
        ),

        (
            "vosr_internal_upscale="
            f"{VOSR_UPSCALE_FACTOR}"
        ),

        (
            "vosr_final_width="
            f"{VOSR_FINAL_WIDTH}"
        ),

        (
            "vosr_final_height="
            f"{VOSR_FINAL_HEIGHT}"
        ),

        (
            "vosr_tile_size="
            f"{VOSR_TILE_SIZE}"
        ),

        (
            "vosr_tile_overlap="
            f"{VOSR_TILE_OVERLAP}"
        ),

        (
            "vosr_vae_tile_size="
            f"{VOSR_VAE_TILE_SIZE}"
        ),

        (
            "vosr_vae_tile_overlap="
            f"{VOSR_VAE_TILE_OVERLAP}"
        ),

        "inline_original_mp4_saved=False",
    ]


# ============================================================
# Reference mapping
# ============================================================

if INFERENCE_MODE.startswith(
    "ref2v_"
):

    info_lines += [
        (
            "reference_image_size="
            f"{REFERENCE_IMAGE_SIZE}"
        ),

        "",
        "============================================================",
        "Reference Media Mapping",
        "============================================================",
        "",
    ]

    for (
        n,
        item,
    ) in PICTURE_MAP.items():

        info_lines.append(
            (
                f"<Picture {n}>="
                f"{item['original_name']} "
                f"-> {item['filename']}"
            )
        )

    for (
        n,
        item,
    ) in VIDEO_MAP.items():

        info_lines.append(
            (
                f"<Video {n}>="
                f"{item['original_name']} "
                f"-> {item['filename']}"
            )
        )

    for (
        n,
        item,
    ) in AUDIO_MAP.items():

        info_lines.append(
            (
                f"<Audio {n}>="
                f"{item['original_name']} "
                f"-> {item['filename']}"
            )
        )


if INFERENCE_MODE in {
    "i2v_native",
    "fast_i2v",
}:

    info_lines += [
        (
            "first_frame="
            f"{FIRST_FRAME_INFO['original_name'] if FIRST_FRAME_INFO else 'N/A'}"
        ),

        (
            "last_frame="
            f"{LAST_FRAME_INFO['original_name'] if LAST_FRAME_INFO else 'N/A'}"
        ),
    ]


# ============================================================
# Prompt
# ============================================================

info_lines += [
    "",
    "============================================================",
    "PROMPT BEGIN",
    "============================================================",
    "",

    PROMPT,

    "",
    "============================================================",
    "PROMPT END",
    "============================================================",
    "",
]


info_txt_path.write_text(
    "\n".join(
        info_lines
    ),
    encoding="utf-8",
)


# ============================================================
# Final summary
# ============================================================

elapsed = int(
    generation_wait_seconds
)


hours, remainder = divmod(
    elapsed,
    3600,
)


minutes, seconds = divmod(
    remainder,
    60,
)


elapsed_text = (
    (
        f"{hours}小時 "
        f"{minutes}分 "
        f"{seconds}秒"
    )
    if hours
    else (
        f"{minutes}分 "
        f"{seconds}秒"
    )
)


print()
print(
    "=" * 72
)

print(
    "影片生成完成"
)

print(
    "=" * 72
)

print(
    "影片:",
    video_path,
)

print(
    "TXT:",
    info_txt_path,
)

print(
    "輸出尺寸:",
    (
        f"{actual_output_width}"
        f"x"
        f"{actual_output_height}"
    ),
)

print(
    "大小:",
    (
        f"{video_path.stat().st_size / 2**20:.1f} MiB"
    ),
)

print(
    "Streams:",
    sorted(
        streams
    ),
)

print(
    "SEED:",
    SEED,
)


if H3_SHIFT_ENABLED:

    print(
        "ModelSamplingMiniMaxH3:",
        (
            f"Video {H3_SHIFT_VIDEO} / "
            f"Audio {H3_SHIFT_AUDIO}"
        ),
    )


print(
    "總等待時間:",
    elapsed_text,
)


print(
    "觀察到的峰值 GPU memory:",
    (
        f"{peak_gpu_memory_mib} MiB"
        if peak_gpu_memory_mib
        is not None
        else "N/A"
    ),
)


if (
    VOSR_MODE
    == "inline"
):

    print(
        "Inline VOSR2: enabled / "
        "原始解析度 MP4 不另外保留"
    )


print(
    "=" * 72
)


# ============================================================
# Preview / download
# ============================================================

from IPython.display import (
    Video,
    display,
)

from google.colab import files


display(
    Video(
        str(
            video_path
        ),
        embed=True,
    )
)


print(
    "準備下載 MP4..."
)

files.download(
    str(
        video_path
    )
)


print(
    "準備下載 TXT..."
)

files.download(
    str(
        info_txt_path
    )
)

## Cell 9 — 重跑指南

| 想更換的內容 | 建議重跑 |
|---|---|
| 只改 Prompt / seed / 生成尺寸 / 秒數 / steps / sampler / scheduler / LoRA shift | Cell 2 套用 → **7 → 8** |
| 換主模型 / LoRA | Cell 2 套用 → **3 → 7 → 8** |
| Ref2V 換 Picture / Video / Audio | **4 → 5 → 7 → 8** |
| I2V 換 First / Last frame | **4 → 5 → 7 → 8** |
| 切換 conditioning workflow | Cell 2 套用 → **3 → 4 → 5 → 6 → 7 → 8** |
| 開 / 關 Inline VOSR2 或改目標尺寸 | Cell 2 套用 → **7 → 8** |
| 只放大既有影片 | Cell 2 設 VOSR 目標 / advanced → **6 → 9A** |
| 更新 ComfyUI / VOSR2 / VideoHelperSuite | 重跑 Cell 1，然後重啟 runtime |
| Colab runtime 中斷 | 從 **Cell 1** 重跑 |

Ref2V tag 是 1-based（`<Picture 1>`），ComfyUI dynamic input 是 0-based；Cell 7 會自動轉換。


## Cell 9A — VOSR2 放大既有影片（不重新跑 H3）

這格沿用 Cell 2 的「目標尺寸 / 長短邊對齊 / VOSR2 advanced」設定。

若沒有設定環境變數 `H3_VOSR_INPUT_VIDEO`，會跳出 Colab 上傳器；適合先生成原始短片、看過結果後再決定是否花時間放大。


In [ ]:
"""## Cell 9A — VOSR2 放大既有影片（含即時進度 / ETA）

用途：
- 上傳一支已存在的影片
- 不重新執行 MiniMax H3
- VHS_LoadVideo → VOSR2 → resize → CreateVideo → SaveVideo
- 顯示 VOSR2 即時 tile progress、耗時、ETA、GPU VRAM
- 保持比例、不裁切
- 最終尺寸強制偶數，避免 H.264 / libx264 奇數尺寸錯誤
"""

import json
import math
import os
import re
import secrets
import shutil
import subprocess
import sys
import threading
import time
import uuid

from fractions import Fraction
from pathlib import Path

import requests

from google.colab import files as colab_files
from IPython.display import Video, display


# ============================================================
# Preconditions
# ============================================================

assert (
    "object_info" in globals()
    and "BASE_URL" in globals()
), (
    "請先執行 Cell 6。"
)

for node_name in (
    "VHS_LoadVideo",
    "VOSR2ModelLoader",
    "VOSR2Upscale",
    "ImageScale",
    "CreateVideo",
    "SaveVideo",
):
    assert (
        node_name in object_info
    ), (
        f"缺少 {node_name}；"
        "請重跑 Cell 1 並重啟 ComfyUI runtime。"
    )


# ============================================================
# websocket-client
# ============================================================

try:
    import websocket

except ImportError:
    print(
        "第一次使用進度顯示，"
        "正在安裝 websocket-client..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "websocket-client",
        ],
        check=True,
    )

    import websocket


# ============================================================
# Helpers
# ============================================================

def _format_seconds(
    seconds,
):
    if (
        seconds is None
        or not math.isfinite(
            float(seconds)
        )
        or seconds < 0
    ):
        return "--:--"

    seconds = int(
        round(
            seconds
        )
    )

    hours, rem = divmod(
        seconds,
        3600,
    )

    minutes, secs = divmod(
        rem,
        60,
    )

    if hours:
        return (
            f"{hours:02d}:"
            f"{minutes:02d}:"
            f"{secs:02d}"
        )

    return (
        f"{minutes:02d}:"
        f"{secs:02d}"
    )


def _gpu_memory_status():
    """
    回傳：
    used MiB / total MiB
    """

    try:
        p = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu="
                "memory.used,memory.total,"
                "utilization.gpu",
                "--format="
                "csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        line = (
            p.stdout
            .strip()
            .splitlines()[0]
        )

        used, total, util = [
            int(
                x.strip()
            )
            for x
            in line.split(",")
        ]

        return {
            "used_mib":
                used,

            "total_mib":
                total,

            "util_percent":
                util,
        }

    except Exception:
        return {
            "used_mib":
                None,

            "total_mib":
                None,

            "util_percent":
                None,
        }


def _probe_video_for_vosr(
    path,
):
    p = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            (
                "stream="
                "codec_type,"
                "width,"
                "height,"
                "avg_frame_rate,"
                "nb_frames,"
                "duration"
            ),
            "-of",
            "json",
            str(
                path
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    data = json.loads(
        p.stdout
    )

    video_stream = next(
        (
            s
            for s
            in data.get(
                "streams",
                [],
            )
            if (
                s.get(
                    "codec_type"
                )
                == "video"
            )
        ),
        None,
    )

    if not video_stream:
        raise ValueError(
            "輸入檔沒有 video stream。"
        )

    width = int(
        video_stream[
            "width"
        ]
    )

    height = int(
        video_stream[
            "height"
        ]
    )

    rate_text = str(
        video_stream.get(
            "avg_frame_rate"
        )
        or "24/1"
    )

    try:
        fps = float(
            Fraction(
                rate_text
            )
        )

    except Exception:
        fps = 24.0

    if (
        not math.isfinite(
            fps
        )
        or fps <= 0
    ):
        fps = 24.0

    # Frame count
    frame_count = None

    raw_frames = (
        video_stream.get(
            "nb_frames"
        )
    )

    try:
        if (
            raw_frames
            and str(
                raw_frames
            ).upper()
            != "N/A"
        ):
            frame_count = int(
                raw_frames
            )

    except Exception:
        frame_count = None

    # 有些 container 沒有 nb_frames，
    # 就用 duration × fps 估。
    if frame_count is None:
        try:
            duration = float(
                video_stream.get(
                    "duration"
                )
            )

            if (
                math.isfinite(
                    duration
                )
                and duration > 0
            ):
                frame_count = round(
                    duration
                    * fps
                )

        except Exception:
            pass

    has_audio = any(
        (
            s.get(
                "codec_type"
            )
            == "audio"
        )
        for s
        in data.get(
            "streams",
            [],
        )
    )

    return (
        width,
        height,
        fps,
        has_audio,
        frame_count,
    )


def _even_floor(
    value,
):
    """
    H.264 / yuv420p 安全尺寸：
    不超過原值的最近偶數。
    """

    value = int(
        math.floor(
            float(value)
        )
    )

    value = max(
        2,
        value,
    )

    if (
        value % 2
    ):
        value -= 1

    return max(
        2,
        value,
    )


def _make_codec_safe_plan(
    plan,
):
    """
    即使 Cell 2 尚未更新到偶數修正版，
    Cell 9A 也自己再保險一次。
    """

    safe = dict(
        plan
    )

    safe[
        "final_width"
    ] = _even_floor(
        safe[
            "final_width"
        ]
    )

    safe[
        "final_height"
    ] = _even_floor(
        safe[
            "final_height"
        ]
    )

    # 不再讓 ImageScale 用 width=0 / height=0
    # 自己重新算出奇數。
    safe[
        "scale_width"
    ] = safe[
        "final_width"
    ]

    safe[
        "scale_height"
    ] = safe[
        "final_height"
    ]

    return safe


def _find_post_mp4_infos(
    obj,
    found=None,
):
    if found is None:
        found = []

    if isinstance(
        obj,
        dict,
    ):
        filename = obj.get(
            "filename"
        )

        if (
            isinstance(
                filename,
                str,
            )
            and filename.lower().endswith(
                ".mp4"
            )
        ):
            found.append(
                obj
            )

        for value in obj.values():
            _find_post_mp4_infos(
                value,
                found,
            )

    elif isinstance(
        obj,
        (
            list,
            tuple,
        ),
    ):
        for value in obj:
            _find_post_mp4_infos(
                value,
                found,
            )

    return found


# ============================================================
# Input video
# ============================================================

existing_path = (
    os.environ.get(
        "H3_VOSR_INPUT_VIDEO",
        "",
    ).strip()
)


if existing_path:

    source_path = Path(
        existing_path
    ).expanduser()

    if not source_path.is_file():
        raise FileNotFoundError(
            "H3_VOSR_INPUT_VIDEO 不存在："
            f"{source_path}"
        )

    source_original_name = (
        source_path.name
    )

else:

    print(
        "請上傳 1 支要放大的影片。"
    )

    uploaded = (
        colab_files.upload()
    )

    if (
        len(
            uploaded
        )
        != 1
    ):
        raise RuntimeError(
            "請一次只上傳 1 支影片；"
            f"目前收到 {len(uploaded)} 個檔案。"
        )

    (
        source_original_name,
        raw,
    ) = next(
        iter(
            uploaded.items()
        )
    )

    suffix = (
        Path(
            source_original_name
        ).suffix.lower()
        or ".mp4"
    )

    source_path = (
        COMFY
        / "input"
        / (
            "vosr_post_"
            f"{secrets.token_hex(6)}"
            f"{suffix}"
        )
    )

    source_path.write_bytes(
        bytes(
            raw
        )
    )


if (
    not source_path.is_file()
    or source_path.stat().st_size <= 0
):
    raise RuntimeError(
        "輸入影片不存在或是空檔。"
    )


# ============================================================
# Ensure file is inside ComfyUI/input
# ============================================================

try:
    source_rel = str(
        source_path.resolve().relative_to(
            (
                COMFY
                / "input"
            ).resolve()
        )
    ).replace(
        "\\",
        "/",
    )

except ValueError:

    suffix = (
        source_path.suffix.lower()
        or ".mp4"
    )

    copied = (
        COMFY
        / "input"
        / (
            "vosr_post_"
            f"{secrets.token_hex(6)}"
            f"{suffix}"
        )
    )

    shutil.copy2(
        source_path,
        copied,
    )

    source_path = (
        copied
    )

    source_rel = (
        source_path.name
    )


# ============================================================
# Probe
# ============================================================

(
    source_w,
    source_h,
    source_fps,
    source_has_audio,
    source_frame_count,
) = _probe_video_for_vosr(
    source_path
)


plan = plan_vosr_output(
    source_w,
    source_h,
    VOSR_TARGET_WIDTH,
    VOSR_TARGET_HEIGHT,
    VOSR_FIT_SIDE,
)


# 保險：
# 最終影片尺寸永遠是偶數。
plan = _make_codec_safe_plan(
    plan
)


print()
print(
    "=" * 72
)

print(
    "VOSR2 影片放大"
)

print(
    "=" * 72
)

print(
    "來源:",
    source_original_name,
)

print(
    "來源尺寸:",
    f"{source_w}x{source_h}",
)

print(
    "FPS:",
    f"{source_fps:.3f}",
)

print(
    "Audio:",
    source_has_audio,
)

if (
    source_frame_count
    is not None
):
    print(
        "影片影格數:",
        source_frame_count,
    )

    print(
        "影片長度約:",
        (
            f"{source_frame_count / source_fps:.2f} 秒"
        ),
    )


print(
    "VOSR2 輸出:",
    (
        f"{plan['final_width']}"
        f"x"
        f"{plan['final_height']}"
    ),
)

print(
    "目標設定:",
    (
        f"{VOSR_TARGET_WIDTH}"
        f"x"
        f"{VOSR_TARGET_HEIGHT}"
    ),
)

print(
    "尺寸模式:",
    (
        f"fit={VOSR_FIT_SIDE} / "
        "保持比例 / 不裁切"
    ),
)

print(
    "VOSR2 internal upscale:",
    (
        f"{plan['upscale_factor']}x"
    ),
)

print(
    "Tile:",
    (
        f"{VOSR_TILE_SIZE}"
        f" / overlap "
        f"{VOSR_TILE_OVERLAP}"
    ),
)

print(
    "VAE tile:",
    (
        f"{VOSR_VAE_TILE_SIZE}"
        f" / overlap "
        f"{VOSR_VAE_TILE_OVERLAP}"
    ),
)

print(
    "=" * 72
)


# ============================================================
# Build graph
# ============================================================

post_graph = {}

post_counter = 0


def post_add(
    class_type,
    inputs,
):
    global post_counter

    post_counter += 1

    nid = str(
        post_counter
    )

    post_graph[
        nid
    ] = {
        "class_type":
            class_type,

        "inputs":
            inputs,
    }

    return nid


# Node 1
load_video_id = post_add(
    "VHS_LoadVideo",
    {
        "video":
            source_rel,

        "force_rate":
            0,

        "custom_width":
            0,

        "custom_height":
            0,

        "frame_load_cap":
            0,

        "skip_first_frames":
            0,

        "select_every_nth":
            1,
    },
)


# Node 2
vosr_loader_id = post_add(
    "VOSR2ModelLoader",
    {
        "model":
            VOSR_MODEL_BUNDLE,

        "dtype":
            VOSR_DTYPE,
    },
)


# Node 3
vosr_id = post_add(
    "VOSR2Upscale",
    {
        "model":
            [
                vosr_loader_id,
                0,
            ],

        "image":
            [
                load_video_id,
                0,
            ],

        "upscale":
            plan[
                "upscale_factor"
            ],

        "seed":
            VOSR_SEED,

        "color_alignment":
            VOSR_COLOR_ALIGNMENT,

        "tile_size":
            VOSR_TILE_SIZE,

        "tile_overlap":
            VOSR_TILE_OVERLAP,

        "vae_tile_size":
            VOSR_VAE_TILE_SIZE,

        "vae_tile_overlap":
            VOSR_VAE_TILE_OVERLAP,
    },
)


# Node 4
scale_id = post_add(
    "ImageScale",
    {
        "image":
            [
                vosr_id,
                0,
            ],

        "upscale_method":
            "lanczos",

        # 直接指定兩個偶數尺寸，
        # 不讓 ComfyUI 自己再算出奇數。
        "width":
            plan[
                "scale_width"
            ],

        "height":
            plan[
                "scale_height"
            ],

        "crop":
            "disabled",
    },
)


# Node 5
create_inputs = {
    "images":
        [
            scale_id,
            0,
        ],

    "fps":
        source_fps,

    "bit_depth":
        8,

    "color_space":
        "sRGB",

    "codec":
        "none",
}


if source_has_audio:
    create_inputs[
        "audio"
    ] = [
        load_video_id,
        2,
    ]


create_id = post_add(
    "CreateVideo",
    create_inputs,
)


# Node 6
post_prefix = (
    "video/"
    "VOSR2_"
    f"{Path(source_original_name).stem}_"
    f"{int(time.time())}"
)


POST_OUTPUT_NODE_ID = post_add(
    "SaveVideo",
    {
        "video":
            [
                create_id,
                0,
            ],

        "filename_prefix":
            post_prefix,

        "format":
            "auto",

        "codec":
            "auto",
    },
)


# ============================================================
# Node labels
# ============================================================

POST_NODE_LABELS = {
    load_video_id:
        "讀取影片 / 解碼影格",

    vosr_loader_id:
        "載入 VOSR2 模型",

    vosr_id:
        "VOSR2 AI 放大",

    scale_id:
        "縮放到最終尺寸",

    create_id:
        "建立影片 / 合併音訊",

    POST_OUTPUT_NODE_ID:
        "H.264 編碼 / 儲存 MP4",
}


# ============================================================
# Basic schema validation
# ============================================================

for node in post_graph.values():

    class_type = (
        node[
            "class_type"
        ]
    )

    spec = (
        object_info[
            class_type
        ][
            "input"
        ]
    )

    known = (
        set(
            spec.get(
                "required",
                {},
            )
        )
        | set(
            spec.get(
                "optional",
                {},
            )
        )
    )

    unexpected = (
        set(
            node[
                "inputs"
            ]
        )
        - known
    )

    if unexpected:
        raise ValueError(
            f"{class_type} input schema 已變更；"
            "未知參數："
            f"{sorted(unexpected)}"
        )


# ============================================================
# Progress state
# ============================================================

POST_CLIENT_ID = (
    "h3-vosr2-post-"
    f"{uuid.uuid4().hex}"
)


progress_lock = (
    threading.Lock()
)


progress_data = {
    "node_id":
        None,

    "node_label":
        "等待開始",

    "value":
        None,

    "max":
        None,

    "percent":
        None,

    "first_progress_time":
        None,

    "last_progress_time":
        None,

    "last_value":
        None,

    "last_max":
        None,

    "ws_connected":
        False,

    "ws_error":
        None,
}


progress_stop = (
    threading.Event()
)


post_prompt_id = None


# ============================================================
# WebSocket progress reader
# ============================================================

def _progress_ws_worker():

    ws = None

    try:
        ws_url = (
            BASE_URL
            .replace(
                "http://",
                "ws://",
            )
            + "/ws"
            + f"?clientId={POST_CLIENT_ID}"
        )

        ws = websocket.create_connection(
            ws_url,
            timeout=3,
        )

        ws.settimeout(
            2
        )

        with progress_lock:
            progress_data[
                "ws_connected"
            ] = True

        while (
            not progress_stop.is_set()
        ):
            try:
                message = (
                    ws.recv()
                )

            except (
                websocket.WebSocketTimeoutException
            ):
                continue

            except Exception as exc:
                if not progress_stop.is_set():
                    with progress_lock:
                        progress_data[
                            "ws_error"
                        ] = str(
                            exc
                        )
                break

            # Preview images are binary.
            if isinstance(
                message,
                bytes,
            ):
                continue

            try:
                event = json.loads(
                    message
                )

            except Exception:
                continue

            event_type = (
                event.get(
                    "type"
                )
            )

            data = (
                event.get(
                    "data",
                    {}
                )
            )

            event_prompt_id = (
                data.get(
                    "prompt_id"
                )
            )

            if (
                post_prompt_id
                and event_prompt_id
                and event_prompt_id
                != post_prompt_id
            ):
                continue

            # --------------------------------------------
            # Current executing node
            # --------------------------------------------

            if (
                event_type
                == "executing"
            ):
                node_id = (
                    data.get(
                        "node"
                    )
                )

                if (
                    node_id
                    is not None
                ):
                    with progress_lock:
                        progress_data[
                            "node_id"
                        ] = str(
                            node_id
                        )

                        progress_data[
                            "node_label"
                        ] = (
                            POST_NODE_LABELS.get(
                                str(
                                    node_id
                                ),
                                (
                                    "節點 "
                                    f"{node_id}"
                                ),
                            )
                        )

            # --------------------------------------------
            # Modern ComfyUI progress_state
            # --------------------------------------------

            elif (
                event_type
                == "progress_state"
            ):
                nodes = (
                    data.get(
                        "nodes",
                        {}
                    )
                )

                vosr_state = (
                    nodes.get(
                        str(
                            vosr_id
                        )
                    )
                )

                if (
                    vosr_state
                ):
                    value = float(
                        vosr_state.get(
                            "value",
                            0,
                        )
                    )

                    maximum = float(
                        vosr_state.get(
                            "max",
                            0,
                        )
                    )

                    now = (
                        time.monotonic()
                    )

                    with progress_lock:

                        progress_data[
                            "node_id"
                        ] = str(
                            vosr_id
                        )

                        progress_data[
                            "node_label"
                        ] = (
                            "VOSR2 AI 放大"
                        )

                        progress_data[
                            "value"
                        ] = value

                        progress_data[
                            "max"
                        ] = maximum

                        progress_data[
                            "percent"
                        ] = (
                            (
                                value
                                / maximum
                                * 100.0
                            )
                            if maximum > 0
                            else None
                        )

                        if (
                            value > 0
                            and progress_data[
                                "first_progress_time"
                            ]
                            is None
                        ):
                            progress_data[
                                "first_progress_time"
                            ] = now

                        progress_data[
                            "last_progress_time"
                        ] = now

                        progress_data[
                            "last_value"
                        ] = value

                        progress_data[
                            "last_max"
                        ] = maximum

            # --------------------------------------------
            # Older ComfyUI progress event fallback
            # --------------------------------------------

            elif (
                event_type
                == "progress"
            ):
                node_id = str(
                    data.get(
                        "node",
                        ""
                    )
                )

                if (
                    node_id
                    == str(
                        vosr_id
                    )
                ):
                    value = float(
                        data.get(
                            "value",
                            0,
                        )
                    )

                    maximum = float(
                        data.get(
                            "max",
                            0,
                        )
                    )

                    now = (
                        time.monotonic()
                    )

                    with progress_lock:

                        progress_data[
                            "node_id"
                        ] = node_id

                        progress_data[
                            "node_label"
                        ] = (
                            "VOSR2 AI 放大"
                        )

                        progress_data[
                            "value"
                        ] = value

                        progress_data[
                            "max"
                        ] = maximum

                        progress_data[
                            "percent"
                        ] = (
                            (
                                value
                                / maximum
                                * 100.0
                            )
                            if maximum > 0
                            else None
                        )

                        if (
                            value > 0
                            and progress_data[
                                "first_progress_time"
                            ]
                            is None
                        ):
                            progress_data[
                                "first_progress_time"
                            ] = now

                        progress_data[
                            "last_progress_time"
                        ] = now

    except Exception as exc:

        with progress_lock:
            progress_data[
                "ws_error"
            ] = str(
                exc
            )

    finally:

        if ws is not None:
            try:
                ws.close()
            except Exception:
                pass


progress_thread = (
    threading.Thread(
        target=_progress_ws_worker,
        daemon=True,
    )
)


progress_thread.start()


# 給 WebSocket 一點連線時間。
time.sleep(
    0.5
)


# ============================================================
# Submit job
# ============================================================

post_started_wallclock = (
    time.time()
)

post_started_monotonic = (
    time.monotonic()
)


resp = requests.post(
    BASE_URL
    + "/prompt",
    json={
        "prompt":
            post_graph,

        "client_id":
            POST_CLIENT_ID,
    },
    timeout=60,
)


if not resp.ok:
    progress_stop.set()

    raise RuntimeError(
        "VOSR2 提交失敗 "
        f"HTTP {resp.status_code}: "
        f"{resp.text}"
    )


payload = (
    resp.json()
)


if payload.get(
    "node_errors"
):
    progress_stop.set()

    raise RuntimeError(
        "VOSR2 節點驗證失敗：\n"
        + json.dumps(
            payload[
                "node_errors"
            ],
            ensure_ascii=False,
            indent=2,
        )
    )


post_prompt_id = (
    payload[
        "prompt_id"
    ]
)


print()
print(
    "VOSR2 job submitted:",
    post_prompt_id,
)

print(
    "開始等待 GPU 工作..."
)

print(
    "進度資訊每約 15 秒更新一次。"
)

print(
    "ETA 是 VOSR2 AI 放大階段的動態估計，"
    "最後 resize / 編碼還會再花一些時間。"
)

print()


# ============================================================
# Wait loop + status output
# ============================================================

post_timeout = float(
    os.environ.get(
        "H3_VOSR_TIMEOUT_SECONDS",
        "10800",
    )
)


post_deadline = (
    time.monotonic()
    + post_timeout
)


post_outputs = None

last_status_print = 0.0

last_print_percent = None


while (
    time.monotonic()
    < post_deadline
):

    # --------------------------------------------------------
    # Comfy history
    # --------------------------------------------------------

    try:
        history = requests.get(
            BASE_URL
            + f"/history/{post_prompt_id}",
            timeout=30,
        ).json()

    except requests.RequestException:
        history = {}

    if (
        post_prompt_id
        in history
    ):
        run = (
            history[
                post_prompt_id
            ]
        )

        status = run.get(
            "status",
            {},
        )

        if (
            status.get(
                "status_str"
            )
            == "error"
        ):
            progress_stop.set()

            raise RuntimeError(
                "VOSR2 推論失敗：\n"
                + json.dumps(
                    status,
                    ensure_ascii=False,
                    indent=2,
                )
            )

        post_outputs = (
            run.get(
                "outputs",
                {},
            )
        )

        if (
            POST_OUTPUT_NODE_ID
            in post_outputs
            or status.get(
                "status_str"
            )
            == "success"
        ):
            break

    # --------------------------------------------------------
    # Snapshot progress
    # --------------------------------------------------------

    with progress_lock:
        snapshot = dict(
            progress_data
        )

    now = (
        time.monotonic()
    )

    elapsed = (
        now
        - post_started_monotonic
    )

    percent = (
        snapshot.get(
            "percent"
        )
    )

    # 每 15 秒至少印一次；
    # 若進度跨過約 2%，也可以提前印。
    percent_changed = False

    if (
        percent is not None
    ):
        if (
            last_print_percent
            is None
            or abs(
                percent
                - last_print_percent
            )
            >= 2.0
        ):
            percent_changed = True

    should_print = (
        (
            now
            - last_status_print
            >= 15
        )
        or percent_changed
    )

    if should_print:

        last_status_print = (
            now
        )

        if (
            percent is not None
        ):
            last_print_percent = (
                percent
            )

        node_label = (
            snapshot.get(
                "node_label"
            )
            or "等待 GPU"
        )

        # ----------------------------------------------------
        # ETA
        # ----------------------------------------------------

        eta = None

        value = snapshot.get(
            "value"
        )

        maximum = snapshot.get(
            "max"
        )

        first_progress_time = (
            snapshot.get(
                "first_progress_time"
            )
        )

        if (
            value is not None
            and maximum is not None
            and value > 0
            and maximum > value
            and first_progress_time
            is not None
        ):
            progress_elapsed = (
                now
                - first_progress_time
            )

            # 用平均 tile rate 估算。
            if (
                progress_elapsed
                > 3
            ):
                tiles_per_second = (
                    value
                    / progress_elapsed
                )

                if (
                    tiles_per_second
                    > 0
                ):
                    eta = (
                        (
                            maximum
                            - value
                        )
                        / tiles_per_second
                    )

        gpu = (
            _gpu_memory_status()
        )

        gpu_text = ""

        if (
            gpu[
                "used_mib"
            ]
            is not None
        ):
            gpu_text = (
                " | GPU "
                f"{gpu['used_mib']}"
                "/"
                f"{gpu['total_mib']} MiB"
                " | util "
                f"{gpu['util_percent']}%"
            )

        if (
            percent is not None
            and maximum
        ):
            print(
                "[VOSR2] "
                f"{node_label}"
                " | "
                f"{percent:6.2f}%"
                " | "
                f"{int(value)}/{int(maximum)} tiles"
                " | elapsed "
                f"{_format_seconds(elapsed)}"
                " | ETA "
                f"{_format_seconds(eta)}"
                f"{gpu_text}"
            )

        else:
            print(
                "[VOSR2] "
                f"{node_label}"
                " | elapsed "
                f"{_format_seconds(elapsed)}"
                " | 正在等待可用的進度資料"
                f"{gpu_text}"
            )

            if (
                snapshot.get(
                    "ws_error"
                )
            ):
                print(
                    "         "
                    "WebSocket progress unavailable：",
                    snapshot[
                        "ws_error"
                    ],
                )

    time.sleep(
        3
    )


else:
    progress_stop.set()

    raise TimeoutError(
        "VOSR2 等待超過 "
        f"{post_timeout:.0f} 秒。"
    )


# Stop websocket reader
progress_stop.set()


total_elapsed = (
    time.monotonic()
    - post_started_monotonic
)


print()
print(
    "[VOSR2] ComfyUI 工作完成，"
    "正在確認輸出影片..."
)

print(
    "總處理時間:",
    _format_seconds(
        total_elapsed
    ),
)


# ============================================================
# Locate MP4
# ============================================================

post_candidates = (
    _find_post_mp4_infos(
        (
            post_outputs.get(
                POST_OUTPUT_NODE_ID,
                {},
            )
            if post_outputs
            else {}
        )
    )
)


post_video_path = None


for info in post_candidates:

    p = (
        COMFY
        / "output"
        / info.get(
            "subfolder",
            "",
        )
        / info[
            "filename"
        ]
    )

    if (
        p.is_file()
        and p.stat().st_size > 0
    ):
        post_video_path = p
        break


if (
    post_video_path
    is None
):
    recent = [
        p
        for p
        in (
            COMFY
            / "output"
        ).rglob(
            "*.mp4"
        )
        if (
            p.is_file()
            and p.stat().st_mtime
            >= post_started_wallclock - 2
        )
    ]

    if recent:
        post_video_path = max(
            recent,
            key=lambda p:
                p.stat().st_mtime,
        )


if (
    post_video_path
    is None
):
    raise RuntimeError(
        "VOSR2 完成但找不到輸出 MP4。"
    )


# ============================================================
# Verify final MP4
# ============================================================

probe = subprocess.run(
    [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "stream=codec_type,width,height",
        "-of",
        "json",
        str(
            post_video_path
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)


probe_data = json.loads(
    probe.stdout
)


final_streams = {
    item.get(
        "codec_type"
    )
    for item
    in probe_data.get(
        "streams",
        [],
    )
}


if (
    "video"
    not in final_streams
):
    raise RuntimeError(
        "輸出 MP4 沒有 video stream。"
    )


video_stream = next(
    (
        item
        for item
        in probe_data.get(
            "streams",
            [],
        )
        if (
            item.get(
                "codec_type"
            )
            == "video"
        )
    ),
    {},
)


actual_final_w = (
    video_stream.get(
        "width"
    )
)


actual_final_h = (
    video_stream.get(
        "height"
    )
)


# ============================================================
# TXT
# ============================================================

post_info_path = (
    post_video_path.with_suffix(
        ".txt"
    )
)


post_info_lines = [
    "============================================================",
    "VOSR UPSCALE INFO",
    "============================================================",
    "",
    (
        "source_file="
        f"{source_original_name}"
    ),
    (
        "source_width="
        f"{source_w}"
    ),
    (
        "source_height="
        f"{source_h}"
    ),
    (
        "source_fps="
        f"{source_fps}"
    ),
    (
        "source_frame_count="
        f"{source_frame_count if source_frame_count is not None else 'N/A'}"
    ),
    (
        "source_has_audio="
        f"{source_has_audio}"
    ),
    (
        "vosr_model_bundle="
        f"{VOSR_MODEL_BUNDLE}"
    ),
    (
        "vosr_dtype="
        f"{VOSR_DTYPE}"
    ),
    (
        "vosr_seed="
        f"{VOSR_SEED}"
    ),
    (
        "vosr_color_alignment="
        f"{VOSR_COLOR_ALIGNMENT}"
    ),
    (
        "target_width="
        f"{VOSR_TARGET_WIDTH}"
    ),
    (
        "target_height="
        f"{VOSR_TARGET_HEIGHT}"
    ),
    (
        "fit_side="
        f"{VOSR_FIT_SIDE}"
    ),
    (
        "internal_upscale="
        f"{plan['upscale_factor']}"
    ),
    (
        "planned_final_width="
        f"{plan['final_width']}"
    ),
    (
        "planned_final_height="
        f"{plan['final_height']}"
    ),
    (
        "actual_final_width="
        f"{actual_final_w}"
    ),
    (
        "actual_final_height="
        f"{actual_final_h}"
    ),
    (
        "codec_even_dimensions=True"
    ),
    (
        "tile_size="
        f"{VOSR_TILE_SIZE}"
    ),
    (
        "tile_overlap="
        f"{VOSR_TILE_OVERLAP}"
    ),
    (
        "vae_tile_size="
        f"{VOSR_VAE_TILE_SIZE}"
    ),
    (
        "vae_tile_overlap="
        f"{VOSR_VAE_TILE_OVERLAP}"
    ),
    (
        "processing_seconds="
        f"{total_elapsed:.3f}"
    ),
    (
        "prompt_id="
        f"{post_prompt_id}"
    ),
    "",
]


post_info_path.write_text(
    "\n".join(
        post_info_lines
    ),
    encoding="utf-8",
)


# ============================================================
# Done
# ============================================================

print()
print(
    "=" * 72
)

print(
    "VOSR2 完成"
)

print(
    "=" * 72
)

print(
    "影片:",
    post_video_path,
)

print(
    "TXT:",
    post_info_path,
)

print(
    "實際輸出尺寸:",
    (
        f"{actual_final_w}"
        f"x"
        f"{actual_final_h}"
    ),
)

print(
    "總處理時間:",
    _format_seconds(
        total_elapsed
    ),
)

print(
    "影片大小:",
    (
        f"{post_video_path.stat().st_size / 2**20:.1f} MiB"
    ),
)

print(
    "=" * 72
)


display(
    Video(
        str(
            post_video_path
        ),
        embed=True,
    )
)


print(
    "準備下載 MP4..."
)

colab_files.download(
    str(
        post_video_path
    )
)


print(
    "準備下載 TXT..."
)

colab_files.download(
    str(
        post_info_path
    )
)

## Cell 10 — Remote API （不需要 Colab CLI）

提供：

- `POST /generate`：H3 生成，可選 Inline VOSR2。
- `POST /upscale`：上傳既有影片，用 VHS + VOSR2 放大，不重新執行 H3。
- `GET /jobs/{job_id}`：輪詢工作。
- `GET /result/{job_id}`：下載 MP4。
- `GET /info/{job_id}`：下載 TXT。
- `GET /models`：查看已安裝模型 / LoRA 以及官方清單。

Quick Tunnel 與 Token 在 runtime 重啟後會改變，謹慎保存Token。


In [ ]:
import json
import math
import queue
import re
import secrets
import shutil
import subprocess
import threading
import time
import uuid

from fractions import Fraction
from io import BytesIO
from pathlib import Path
from typing import Optional, List

import requests

from PIL import Image

from fastapi import (
    FastAPI,
    UploadFile,
    File,
    Form,
    Header,
    HTTPException,
)

from fastapi.responses import (
    FileResponse,
)

import uvicorn


assert (
    "BASE_URL"
    in globals()
    and "object_info"
    in globals()
), (
    "請先執行 Cell 6 啟動 ComfyUI。"
)


# ============================================================
# API settings
# ============================================================

REMOTE_API_HOST = (
    "127.0.0.1"
)

REMOTE_API_PORT = int(
    os.environ.get(
        "H3_REMOTE_API_PORT",
        "7860",
    )
)


REMOTE_API_TOKEN = (
    secrets.token_urlsafe(
        32
    )
)


REMOTE_OUTPUT_DIR = Path(
    "/content/h3_remote_outputs"
)


REMOTE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REMOTE_MAX_IMAGE_BYTES = (
    50 * 2**20
)

REMOTE_MAX_VIDEO_BYTES = (
    1024 * 2**20
)

REMOTE_MAX_AUDIO_BYTES = (
    256 * 2**20
)


REMOTE_JOBS = {}

REMOTE_JOBS_LOCK = (
    threading.Lock()
)

REMOTE_QUEUE = (
    queue.Queue()
)


# ============================================================
# Job state
# ============================================================

def _remote_set_job(
    job_id,
    **updates,
):
    with REMOTE_JOBS_LOCK:
        REMOTE_JOBS.setdefault(
            job_id,
            {},
        ).update(
            updates
        )


def _remote_get_job(
    job_id,
):
    with REMOTE_JOBS_LOCK:
        item = REMOTE_JOBS.get(
            job_id
        )

        return (
            dict(item)
            if item
            else None
        )


def _remote_require_auth(
    authorization,
):
    if (
        authorization
        != f"Bearer {REMOTE_API_TOKEN}"
    ):
        raise HTTPException(
            status_code=401,
            detail="Unauthorized",
        )


# ============================================================
# Generic helpers
# ============================================================

def _remote_frames(
    seconds,
):
    seconds = float(
        seconds
    )

    if (
        not math.isfinite(
            seconds
        )
        or not 4 <= seconds <= 15
    ):
        raise ValueError(
            "duration_seconds 必須介於 4–15。"
        )

    requested = max(
        5,
        round(
            seconds * 24
        ),
    )

    return (
        requested
        + (
            5
            - requested % 17
        )
        % 17
    )


def _remote_preset_model(
    mode,
):
    table = {
        "ref2v_native":
            (
                "minimax_h3_ref2va_"
                "pruned_int8_convrot.safetensors"
            ),

        "i2v_native":
            (
                "minimax_h3_fl2va_"
                "pruned_int8_convrot.safetensors"
            ),

        "t2v_native":
            (
                "minimax_h3_fl2va_"
                "pruned_int8_convrot.safetensors"
            ),

        "fast_i2v":
            FAST_MODEL,

        "fast_t2v":
            FAST_MODEL,
    }

    if mode not in table:
        raise ValueError(
            "不支援的 mode："
            f"{mode}"
        )

    return table[
        mode
    ]


def _remote_clean_model_name(
    value,
    folder,
    label,
):
    value = str(
        value
        or ""
    ).strip()

    if not value:
        raise ValueError(
            f"{label} 檔名不可空白。"
        )

    name = Path(
        value
    ).name

    if (
        name
        != value
    ):
        raise ValueError(
            f"{label} 只能傳檔名，"
            "不能傳路徑："
            f"{value}"
        )

    path = (
        Path(folder)
        / name
    )

    if (
        not path.is_file()
        or path.stat().st_size <= 0
    ):
        raise FileNotFoundError(
            f"找不到 {label}："
            f"{path}"
        )

    return name


def _remote_extract_options(
    spec,
):
    if isinstance(
        spec,
        (
            list,
            tuple,
        ),
    ):
        if (
            spec
            and isinstance(
                spec[0],
                (
                    list,
                    tuple,
                ),
            )
        ):
            return list(
                spec[0]
            )

        for item in spec[1:]:
            if isinstance(
                item,
                dict,
            ):
                for key in (
                    "options",
                    "values",
                    "choices",
                ):
                    values = item.get(
                        key
                    )

                    if isinstance(
                        values,
                        (
                            list,
                            tuple,
                        ),
                    ):
                        return list(
                            values
                        )

    if isinstance(
        spec,
        dict,
    ):
        for key in (
            "options",
            "values",
            "choices",
        ):
            values = spec.get(
                key
            )

            if isinstance(
                values,
                (
                    list,
                    tuple,
                ),
            ):
                return list(
                    values
                )

    return None


def _remote_validate_graph(
    graph,
    current_object_info,
):
    for (
        node_id,
        node,
    ) in graph.items():

        class_type = (
            node[
                "class_type"
            ]
        )

        if (
            class_type
            not in current_object_info
        ):
            raise ValueError(
                "ComfyUI 缺少節點："
                f"{class_type}"
            )

        spec = (
            current_object_info[
                class_type
            ][
                "input"
            ]
        )

        known = (
            set(
                spec.get(
                    "required",
                    {},
                )
            )
            | set(
                spec.get(
                    "optional",
                    {},
                )
            )
        )

        dynamic = set()

        if (
            class_type
            == "MiniMaxH3ReferenceToVideo"
        ):
            dynamic |= {
                key
                for key
                in node[
                    "inputs"
                ]
                if re.fullmatch(
                    (
                        r"ref_(?:"
                        r"images\.ref_image|"
                        r"videos\.ref_video|"
                        r"video_audios\.ref_video_audio|"
                        r"audios\.ref_audio"
                        r")_\d+"
                    ),
                    key,
                )
            }

        elif (
            class_type
            == "BlockSparseAttention"
        ):
            dynamic |= {
                key
                for key
                in node[
                    "inputs"
                ]
                if key.startswith(
                    "selection."
                )
            }

        unexpected = (
            set(
                node[
                    "inputs"
                ]
            )
            - known
            - dynamic
        )

        if unexpected:
            raise ValueError(
                f"{class_type} input schema 已變更；"
                "未知參數："
                f"{sorted(unexpected)}"
            )


# ============================================================
# VOSR helpers
# ============================================================

def _remote_vosr_plan(
    source_width,
    source_height,
    target_width,
    target_height,
    fit_side,
):
    return plan_vosr_output(
        source_width,
        source_height,
        target_width,
        target_height,
        fit_side,
    )


def _remote_vosr_cfg(
    *,
    source_width,
    source_height,
    target_width,
    target_height,
    fit_side,
    dtype,
    seed,
    color_alignment,
    tile_size,
    tile_overlap,
    vae_tile_size,
    vae_tile_overlap,
):
    plan = _remote_vosr_plan(
        source_width,
        source_height,
        target_width,
        target_height,
        fit_side,
    )

    cfg = {
        **plan,

        "model_bundle":
            "VOSR2",

        "dtype":
            str(
                dtype
                or "bf16"
            ).strip(),

        "seed":
            int(
                seed
            ),

        "color_alignment":
            str(
                color_alignment
                or "wavelet"
            ).strip(),

        "tile_size":
            int(
                tile_size
            ),

        "tile_overlap":
            int(
                tile_overlap
            ),

        "vae_tile_size":
            int(
                vae_tile_size
            ),

        "vae_tile_overlap":
            int(
                vae_tile_overlap
            ),
    }

    if (
        cfg[
            "dtype"
        ]
        not in {
            "default",
            "fp16",
            "bf16",
        }
    ):
        raise ValueError(
            "VOSR dtype 必須是 "
            "default / fp16 / bf16。"
        )

    if (
        cfg[
            "color_alignment"
        ]
        not in {
            "wavelet",
            "adain",
            "none",
        }
    ):
        raise ValueError(
            "VOSR color_alignment 必須是 "
            "wavelet / adain / none。"
        )

    if cfg["seed"] < 0:
        raise ValueError(
            "VOSR seed 必須 >= 0。"
        )

    if (
        cfg["tile_size"] < 0
        or cfg["tile_overlap"] < 0
        or cfg["vae_tile_size"] < 0
        or cfg["vae_tile_overlap"] < 0
    ):
        raise ValueError(
            "VOSR tile 參數不可為負數。"
        )

    if (
        cfg["tile_size"] > 0
        and cfg["tile_overlap"]
        >= cfg["tile_size"]
    ):
        raise ValueError(
            "VOSR tile_overlap 必須小於 tile_size。"
        )

    if (
        cfg["vae_tile_size"] > 0
        and cfg["vae_tile_overlap"]
        >= cfg["vae_tile_size"]
    ):
        raise ValueError(
            "VOSR vae_tile_overlap "
            "必須小於 vae_tile_size。"
        )

    return cfg


def _remote_add_vosr_nodes(
    add_node,
    image_ref,
    vosr,
):
    loader_id = add_node(
        "VOSR2ModelLoader",
        {
            "model":
                vosr[
                    "model_bundle"
                ],

            "dtype":
                vosr[
                    "dtype"
                ],
        },
    )

    upscale_id = add_node(
        "VOSR2Upscale",
        {
            "model":
                [
                    loader_id,
                    0,
                ],

            "image":
                image_ref,

            "upscale":
                vosr[
                    "upscale_factor"
                ],

            "seed":
                vosr[
                    "seed"
                ],

            "color_alignment":
                vosr[
                    "color_alignment"
                ],

            "tile_size":
                vosr[
                    "tile_size"
                ],

            "tile_overlap":
                vosr[
                    "tile_overlap"
                ],

            "vae_tile_size":
                vosr[
                    "vae_tile_size"
                ],

            "vae_tile_overlap":
                vosr[
                    "vae_tile_overlap"
                ],
        },
    )

    scale_id = add_node(
        "ImageScale",
        {
            "image":
                [
                    upscale_id,
                    0,
                ],

            "upscale_method":
                "lanczos",

            "width":
                vosr[
                    "scale_width"
                ],

            "height":
                vosr[
                    "scale_height"
                ],

            "crop":
                "disabled",
        },
    )

    return [
        scale_id,
        0,
    ]


# ============================================================
# Generation graph
# ============================================================

def _remote_build_graph(
    cfg,
    image_files,
    video_files=None,
    audio_files=None,
):
    current_object_info = requests.get(
        BASE_URL
        + "/object_info",
        timeout=60,
    ).json()

    mode = cfg[
        "mode"
    ]

    video_files = list(
        video_files
        or []
    )

    audio_files = list(
        audio_files
        or []
    )

    is_fast = (
        mode.startswith(
            "fast_"
        )
    )

    is_ref = (
        mode.startswith(
            "ref2v_"
        )
    )

    is_i2v = (
        mode
        in {
            "i2v_native",
            "fast_i2v",
        }
    )

    graph = {}
    counter = 0

    def add_node(
        class_type,
        inputs,
    ):
        nonlocal counter

        counter += 1

        node_id = str(
            counter
        )

        graph[
            node_id
        ] = {
            "class_type":
                class_type,

            "inputs":
                inputs,
        }

        return node_id

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    unet_id = add_node(
        "UNETLoader",
        {
            "unet_name":
                cfg[
                    "model"
                ],

            "weight_dtype":
                "default",
        },
    )

    clip_id = add_node(
        "CLIPLoader",
        {
            "clip_name":
                TEXT_ENCODER,

            "type":
                "minimax",

            "device":
                "default",
        },
    )

    video_vae_id = add_node(
        "VAELoader",
        {
            "vae_name":
                VIDEO_VAE,
        },
    )

    audio_vae_id = add_node(
        "VAELoader",
        {
            "vae_name":
                AUDIO_VAE,
        },
    )

    # --------------------------------------------------------
    # Model chain
    # --------------------------------------------------------

    if is_fast:

        if cfg.get(
            "lora"
        ):
            raise ValueError(
                "Remote Fast H3 維持 "
                "FastVSA 專用 graph，"
                "不支援 LoraLoaderModelOnly。"
            )

        sigma_id = add_node(
            "MiniMaxH3SigmaShift",
            {
                "model":
                    [
                        unet_id,
                        0,
                    ],

                "shift_video":
                    cfg[
                        "fast_shift_video"
                    ],

                "shift_audio":
                    cfg[
                        "fast_shift_audio"
                    ],
            },
        )

        backend_id = add_node(
            "ModelAttentionBackend",
            {
                "model":
                    [
                        sigma_id,
                        0,
                    ],

                "attention":
                    "comfy kitchen attention",
            },
        )

        if cfg[
            "fast_use_vsa"
        ]:
            sparse_id = add_node(
                "BlockSparseAttention",
                {
                    "model":
                        [
                            backend_id,
                            0,
                        ],

                    "selection":
                        "vsa",

                    "selection.keep_percent":
                        10.0,

                    "start_percent":
                        0.2,

                    "end_percent":
                        1.0,

                    "dense_blocks":
                        "",

                    "min_tokens":
                        12288,

                    "extra_tokens":
                        256,

                    "sink_conditioning":
                        "exact_kv_and_rows",

                    "verbose":
                        False,
                },
            )

            sampling_model = [
                sparse_id,
                0,
            ]

        else:
            sampling_model = [
                backend_id,
                0,
            ]

    else:

        model_source = [
            unet_id,
            0,
        ]

        if cfg.get(
            "lora"
        ):
            lora_id = add_node(
                "LoraLoaderModelOnly",
                {
                    "model":
                        model_source,

                    "lora_name":
                        cfg[
                            "lora"
                        ],

                    "strength_model":
                        cfg[
                            "lora_strength"
                        ],
                },
            )

            model_source = [
                lora_id,
                0,
            ]

        # ----------------------------------------------------
        # ModelSamplingMiniMaxH3
        # API class_type = MiniMaxH3SigmaShift
        # ----------------------------------------------------

        if (
            cfg.get(
                "lora"
            )
            and cfg.get(
                "h3_shift_enabled",
                False,
            )
        ):
            shift_id = add_node(
                "MiniMaxH3SigmaShift",
                {
                    "model":
                        model_source,

                    "shift_video":
                        cfg[
                            "h3_shift_video"
                        ],

                    "shift_audio":
                        cfg[
                            "h3_shift_audio"
                        ],
                },
            )

            model_source = [
                shift_id,
                0,
            ]

        backend_id = add_node(
            "ModelAttentionBackend",
            {
                "model":
                    model_source,

                "attention":
                    "comfy kitchen attention",
            },
        )

        sampling_model = [
            backend_id,
            0,
        ]

    # --------------------------------------------------------
    # Conditioning
    # --------------------------------------------------------

    if is_ref:

        ref_inputs = {
            "clip":
                [
                    clip_id,
                    0,
                ],

            "vae":
                [
                    video_vae_id,
                    0,
                ],

            "audio_vae":
                [
                    audio_vae_id,
                    0,
                ],

            "prompt":
                cfg[
                    "prompt"
                ],

            "width":
                cfg[
                    "width"
                ],

            "height":
                cfg[
                    "height"
                ],

            "length":
                cfg[
                    "frames"
                ],

            "ref_image_size":
                cfg[
                    "ref_image_size"
                ],
        }

        for (
            index,
            filename,
        ) in enumerate(
            image_files,
            start=1,
        ):
            load_id = add_node(
                "LoadImage",
                {
                    "image":
                        filename,
                },
            )

            ref_inputs[
                (
                    "ref_images."
                    f"ref_image_{index - 1}"
                )
            ] = [
                load_id,
                0,
            ]

        for (
            index,
            filename,
        ) in enumerate(
            video_files,
            start=1,
        ):
            load_id = add_node(
                "LoadVideo",
                {
                    "file":
                        filename,
                },
            )

            split_id = add_node(
                "GetVideoComponents",
                {
                    "video":
                        [
                            load_id,
                            0,
                        ],
                },
            )

            ref_inputs[
                (
                    "ref_videos."
                    f"ref_video_{index - 1}"
                )
            ] = [
                split_id,
                0,
            ]

            ref_inputs[
                (
                    "ref_video_audios."
                    f"ref_video_audio_{index - 1}"
                )
            ] = [
                split_id,
                1,
            ]

        for (
            index,
            filename,
        ) in enumerate(
            audio_files,
            start=1,
        ):
            load_id = add_node(
                "LoadAudio",
                {
                    "audio":
                        filename,
                },
            )

            ref_inputs[
                (
                    "ref_audios."
                    f"ref_audio_{index - 1}"
                )
            ] = [
                load_id,
                0,
            ]

        conditioning_id = add_node(
            "MiniMaxH3ReferenceToVideo",
            ref_inputs,
        )

    else:

        i2v_inputs = {
            "clip":
                [
                    clip_id,
                    0,
                ],

            "vae":
                [
                    video_vae_id,
                    0,
                ],

            "prompt":
                cfg[
                    "prompt"
                ],

            "width":
                cfg[
                    "width"
                ],

            "height":
                cfg[
                    "height"
                ],

            "length":
                cfg[
                    "frames"
                ],
        }

        if is_i2v:

            first_id = add_node(
                "LoadImage",
                {
                    "image":
                        image_files[
                            0
                        ],
                },
            )

            i2v_inputs[
                "first_frame"
            ] = [
                first_id,
                0,
            ]

            if (
                len(
                    image_files
                )
                > 1
            ):
                last_id = add_node(
                    "LoadImage",
                    {
                        "image":
                            image_files[
                                1
                            ],
                    },
                )

                i2v_inputs[
                    "last_frame"
                ] = [
                    last_id,
                    0,
                ]

        conditioning_id = add_node(
            "MiniMaxH3ImageToVideo",
            i2v_inputs,
        )

    # --------------------------------------------------------
    # Sampling
    # --------------------------------------------------------

    guider_id = add_node(
        "BasicGuider",
        {
            "model":
                sampling_model,

            "conditioning":
                [
                    conditioning_id,
                    0,
                ],
        },
    )

    noise_id = add_node(
        "RandomNoise",
        {
            "noise_seed":
                cfg[
                    "seed"
                ],
        },
    )

    sampler_id = add_node(
        "KSamplerSelect",
        {
            "sampler_name":
                cfg[
                    "sampler"
                ],
        },
    )

    scheduler_id = add_node(
        "BasicScheduler",
        {
            "model":
                sampling_model,

            "scheduler":
                cfg[
                    "scheduler"
                ],

            "steps":
                cfg[
                    "steps"
                ],

            "denoise":
                1.0,
        },
    )

    sample_id = add_node(
        "SamplerCustomAdvanced",
        {
            "noise":
                [
                    noise_id,
                    0,
                ],

            "guider":
                [
                    guider_id,
                    0,
                ],

            "sampler":
                [
                    sampler_id,
                    0,
                ],

            "sigmas":
                [
                    scheduler_id,
                    0,
                ],

            "latent_image":
                [
                    conditioning_id,
                    1,
                ],
        },
    )

    # --------------------------------------------------------
    # Decode
    # --------------------------------------------------------

    video_decode_id = add_node(
        "VAEDecode",
        {
            "samples":
                [
                    sample_id,
                    0,
                ],

            "vae":
                [
                    video_vae_id,
                    0,
                ],
        },
    )

    audio_decode_id = add_node(
        "VAEDecodeAudio",
        {
            "samples":
                [
                    sample_id,
                    0,
                ],

            "vae":
                [
                    audio_vae_id,
                    0,
                ],
        },
    )

    images_ref = [
        video_decode_id,
        0,
    ]

    # --------------------------------------------------------
    # Inline VOSR
    # --------------------------------------------------------

    if (
        cfg[
            "upscale_mode"
        ]
        == "inline"
    ):
        images_ref = (
            _remote_add_vosr_nodes(
                add_node,
                images_ref,
                cfg[
                    "vosr"
                ],
            )
        )

    # --------------------------------------------------------
    # Create / Save
    # --------------------------------------------------------

    create_id = add_node(
        "CreateVideo",
        {
            "images":
                images_ref,

            "audio":
                [
                    audio_decode_id,
                    0,
                ],

            "fps":
                24,

            "bit_depth":
                8,

            "color_space":
                "sRGB",

            "codec":
                "none",
        },
    )

    output_node_id = add_node(
        "SaveVideo",
        {
            "video":
                [
                    create_id,
                    0,
                ],

            "filename_prefix":
                cfg[
                    "output_prefix"
                ],

            "format":
                "auto",

            "codec":
                "auto",
        },
    )

    _remote_validate_graph(
        graph,
        current_object_info,
    )

    # Validate sampler
    sampler_options = (
        _remote_extract_options(
            current_object_info[
                "KSamplerSelect"
            ][
                "input"
            ][
                "required"
            ][
                "sampler_name"
            ]
        )
    )

    if (
        sampler_options
        is not None
        and cfg[
            "sampler"
        ]
        not in sampler_options
    ):
        raise ValueError(
            f"Sampler '{cfg['sampler']}' 不存在。"
        )

    scheduler_options = (
        _remote_extract_options(
            current_object_info[
                "BasicScheduler"
            ][
                "input"
            ][
                "required"
            ][
                "scheduler"
            ]
        )
    )

    if (
        scheduler_options
        is not None
        and cfg[
            "scheduler"
        ]
        not in scheduler_options
    ):
        raise ValueError(
            f"Scheduler '{cfg['scheduler']}' 不存在。"
        )

    return (
        graph,
        output_node_id,
    )


# ============================================================
# ffprobe helpers
# ============================================================

def _remote_probe_has_stream(
    path,
    stream_type,
):
    p = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "stream=codec_type",
            "-of",
            "json",
            str(
                path
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    data = json.loads(
        p.stdout
    )

    return any(
        (
            item.get(
                "codec_type"
            )
            == stream_type
        )
        for item
        in data.get(
            "streams",
            [],
        )
    )


def _remote_probe_video(
    path,
):
    p = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            (
                "stream="
                "codec_type,"
                "width,"
                "height,"
                "avg_frame_rate"
            ),
            "-of",
            "json",
            str(
                path
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    data = json.loads(
        p.stdout
    )

    stream = next(
        (
            s
            for s
            in data.get(
                "streams",
                [],
            )
            if (
                s.get(
                    "codec_type"
                )
                == "video"
            )
        ),
        None,
    )

    if not stream:
        raise ValueError(
            "影片沒有 video stream。"
        )

    width = int(
        stream[
            "width"
        ]
    )

    height = int(
        stream[
            "height"
        ]
    )

    try:
        fps = float(
            Fraction(
                str(
                    stream.get(
                        "avg_frame_rate"
                    )
                    or "24/1"
                )
            )
        )

    except Exception:
        fps = 24.0

    if (
        not math.isfinite(
            fps
        )
        or fps <= 0
    ):
        fps = 24.0

    has_audio = any(
        (
            s.get(
                "codec_type"
            )
            == "audio"
        )
        for s
        in data.get(
            "streams",
            [],
        )
    )

    return (
        width,
        height,
        fps,
        has_audio,
    )


# ============================================================
# Existing-video VOSR graph
# ============================================================

def _remote_build_upscale_graph(
    cfg,
    video_file,
):
    current_object_info = requests.get(
        BASE_URL
        + "/object_info",
        timeout=60,
    ).json()

    graph = {}
    counter = 0

    def add_node(
        class_type,
        inputs,
    ):
        nonlocal counter

        counter += 1

        node_id = str(
            counter
        )

        graph[
            node_id
        ] = {
            "class_type":
                class_type,

            "inputs":
                inputs,
        }

        return node_id

    load_id = add_node(
        "VHS_LoadVideo",
        {
            "video":
                video_file,

            "force_rate":
                0,

            "custom_width":
                0,

            "custom_height":
                0,

            "frame_load_cap":
                0,

            "skip_first_frames":
                0,

            "select_every_nth":
                1,
        },
    )

    images_ref = (
        _remote_add_vosr_nodes(
            add_node,
            [
                load_id,
                0,
            ],
            cfg[
                "vosr"
            ],
        )
    )

    create_inputs = {
        "images":
            images_ref,

        "fps":
            cfg[
                "source_fps"
            ],

        "bit_depth":
            8,

        "color_space":
            "sRGB",

        "codec":
            "none",
    }

    if cfg[
        "source_has_audio"
    ]:
        create_inputs[
            "audio"
        ] = [
            load_id,
            2,
        ]

    create_id = add_node(
        "CreateVideo",
        create_inputs,
    )

    output_node_id = add_node(
        "SaveVideo",
        {
            "video":
                [
                    create_id,
                    0,
                ],

            "filename_prefix":
                cfg[
                    "output_prefix"
                ],

            "format":
                "auto",

            "codec":
                "auto",
        },
    )

    _remote_validate_graph(
        graph,
        current_object_info,
    )

    return (
        graph,
        output_node_id,
    )


# ============================================================
# Upload helper
# ============================================================

def _remote_save_upload_stream(
    upload,
    target,
    max_bytes,
    label,
):
    target = Path(
        target
    )

    target.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    total = 0

    with target.open(
        "wb"
    ) as f:

        while True:

            chunk = (
                upload.file.read(
                    1024 * 1024
                )
            )

            if not chunk:
                break

            total += len(
                chunk
            )

            if (
                total
                > max_bytes
            ):
                raise ValueError(
                    f"{label} 超過 "
                    f"{max_bytes / 2**20:.0f} MiB 上限。"
                )

            f.write(
                chunk
            )

    if (
        total <= 0
        or not target.is_file()
    ):
        raise ValueError(
            f"{label} 是空檔。"
        )

    return total


# ============================================================
# Output discovery
# ============================================================

def _remote_find_mp4_infos(
    obj,
    found=None,
):
    if found is None:
        found = []

    if isinstance(
        obj,
        dict,
    ):
        filename = obj.get(
            "filename"
        )

        if (
            isinstance(
                filename,
                str,
            )
            and filename.lower().endswith(
                ".mp4"
            )
        ):
            found.append(
                obj
            )

        for value in obj.values():
            _remote_find_mp4_infos(
                value,
                found,
            )

    elif isinstance(
        obj,
        (
            list,
            tuple,
        ),
    ):
        for value in obj:
            _remote_find_mp4_infos(
                value,
                found,
            )

    return found


def _remote_gpu_memory_used_mib():
    try:
        p = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=memory.used",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        values = [
            int(
                x.strip()
            )
            for x
            in p.stdout.splitlines()
            if x.strip()
        ]

        return (
            max(
                values
            )
            if values
            else None
        )

    except Exception:
        return None


# ============================================================
# Execute Comfy graph
# ============================================================

def _remote_execute_graph(
    job_id,
    graph,
    output_node_id,
    timeout_seconds,
    *,
    require_audio,
):
    started_wallclock = (
        time.time()
    )

    started_monotonic = (
        time.monotonic()
    )

    peak_gpu_memory_mib = (
        _remote_gpu_memory_used_mib()
    )

    response = requests.post(
        BASE_URL
        + "/prompt",
        json={
            "prompt":
                graph,

            "client_id":
                (
                    "h3-remote-"
                    f"{job_id[:12]}"
                ),
        },
        timeout=60,
    )

    if not response.ok:
        raise RuntimeError(
            "ComfyUI 提交失敗 "
            f"HTTP {response.status_code}: "
            f"{response.text}"
        )

    payload = response.json()

    if payload.get(
        "node_errors"
    ):
        raise RuntimeError(
            "ComfyUI 節點驗證失敗：\n"
            + json.dumps(
                payload[
                    "node_errors"
                ],
                ensure_ascii=False,
                indent=2,
            )
        )

    prompt_id = (
        payload[
            "prompt_id"
        ]
    )

    _remote_set_job(
        job_id,
        prompt_id=prompt_id,
    )

    deadline = (
        time.monotonic()
        + timeout_seconds
    )

    outputs = None

    while (
        time.monotonic()
        < deadline
    ):
        try:
            history = requests.get(
                BASE_URL
                + f"/history/{prompt_id}",
                timeout=30,
            ).json()

        except requests.RequestException:
            time.sleep(
                2
            )
            continue

        current_gpu = (
            _remote_gpu_memory_used_mib()
        )

        if (
            current_gpu
            is not None
        ):
            peak_gpu_memory_mib = max(
                peak_gpu_memory_mib
                or 0,
                current_gpu,
            )

        if (
            prompt_id
            in history
        ):
            run = (
                history[
                    prompt_id
                ]
            )

            status = run.get(
                "status",
                {},
            )

            if (
                status.get(
                    "status_str"
                )
                == "error"
            ):
                raise RuntimeError(
                    "ComfyUI 推論失敗：\n"
                    + json.dumps(
                        status,
                        ensure_ascii=False,
                        indent=2,
                    )
                )

            outputs = run.get(
                "outputs",
                {},
            )

            if (
                output_node_id
                in outputs
                or status.get(
                    "status_str"
                )
                == "success"
            ):
                break

        time.sleep(
            5
        )

    else:
        raise TimeoutError(
            "等待工作超過 "
            f"{timeout_seconds:.0f} 秒。"
        )

    candidates = (
        _remote_find_mp4_infos(
            (
                outputs.get(
                    output_node_id,
                    {},
                )
                if outputs
                else {}
            )
        )
    )

    video_path = None

    for info in candidates:

        p = (
            COMFY
            / "output"
            / info.get(
                "subfolder",
                "",
            )
            / info[
                "filename"
            ]
        )

        if (
            p.is_file()
            and p.stat().st_size > 0
        ):
            video_path = p
            break

    if (
        video_path
        is None
    ):
        recent = [
            p
            for p
            in (
                COMFY
                / "output"
            ).rglob(
                "*.mp4"
            )
            if (
                p.is_file()
                and p.stat().st_mtime
                >= started_wallclock - 2
            )
        ]

        if recent:
            video_path = max(
                recent,
                key=lambda p:
                    p.stat().st_mtime,
            )

    if (
        video_path
        is None
    ):
        raise RuntimeError(
            "任務完成，但找不到本次 MP4。"
        )

    probe = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "stream=codec_type,width,height",
            "-of",
            "json",
            str(
                video_path
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    probe_data = json.loads(
        probe.stdout
    )

    streams = {
        x.get(
            "codec_type"
        )
        for x
        in probe_data.get(
            "streams",
            [],
        )
    }

    if (
        "video"
        not in streams
        or (
            require_audio
            and "audio"
            not in streams
        )
    ):
        raise RuntimeError(
            "輸出 MP4 stream 不完整；目前："
            f"{sorted(streams)}"
        )

    vstream = next(
        (
            x
            for x
            in probe_data.get(
                "streams",
                [],
            )
            if (
                x.get(
                    "codec_type"
                )
                == "video"
            )
        ),
        {},
    )

    return {
        "video_path":
            video_path,

        "prompt_id":
            prompt_id,

        "wait_seconds":
            (
                time.monotonic()
                - started_monotonic
            ),

        "peak_gpu_memory_mib":
            peak_gpu_memory_mib,

        "output_width":
            vstream.get(
                "width"
            ),

        "output_height":
            vstream.get(
                "height"
            ),

        "streams":
            sorted(
                streams
            ),
    }


# ============================================================
# TXT
# ============================================================

def _remote_write_generation_info(
    job_id,
    cfg,
    run,
):
    info_path = (
        REMOTE_OUTPUT_DIR
        / f"{job_id}.txt"
    )

    info_lines = [
        "============================================================",
        "MiniMax H3 Generation Information",
        "============================================================",
        "",
        f"job_id={job_id}",
        f"prompt_id={run['prompt_id']}",
        f"seed={cfg['seed']}",
        f"inference_mode={cfg['mode']}",
        f"diffusion={cfg['model']}",
        (
            "lora="
            f"{cfg['lora'] or 'N/A'}"
        ),
        (
            "lora_strength="
            f"{cfg['lora_strength'] if cfg['lora'] else 'N/A'}"
        ),

        # 新增
        (
            "h3_shift_enabled="
            f"{cfg['h3_shift_enabled']}"
        ),
        (
            "h3_shift_video="
            f"{cfg['h3_shift_video'] if cfg['h3_shift_enabled'] else 'N/A'}"
        ),
        (
            "h3_shift_audio="
            f"{cfg['h3_shift_audio'] if cfg['h3_shift_enabled'] else 'N/A'}"
        ),

        f"text_encoder={TEXT_ENCODER}",
        f"video_vae={VIDEO_VAE}",
        f"audio_vae={AUDIO_VAE}",
        (
            "generation_width="
            f"{cfg['width']}"
        ),
        (
            "generation_height="
            f"{cfg['height']}"
        ),
        (
            "output_width="
            f"{run.get('output_width')}"
        ),
        (
            "output_height="
            f"{run.get('output_height')}"
        ),
        f"frames={cfg['frames']}",
        (
            "requested_duration_seconds="
            f"{cfg['duration_seconds']}"
        ),
        (
            "actual_duration_seconds="
            f"{cfg['frames'] / 24}"
        ),
        "fps=24",
        f"steps={cfg['steps']}",
        f"sampler={cfg['sampler']}",
        f"scheduler={cfg['scheduler']}",
        (
            "generation_wait_seconds="
            f"{run['wait_seconds']:.3f}"
        ),
        (
            "peak_gpu_memory_used_mib_observed="
            f"{run['peak_gpu_memory_mib'] if run['peak_gpu_memory_mib'] is not None else 'N/A'}"
        ),
        f"upscale_mode={cfg['upscale_mode']}",
    ]

    if cfg[
        "mode"
    ].startswith(
        "fast_"
    ):
        info_lines += [
            (
                "fast_use_native_vsa="
                f"{cfg['fast_use_vsa']}"
            ),
            (
                "fast_shift_video="
                f"{cfg['fast_shift_video']}"
            ),
            (
                "fast_shift_audio="
                f"{cfg['fast_shift_audio']}"
            ),
        ]

    if (
        cfg[
            "upscale_mode"
        ]
        == "inline"
    ):
        v = (
            cfg[
                "vosr"
            ]
        )

        info_lines += [
            (
                "vosr_model_bundle="
                f"{v['model_bundle']}"
            ),
            (
                "vosr_dtype="
                f"{v['dtype']}"
            ),
            (
                "vosr_seed="
                f"{v['seed']}"
            ),
            (
                "vosr_color_alignment="
                f"{v['color_alignment']}"
            ),
            (
                "vosr_target_width="
                f"{v['target_width']}"
            ),
            (
                "vosr_target_height="
                f"{v['target_height']}"
            ),
            (
                "vosr_fit_side="
                f"{v['fit_side']}"
            ),
            (
                "vosr_internal_upscale="
                f"{v['upscale_factor']}"
            ),
            (
                "vosr_final_width="
                f"{v['final_width']}"
            ),
            (
                "vosr_final_height="
                f"{v['final_height']}"
            ),
            (
                "vosr_tile_size="
                f"{v['tile_size']}"
            ),
            (
                "vosr_tile_overlap="
                f"{v['tile_overlap']}"
            ),
            (
                "vosr_vae_tile_size="
                f"{v['vae_tile_size']}"
            ),
            (
                "vosr_vae_tile_overlap="
                f"{v['vae_tile_overlap']}"
            ),
            "inline_original_mp4_saved=False",
        ]

    media = cfg.get(
        "input_media",
        {},
    )

    pictures = media.get(
        "pictures",
        [],
    )

    videos = media.get(
        "videos",
        [],
    )

    audios = media.get(
        "audios",
        [],
    )

    if cfg[
        "mode"
    ].startswith(
        "ref2v_"
    ):
        info_lines += [
            (
                "reference_image_size="
                f"{cfg['ref_image_size']}"
            ),
            (
                "reference_picture_count="
                f"{len(pictures)}"
            ),
            (
                "reference_video_count="
                f"{len(videos)}"
            ),
            (
                "reference_audio_count="
                f"{len(audios)}"
            ),
            "",
            "============================================================",
            "Reference Media Mapping",
            "============================================================",
            "",
        ]

        for (
            index,
            item,
        ) in enumerate(
            pictures,
            start=1,
        ):
            info_lines.append(
                (
                    f"<Picture {index}>="
                    f"{item['original_name']} "
                    f"-> {item['filename']}"
                )
            )

        for (
            index,
            item,
        ) in enumerate(
            videos,
            start=1,
        ):
            info_lines.append(
                (
                    f"<Video {index}>="
                    f"{item['original_name']} "
                    f"-> {item['filename']}"
                )
            )

        for (
            index,
            item,
        ) in enumerate(
            audios,
            start=1,
        ):
            info_lines.append(
                (
                    f"<Audio {index}>="
                    f"{item['original_name']} "
                    f"-> {item['filename']}"
                )
            )

    elif cfg[
        "mode"
    ] in {
        "i2v_native",
        "fast_i2v",
    }:
        info_lines += [
            "",
            "============================================================",
            "Input Frames",
            "============================================================",
            "",
            (
                "first_frame="
                f"{pictures[0]['original_name'] if len(pictures) >= 1 else 'N/A'}"
            ),
            (
                "last_frame="
                f"{pictures[1]['original_name'] if len(pictures) >= 2 else 'N/A'}"
            ),
        ]

    info_lines += [
        "",
        "============================================================",
        "PROMPT BEGIN",
        "============================================================",
        "",
        cfg[
            "prompt"
        ],
        "",
        "============================================================",
        "PROMPT END",
        "============================================================",
        "",
    ]

    info_path.write_text(
        "\n".join(
            info_lines
        ),
        encoding="utf-8",
    )

    return info_path


def _remote_write_upscale_info(
    job_id,
    cfg,
    run,
):
    v = (
        cfg[
            "vosr"
        ]
    )

    info_path = (
        REMOTE_OUTPUT_DIR
        / f"{job_id}.txt"
    )

    lines = [
        "============================================================",
        "VOSR UPSCALE INFO",
        "============================================================",
        "",
        f"job_id={job_id}",
        f"prompt_id={run['prompt_id']}",
        (
            "source_file="
            f"{cfg['source_original_name']}"
        ),
        (
            "source_width="
            f"{cfg['source_width']}"
        ),
        (
            "source_height="
            f"{cfg['source_height']}"
        ),
        (
            "source_fps="
            f"{cfg['source_fps']}"
        ),
        (
            "source_has_audio="
            f"{cfg['source_has_audio']}"
        ),
        (
            "vosr_model_bundle="
            f"{v['model_bundle']}"
        ),
        (
            "vosr_dtype="
            f"{v['dtype']}"
        ),
        (
            "vosr_seed="
            f"{v['seed']}"
        ),
        (
            "vosr_color_alignment="
            f"{v['color_alignment']}"
        ),
        (
            "target_width="
            f"{v['target_width']}"
        ),
        (
            "target_height="
            f"{v['target_height']}"
        ),
        (
            "fit_side="
            f"{v['fit_side']}"
        ),
        (
            "internal_upscale="
            f"{v['upscale_factor']}"
        ),
        (
            "final_width="
            f"{v['final_width']}"
        ),
        (
            "final_height="
            f"{v['final_height']}"
        ),
        (
            "actual_output_width="
            f"{run.get('output_width')}"
        ),
        (
            "actual_output_height="
            f"{run.get('output_height')}"
        ),
        (
            "tile_size="
            f"{v['tile_size']}"
        ),
        (
            "tile_overlap="
            f"{v['tile_overlap']}"
        ),
        (
            "vae_tile_size="
            f"{v['vae_tile_size']}"
        ),
        (
            "vae_tile_overlap="
            f"{v['vae_tile_overlap']}"
        ),
        (
            "upscale_wait_seconds="
            f"{run['wait_seconds']:.3f}"
        ),
        (
            "peak_gpu_memory_used_mib_observed="
            f"{run['peak_gpu_memory_mib'] if run['peak_gpu_memory_mib'] is not None else 'N/A'}"
        ),
        "",
    ]

    info_path.write_text(
        "\n".join(
            lines
        ),
        encoding="utf-8",
    )

    return info_path


# ============================================================
# Run generation / upscale
# ============================================================

def _remote_run_generation(
    job_id,
    cfg,
    media_files,
):
    graph, output_node_id = (
        _remote_build_graph(
            cfg,
            media_files.get(
                "images",
                [],
            ),
            media_files.get(
                "videos",
                [],
            ),
            media_files.get(
                "audios",
                [],
            ),
        )
    )

    run = _remote_execute_graph(
        job_id,
        graph,
        output_node_id,
        cfg[
            "timeout_seconds"
        ],
        require_audio=True,
    )

    final_path = (
        REMOTE_OUTPUT_DIR
        / f"{job_id}.mp4"
    )

    shutil.copy2(
        run[
            "video_path"
        ],
        final_path,
    )

    info_path = (
        _remote_write_generation_info(
            job_id,
            cfg,
            run,
        )
    )

    return (
        final_path,
        info_path,
        run,
    )


def _remote_run_upscale(
    job_id,
    cfg,
    video_file,
):
    graph, output_node_id = (
        _remote_build_upscale_graph(
            cfg,
            video_file,
        )
    )

    run = _remote_execute_graph(
        job_id,
        graph,
        output_node_id,
        cfg[
            "timeout_seconds"
        ],
        require_audio=bool(
            cfg[
                "source_has_audio"
            ]
        ),
    )

    final_path = (
        REMOTE_OUTPUT_DIR
        / f"{job_id}.mp4"
    )

    shutil.copy2(
        run[
            "video_path"
        ],
        final_path,
    )

    info_path = (
        _remote_write_upscale_info(
            job_id,
            cfg,
            run,
        )
    )

    return (
        final_path,
        info_path,
        run,
    )


# ============================================================
# Worker
# ============================================================

def _remote_worker():

    while True:

        (
            task_type,
            job_id,
            cfg,
            payload,
        ) = REMOTE_QUEUE.get()

        try:
            _remote_set_job(
                job_id,
                status="processing",
                started_at=time.time(),
            )

            if (
                task_type
                == "generate"
            ):
                (
                    result_path,
                    info_path,
                    run,
                ) = _remote_run_generation(
                    job_id,
                    cfg,
                    payload,
                )

            elif (
                task_type
                == "upscale"
            ):
                (
                    result_path,
                    info_path,
                    run,
                ) = _remote_run_upscale(
                    job_id,
                    cfg,
                    payload,
                )

            else:
                raise ValueError(
                    "Unknown task type: "
                    f"{task_type}"
                )

            _remote_set_job(
                job_id,
                status="completed",
                finished_at=time.time(),
                result_path=str(
                    result_path
                ),
                info_path=str(
                    info_path
                ),
                bytes=result_path.stat().st_size,
                info_bytes=info_path.stat().st_size,
                output_width=run.get(
                    "output_width"
                ),
                output_height=run.get(
                    "output_height"
                ),
            )

        except Exception as exc:

            _remote_set_job(
                job_id,
                status="failed",
                finished_at=time.time(),
                error=str(
                    exc
                ),
            )

        finally:

            cleanup = cfg.get(
                "cleanup_dir"
            )

            if cleanup:
                shutil.rmtree(
                    cleanup,
                    ignore_errors=True,
                )

            REMOTE_QUEUE.task_done()


REMOTE_WORKER_THREAD = (
    threading.Thread(
        target=_remote_worker,
        daemon=True,
    )
)

REMOTE_WORKER_THREAD.start()


# ============================================================
# FastAPI
# ============================================================

app = FastAPI(
    title=(
        "MiniMax H3 "
        "Colab Remote API"
    ),
    version="1.4",
)


@app.get(
    "/health"
)
def remote_health(
    authorization: Optional[str]
    = Header(None),
):
    _remote_require_auth(
        authorization
    )

    requests.get(
        BASE_URL
        + "/system_stats",
        timeout=5,
    ).raise_for_status()

    return {
        "ok":
            True,

        "api_version":
            "1.4",

        "gpu":
            torch.cuda.get_device_name(
                0
            ),

        "current_mode":
            globals().get(
                "INFERENCE_MODE"
            ),

        "current_model":
            globals().get(
                "DIFFUSION"
            ),

        "current_lora":
            globals().get(
                "LORA"
            ),

        "current_h3_shift":
            {
                "enabled":
                    bool(
                        globals().get(
                            "H3_SHIFT_ENABLED",
                            False,
                        )
                    ),

                "video":
                    globals().get(
                        "H3_SHIFT_VIDEO",
                        6.0,
                    ),

                "audio":
                    globals().get(
                        "H3_SHIFT_AUDIO",
                        3.0,
                    ),
            },

        "current_vosr_mode":
            globals().get(
                "VOSR_MODE",
                "none",
            ),

        "queue_size":
            REMOTE_QUEUE.qsize(),

        "info_txt":
            True,

        "vosr2":
            True,

        "existing_video_upscale":
            True,

        "ref2v_multimedia":
            True,

        "ref2v_limits":
            {
                "pictures":
                    9,

                "videos":
                    3,

                "audios":
                    3,
            },
    }


@app.get(
    "/models"
)
def remote_models(
    authorization: Optional[str]
    = Header(None),
):
    _remote_require_auth(
        authorization
    )

    diffusion_dir = (
        COMFY
        / "models"
        / "diffusion_models"
    )

    lora_dir = (
        COMFY
        / "models"
        / "loras"
    )

    model_ext = {
        ".safetensors",
        ".ckpt",
        ".pt",
        ".pth",
        ".bin",
    }

    diffusion = sorted(
        p.name
        for p
        in diffusion_dir.iterdir()
        if (
            p.is_file()
            and p.suffix.lower()
            in model_ext
        )
    )

    loras = sorted(
        p.name
        for p
        in lora_dir.iterdir()
        if (
            p.is_file()
            and p.suffix.lower()
            in model_ext
        )
    )

    return {
        "diffusion_models":
            diffusion,

        "loras":
            loras,

        "official_repo":
            OFFICIAL_REPO,

        "official_diffusion_models":
            globals().get(
                "OFFICIAL_DIFFUSION_MODELS",
                [],
            ),

        "official_loras":
            globals().get(
                "OFFICIAL_LORAS",
                [],
            ),

        "tenstrip":
            {
                "repo":
                    TENSTRIP_REPO,

                "model":
                    TENSTRIP_MODEL,
            },

        "current":
            {
                "mode":
                    globals().get(
                        "INFERENCE_MODE"
                    ),

                "model":
                    globals().get(
                        "DIFFUSION"
                    ),

                "lora":
                    globals().get(
                        "LORA"
                    ),

                "lora_enabled":
                    bool(
                        globals().get(
                            "USE_LORA",
                            False,
                        )
                    ),

                "h3_shift_enabled":
                    bool(
                        globals().get(
                            "H3_SHIFT_ENABLED",
                            False,
                        )
                    ),

                "h3_shift_video":
                    globals().get(
                        "H3_SHIFT_VIDEO",
                        6.0,
                    ),

                "h3_shift_audio":
                    globals().get(
                        "H3_SHIFT_AUDIO",
                        3.0,
                    ),

                "vosr_mode":
                    globals().get(
                        "VOSR_MODE",
                        "none",
                    ),
            },
    }


# ============================================================
# Generate endpoint
# ============================================================

@app.post(
    "/generate"
)
async def remote_generate(
    prompt: str = Form(...),
    mode: str = Form("current"),
    duration_seconds: float = Form(10.0),

    width: Optional[int] = Form(None),
    height: Optional[int] = Form(None),

    seed: str = Form(""),

    steps: Optional[int] = Form(None),

    sampler: str = Form(""),
    scheduler: str = Form(""),

    model: str = Form("__CURRENT__"),
    lora: str = Form("__CURRENT__"),
    lora_strength: Optional[float] = Form(None),

    ref_image_size: str = Form(""),

    fast_use_vsa: Optional[bool] = Form(None),

    upscale_mode: str = Form("current"),

    target_width: Optional[int] = Form(None),
    target_height: Optional[int] = Form(None),

    fit_side: str = Form(""),

    vosr_dtype: str = Form(""),
    vosr_seed: Optional[int] = Form(None),
    vosr_color_alignment: str = Form(""),

    vosr_tile_size: Optional[int] = Form(None),
    vosr_tile_overlap: Optional[int] = Form(None),

    vosr_vae_tile_size: Optional[int] = Form(None),
    vosr_vae_tile_overlap: Optional[int] = Form(None),

    timeout_seconds: float = Form(10800.0),

    images: Optional[List[UploadFile]]
    = File(None),

    videos: Optional[List[UploadFile]]
    = File(None),

    audios: Optional[List[UploadFile]]
    = File(None),

    authorization: Optional[str]
    = Header(None),
):
    _remote_require_auth(
        authorization
    )

    prompt = str(
        prompt
        or ""
    ).strip()

    if not prompt:
        raise HTTPException(
            status_code=400,
            detail="prompt 不可空白。",
        )

    current_mode = globals().get(
        "INFERENCE_MODE"
    )

    requested_current = (
        mode
        == "current"
    )

    if requested_current:

        if not current_mode:
            raise HTTPException(
                status_code=400,
                detail=(
                    "notebook 尚未套用 Cell 2；"
                    "請先執行 Cell 2。"
                ),
            )

        mode = (
            current_mode
        )

    allowed_modes = {
        "ref2v_native",
        "i2v_native",
        "t2v_native",
        "fast_i2v",
        "fast_t2v",
    }

    if (
        mode
        not in allowed_modes
    ):
        raise HTTPException(
            status_code=400,
            detail=(
                "不支援的 mode："
                f"{mode}"
            ),
        )

    using_current_mode = (
        requested_current
        or mode == current_mode
    )

    images = list(
        images
        or []
    )

    videos = list(
        videos
        or []
    )

    audios = list(
        audios
        or []
    )

    # --------------------------------------------------------
    # Media validation
    # --------------------------------------------------------

    if mode.startswith(
        "ref2v_"
    ):

        if len(
            images
        ) > 9:
            raise HTTPException(
                status_code=400,
                detail=(
                    "Ref2V 最多 9 張 Picture。"
                ),
            )

        if len(
            videos
        ) > 3:
            raise HTTPException(
                status_code=400,
                detail=(
                    "Ref2V 最多 3 個 Video。"
                ),
            )

        if len(
            audios
        ) > 3:
            raise HTTPException(
                status_code=400,
                detail=(
                    "Ref2V 最多 3 個 Audio。"
                ),
            )

        if not (
            images
            or videos
            or audios
        ):
            raise HTTPException(
                status_code=400,
                detail=(
                    "Ref2V 至少需要 1 個 "
                    "Picture、Video 或 Audio 參照。"
                ),
            )

    elif mode in {
        "i2v_native",
        "fast_i2v",
    }:

        if not (
            1
            <= len(
                images
            )
            <= 2
        ):
            raise HTTPException(
                status_code=400,
                detail=(
                    "I2V 需要 1 張 first frame，"
                    "可選第 2 張 last frame。"
                ),
            )

        if (
            videos
            or audios
        ):
            raise HTTPException(
                status_code=400,
                detail=(
                    "I2V 不接受 Ref2V "
                    "--video / --audio 參照。"
                ),
            )

    else:

        if (
            images
            or videos
            or audios
        ):
            raise HTTPException(
                status_code=400,
                detail=(
                    "T2V 不需要 "
                    "Picture / Video / Audio 參照。"
                ),
            )

    # --------------------------------------------------------
    # Prompt tag validation
    # --------------------------------------------------------

    if mode.startswith(
        "ref2v_"
    ):

        for (
            tag_name,
            pattern,
            count,
        ) in [
            (
                "Picture",
                r"<Picture\s+(\d+)>",
                len(
                    images
                ),
            ),
            (
                "Video",
                r"<Video\s+(\d+)>",
                len(
                    videos
                ),
            ),
            (
                "Audio",
                r"<Audio\s+(\d+)>",
                len(
                    audios
                ),
            ),
        ]:

            invalid_tags = sorted(
                {
                    int(
                        n
                    )
                    for n
                    in re.findall(
                        pattern,
                        prompt,
                    )
                    if (
                        int(
                            n
                        )
                        < 1
                        or int(
                            n
                        )
                        > count
                    )
                }
            )

            if invalid_tags:
                raise HTTPException(
                    status_code=400,
                    detail=(
                        "Prompt 引用不存在的 "
                        f"{tag_name}："
                        f"{invalid_tags}；"
                        f"目前只有 {count} 個。"
                    ),
                )

    # --------------------------------------------------------
    # Runtime config
    # --------------------------------------------------------

    try:
        frames = _remote_frames(
            duration_seconds
        )

        width = int(
            (
                width
                if width
                is not None
                else globals().get(
                    "WIDTH",
                    1152,
                )
            )
        )

        height = int(
            (
                height
                if height
                is not None
                else globals().get(
                    "HEIGHT",
                    640,
                )
            )
        )

        if (
            width <= 0
            or height <= 0
            or width % 32
            or height % 32
        ):
            raise ValueError(
                "width/height 必須為正整數且為 32 的倍數。"
            )

        if str(
            seed
        ).strip():

            seed_value = int(
                seed
            )

            if not (
                0
                <= seed_value
                < 2**64
            ):
                raise ValueError(
                    "seed 必須介於 "
                    "0 到 2^64-1。"
                )

        else:
            seed_value = (
                secrets.randbits(
                    64
                )
            )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        if (
            model
            == "__CURRENT__"
        ):
            candidate = (
                globals().get(
                    "DIFFUSION"
                )
                if using_current_mode
                else _remote_preset_model(
                    mode
                )
            )

        else:
            candidate = model

        model_name = (
            _remote_clean_model_name(
                candidate,
                (
                    COMFY
                    / "models"
                    / "diffusion_models"
                ),
                "主模型",
            )
        )

        # ----------------------------------------------------
        # LoRA
        # ----------------------------------------------------

        if mode.startswith(
            "fast_"
        ):
            lora_name = None

        elif (
            lora
            == "NONE"
        ):
            lora_name = None

        elif (
            lora
            == "__CURRENT__"
        ):
            if (
                using_current_mode
                and globals().get(
                    "USE_LORA",
                    False,
                )
            ):
                lora_name = (
                    _remote_clean_model_name(
                        globals().get(
                            "LORA"
                        ),
                        (
                            COMFY
                            / "models"
                            / "loras"
                        ),
                        "LoRA",
                    )
                )

            else:
                lora_name = None

        else:
            lora_name = (
                _remote_clean_model_name(
                    lora,
                    (
                        COMFY
                        / "models"
                        / "loras"
                    ),
                    "LoRA",
                )
            )

        # ----------------------------------------------------
        # H3 shift
        #
        # 目前 h3_client 不必新增參數：
        # Remote job 直接沿用目前 Cell 2 的 shift。
        # ----------------------------------------------------

        remote_h3_shift_enabled = bool(
            lora_name
            and not mode.startswith(
                "fast_"
            )
            and globals().get(
                "H3_SHIFT_ENABLED",
                True,
            )
        )

        remote_h3_shift_video = float(
            globals().get(
                "H3_SHIFT_VIDEO",
                6.0,
            )
        )

        remote_h3_shift_audio = float(
            globals().get(
                "H3_SHIFT_AUDIO",
                3.0,
            )
        )

        if (
            remote_h3_shift_video <= 0
            or remote_h3_shift_audio <= 0
        ):
            raise ValueError(
                "MiniMax H3 shift 必須 > 0。"
            )

        # ----------------------------------------------------
        # VOSR mode
        # ----------------------------------------------------

        if (
            upscale_mode
            == "current"
        ):
            upscale_mode = (
                globals().get(
                    "VOSR_MODE",
                    "none",
                )
                if using_current_mode
                else "none"
            )

        if (
            upscale_mode
            not in {
                "none",
                "inline",
            }
        ):
            raise ValueError(
                "upscale_mode 必須是 "
                "current / none / inline。"
            )

        cfg = {
            "prompt":
                prompt,

            "mode":
                mode,

            "duration_seconds":
                float(
                    duration_seconds
                ),

            "frames":
                frames,

            "width":
                width,

            "height":
                height,

            "seed":
                seed_value,

            "steps":
                int(
                    (
                        steps
                        if steps
                        is not None
                        else (
                            globals().get(
                                "STEPS",
                                20,
                            )
                            if using_current_mode
                            else (
                                8
                                if mode.startswith(
                                    "fast_"
                                )
                                else 20
                            )
                        )
                    )
                ),

            "sampler":
                (
                    sampler
                    or (
                        globals().get(
                            "SAMPLER_NAME",
                            "res_multistep",
                        )
                        if using_current_mode
                        else "res_multistep"
                    )
                ).strip(),

            "scheduler":
                (
                    scheduler
                    or (
                        globals().get(
                            "SCHEDULER_NAME",
                            "simple",
                        )
                        if using_current_mode
                        else "simple"
                    )
                ).strip(),

            "model":
                model_name,

            "lora":
                lora_name,

            "lora_strength":
                float(
                    (
                        lora_strength
                        if lora_strength
                        is not None
                        else globals().get(
                            "LORA_STRENGTH",
                            1.0,
                        )
                    )
                ),

            # 新增
            "h3_shift_enabled":
                remote_h3_shift_enabled,

            "h3_shift_video":
                remote_h3_shift_video,

            "h3_shift_audio":
                remote_h3_shift_audio,

            "ref_image_size":
                (
                    ref_image_size
                    or globals().get(
                        "REFERENCE_IMAGE_SIZE",
                        "match",
                    )
                ).strip(),

            "fast_use_vsa":
                bool(
                    (
                        (
                            globals().get(
                                "FAST_USE_VSA",
                                True,
                            )
                            if using_current_mode
                            else True
                        )
                        if fast_use_vsa
                        is None
                        else fast_use_vsa
                    )
                ),

            "fast_shift_video":
                float(
                    globals().get(
                        "FAST_SHIFT_VIDEO",
                        10.0,
                    )
                ),

            "fast_shift_audio":
                float(
                    globals().get(
                        "FAST_SHIFT_AUDIO",
                        3.0,
                    )
                ),

            "timeout_seconds":
                float(
                    timeout_seconds
                ),

            "upscale_mode":
                upscale_mode,
        }

        if (
            cfg[
                "steps"
            ]
            <= 0
        ):
            raise ValueError(
                "steps 必須 > 0。"
            )

        if (
            not math.isfinite(
                cfg[
                    "timeout_seconds"
                ]
            )
            or cfg[
                "timeout_seconds"
            ]
            <= 0
        ):
            raise ValueError(
                "timeout_seconds 必須 > 0。"
            )

        if (
            upscale_mode
            == "inline"
        ):
            tw = int(
                (
                    target_width
                    if target_width
                    is not None
                    else globals().get(
                        "VOSR_TARGET_WIDTH",
                        1920,
                    )
                )
            )

            th = int(
                (
                    target_height
                    if target_height
                    is not None
                    else globals().get(
                        "VOSR_TARGET_HEIGHT",
                        1080,
                    )
                )
            )

            fit = (
                fit_side
                or globals().get(
                    "VOSR_FIT_SIDE",
                    "long",
                )
            ).strip()

            cfg[
                "vosr"
            ] = _remote_vosr_cfg(
                source_width=width,
                source_height=height,
                target_width=tw,
                target_height=th,
                fit_side=fit,

                dtype=(
                    vosr_dtype
                    or globals().get(
                        "VOSR_DTYPE",
                        "bf16",
                    )
                ),

                seed=(
                    vosr_seed
                    if vosr_seed
                    is not None
                    else globals().get(
                        "VOSR_SEED",
                        42,
                    )
                ),

                color_alignment=(
                    vosr_color_alignment
                    or globals().get(
                        "VOSR_COLOR_ALIGNMENT",
                        "wavelet",
                    )
                ),

                tile_size=(
                    vosr_tile_size
                    if vosr_tile_size
                    is not None
                    else globals().get(
                        "VOSR_TILE_SIZE",
                        512,
                    )
                ),

                tile_overlap=(
                    vosr_tile_overlap
                    if vosr_tile_overlap
                    is not None
                    else globals().get(
                        "VOSR_TILE_OVERLAP",
                        64,
                    )
                ),

                vae_tile_size=(
                    vosr_vae_tile_size
                    if vosr_vae_tile_size
                    is not None
                    else globals().get(
                        "VOSR_VAE_TILE_SIZE",
                        1024,
                    )
                ),

                vae_tile_overlap=(
                    vosr_vae_tile_overlap
                    if vosr_vae_tile_overlap
                    is not None
                    else globals().get(
                        "VOSR_VAE_TILE_OVERLAP",
                        128,
                    )
                ),
            )

    except Exception as exc:
        raise HTTPException(
            status_code=400,
            detail=str(
                exc
            ),
        )

    # --------------------------------------------------------
    # Job upload directory
    # --------------------------------------------------------

    job_id = (
        uuid.uuid4().hex
    )

    job_input_dir = (
        COMFY
        / "input"
        / f"remote_{job_id}"
    )

    job_input_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    image_files = []
    video_files = []
    audio_files = []

    picture_media = []
    video_media = []
    audio_media = []

    try:
        # Picture
        for (
            index,
            upload,
        ) in enumerate(
            images,
            start=1,
        ):
            raw = (
                await upload.read()
            )

            if not raw:
                raise ValueError(
                    f"Picture {index} 是空檔。"
                )

            if (
                len(
                    raw
                )
                > REMOTE_MAX_IMAGE_BYTES
            ):
                raise ValueError(
                    f"Picture {index} "
                    "超過 50 MiB 上限。"
                )

            target = (
                job_input_dir
                / f"picture_{index:02d}.png"
            )

            with Image.open(
                BytesIO(
                    raw
                )
            ) as im:
                im.convert(
                    "RGB"
                ).save(
                    target,
                    format="PNG",
                )

            relative_name = str(
                target.relative_to(
                    COMFY
                    / "input"
                )
            ).replace(
                "\\",
                "/",
            )

            image_files.append(
                relative_name
            )

            picture_media.append(
                {
                    "original_name":
                        (
                            upload.filename
                            or f"picture_{index}"
                        ),

                    "filename":
                        relative_name,
                }
            )

        # Video
        for (
            index,
            upload,
        ) in enumerate(
            videos,
            start=1,
        ):
            suffix = (
                Path(
                    upload.filename
                    or ""
                )
                .suffix
                .lower()
                or ".mp4"
            )

            target = (
                job_input_dir
                / f"video_{index:02d}{suffix}"
            )

            _remote_save_upload_stream(
                upload,
                target,
                REMOTE_MAX_VIDEO_BYTES,
                f"Video {index}",
            )

            if not _remote_probe_has_stream(
                target,
                "video",
            ):
                raise ValueError(
                    f"Video {index} "
                    "沒有 video stream："
                    f"{upload.filename}"
                )

            relative_name = str(
                target.relative_to(
                    COMFY
                    / "input"
                )
            ).replace(
                "\\",
                "/",
            )

            video_files.append(
                relative_name
            )

            video_media.append(
                {
                    "original_name":
                        (
                            upload.filename
                            or f"video_{index}"
                        ),

                    "filename":
                        relative_name,
                }
            )

        # Audio
        for (
            index,
            upload,
        ) in enumerate(
            audios,
            start=1,
        ):
            suffix = (
                Path(
                    upload.filename
                    or ""
                )
                .suffix
                .lower()
                or ".wav"
            )

            target = (
                job_input_dir
                / f"audio_{index:02d}{suffix}"
            )

            _remote_save_upload_stream(
                upload,
                target,
                REMOTE_MAX_AUDIO_BYTES,
                f"Audio {index}",
            )

            if not _remote_probe_has_stream(
                target,
                "audio",
            ):
                raise ValueError(
                    f"Audio {index} "
                    "沒有 audio stream："
                    f"{upload.filename}"
                )

            relative_name = str(
                target.relative_to(
                    COMFY
                    / "input"
                )
            ).replace(
                "\\",
                "/",
            )

            audio_files.append(
                relative_name
            )

            audio_media.append(
                {
                    "original_name":
                        (
                            upload.filename
                            or f"audio_{index}"
                        ),

                    "filename":
                        relative_name,
                }
            )

    except Exception as exc:

        shutil.rmtree(
            job_input_dir,
            ignore_errors=True,
        )

        raise HTTPException(
            status_code=400,
            detail=(
                "參照素材處理失敗："
                f"{exc}"
            ),
        )

    cfg[
        "output_prefix"
    ] = (
        "video/"
        f"H3Remote_{job_id[:16]}"
    )

    cfg[
        "input_media"
    ] = {
        "pictures":
            picture_media,

        "videos":
            video_media,

        "audios":
            audio_media,
    }

    cfg[
        "cleanup_dir"
    ] = str(
        job_input_dir
    )

    final_w = (
        cfg.get(
            "vosr",
            {},
        ).get(
            "final_width",
            width,
        )
    )

    final_h = (
        cfg.get(
            "vosr",
            {},
        ).get(
            "final_height",
            height,
        )
    )

    _remote_set_job(
        job_id,
        task="generate",
        status="queued",
        created_at=time.time(),
        mode=mode,
        model=model_name,
        lora=lora_name,

        h3_shift_enabled=(
            remote_h3_shift_enabled
        ),

        h3_shift_video=(
            remote_h3_shift_video
            if remote_h3_shift_enabled
            else None
        ),

        h3_shift_audio=(
            remote_h3_shift_audio
            if remote_h3_shift_enabled
            else None
        ),

        seed=seed_value,

        duration_seconds=float(
            duration_seconds
        ),

        width=width,
        height=height,
        frames=frames,

        picture_count=len(
            image_files
        ),

        video_reference_count=len(
            video_files
        ),

        audio_reference_count=len(
            audio_files
        ),

        upscale_mode=upscale_mode,

        final_width=final_w,
        final_height=final_h,
    )

    REMOTE_QUEUE.put(
        (
            "generate",
            job_id,
            cfg,
            {
                "images":
                    image_files,

                "videos":
                    video_files,

                "audios":
                    audio_files,
            },
        )
    )

    return {
        "job_id":
            job_id,

        "status":
            "queued",

        "seed":
            seed_value,

        "h3_shift_enabled":
            remote_h3_shift_enabled,

        "h3_shift_video":
            (
                remote_h3_shift_video
                if remote_h3_shift_enabled
                else None
            ),

        "h3_shift_audio":
            (
                remote_h3_shift_audio
                if remote_h3_shift_enabled
                else None
            ),

        "picture_count":
            len(
                image_files
            ),

        "video_reference_count":
            len(
                video_files
            ),

        "audio_reference_count":
            len(
                audio_files
            ),

        "upscale_mode":
            upscale_mode,

        "final_width":
            final_w,

        "final_height":
            final_h,

        "poll_url":
            f"/jobs/{job_id}",

        "result_url":
            f"/result/{job_id}",

        "info_url":
            f"/info/{job_id}",
    }


# ============================================================
# Upscale endpoint
# ============================================================

@app.post(
    "/upscale"
)
async def remote_upscale_video(
    video: UploadFile = File(...),

    target_width: int = Form(1920),
    target_height: int = Form(1080),

    fit_side: str = Form("long"),

    vosr_dtype: str = Form("bf16"),
    vosr_seed: int = Form(42),

    vosr_color_alignment: str = Form("wavelet"),

    vosr_tile_size: int = Form(512),
    vosr_tile_overlap: int = Form(64),

    vosr_vae_tile_size: int = Form(1024),
    vosr_vae_tile_overlap: int = Form(128),

    timeout_seconds: float = Form(10800.0),

    authorization: Optional[str]
    = Header(None),
):
    _remote_require_auth(
        authorization
    )

    job_id = (
        uuid.uuid4().hex
    )

    job_input_dir = (
        COMFY
        / "input"
        / f"remote_upscale_{job_id}"
    )

    job_input_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    suffix = (
        Path(
            video.filename
            or "input.mp4"
        )
        .suffix
        .lower()
        or ".mp4"
    )

    target = (
        job_input_dir
        / f"input{suffix}"
    )

    try:
        with target.open(
            "wb"
        ) as f:
            shutil.copyfileobj(
                video.file,
                f,
                length=1024 * 1024,
            )

        if (
            not target.is_file()
            or target.stat().st_size <= 0
        ):
            raise ValueError(
                "上傳影片是空檔。"
            )

        (
            source_w,
            source_h,
            source_fps,
            source_has_audio,
        ) = _remote_probe_video(
            target
        )

        vosr = _remote_vosr_cfg(
            source_width=source_w,
            source_height=source_h,
            target_width=target_width,
            target_height=target_height,
            fit_side=fit_side,
            dtype=vosr_dtype,
            seed=vosr_seed,
            color_alignment=vosr_color_alignment,
            tile_size=vosr_tile_size,
            tile_overlap=vosr_tile_overlap,
            vae_tile_size=vosr_vae_tile_size,
            vae_tile_overlap=vosr_vae_tile_overlap,
        )

        if (
            not math.isfinite(
                float(
                    timeout_seconds
                )
            )
            or float(
                timeout_seconds
            )
            <= 0
        ):
            raise ValueError(
                "timeout_seconds 必須 > 0。"
            )

    except Exception as exc:

        shutil.rmtree(
            job_input_dir,
            ignore_errors=True,
        )

        raise HTTPException(
            status_code=400,
            detail=(
                "影片處理失敗："
                f"{exc}"
            ),
        )

    video_rel = str(
        target.relative_to(
            COMFY
            / "input"
        )
    ).replace(
        "\\",
        "/",
    )

    cfg = {
        "source_original_name":
            (
                video.filename
                or "input.mp4"
            ),

        "source_width":
            source_w,

        "source_height":
            source_h,

        "source_fps":
            source_fps,

        "source_has_audio":
            source_has_audio,

        "vosr":
            vosr,

        "timeout_seconds":
            float(
                timeout_seconds
            ),

        "output_prefix":
            (
                "video/"
                f"VOSRRemote_{job_id[:16]}"
            ),

        "cleanup_dir":
            str(
                job_input_dir
            ),
    }

    _remote_set_job(
        job_id,
        task="upscale",
        status="queued",
        created_at=time.time(),
        source_file=cfg[
            "source_original_name"
        ],
        source_width=source_w,
        source_height=source_h,
        seed=vosr[
            "seed"
        ],
        vosr_seed=vosr[
            "seed"
        ],
        target_width=target_width,
        target_height=target_height,
        final_width=vosr[
            "final_width"
        ],
        final_height=vosr[
            "final_height"
        ],
        fit_side=fit_side,
    )

    REMOTE_QUEUE.put(
        (
            "upscale",
            job_id,
            cfg,
            video_rel,
        )
    )

    return {
        "job_id":
            job_id,

        "status":
            "queued",

        "seed":
            vosr[
                "seed"
            ],

        "final_width":
            vosr[
                "final_width"
            ],

        "final_height":
            vosr[
                "final_height"
            ],

        "poll_url":
            f"/jobs/{job_id}",

        "result_url":
            f"/result/{job_id}",

        "info_url":
            f"/info/{job_id}",
    }


# ============================================================
# Status / result
# ============================================================

@app.get(
    "/jobs/{job_id}"
)
def remote_job_status(
    job_id: str,

    authorization: Optional[str]
    = Header(None),
):
    _remote_require_auth(
        authorization
    )

    item = _remote_get_job(
        job_id
    )

    if item is None:
        raise HTTPException(
            status_code=404,
            detail="Unknown job",
        )

    item.pop(
        "result_path",
        None,
    )

    item.pop(
        "info_path",
        None,
    )

    return item


@app.get(
    "/result/{job_id}"
)
def remote_job_result(
    job_id: str,

    authorization: Optional[str]
    = Header(None),
):
    _remote_require_auth(
        authorization
    )

    item = _remote_get_job(
        job_id
    )

    if item is None:
        raise HTTPException(
            status_code=404,
            detail="Unknown job",
        )

    if (
        item.get(
            "status"
        )
        != "completed"
    ):
        raise HTTPException(
            status_code=409,
            detail=(
                "Job status: "
                f"{item.get('status')}"
            ),
        )

    path = Path(
        item[
            "result_path"
        ]
    )

    if not path.is_file():
        raise HTTPException(
            status_code=410,
            detail=(
                "Result file is no longer available"
            ),
        )

    return FileResponse(
        path,
        media_type="video/mp4",
        filename=(
            f"h3_{job_id[:12]}.mp4"
        ),
    )


@app.get(
    "/info/{job_id}"
)
def remote_job_info(
    job_id: str,

    authorization: Optional[str]
    = Header(None),
):
    _remote_require_auth(
        authorization
    )

    item = _remote_get_job(
        job_id
    )

    if item is None:
        raise HTTPException(
            status_code=404,
            detail="Unknown job",
        )

    if (
        item.get(
            "status"
        )
        != "completed"
    ):
        raise HTTPException(
            status_code=409,
            detail=(
                "Job status: "
                f"{item.get('status')}"
            ),
        )

    path = Path(
        item[
            "info_path"
        ]
    )

    if not path.is_file():
        raise HTTPException(
            status_code=410,
            detail=(
                "Info TXT is no longer available"
            ),
        )

    return FileResponse(
        path,
        media_type=(
            "text/plain; charset=utf-8"
        ),
        filename=(
            f"h3_{job_id[:12]}.txt"
        ),
    )


# ============================================================
# Restart old server / tunnel
# ============================================================

if (
    globals().get(
        "REMOTE_UVICORN_SERVER"
    )
    is not None
):
    try:
        REMOTE_UVICORN_SERVER.should_exit = (
            True
        )

    except Exception:
        pass


if (
    globals().get(
        "REMOTE_TUNNEL_PROCESS"
    )
    is not None
):
    try:
        if (
            REMOTE_TUNNEL_PROCESS.poll()
            is None
        ):
            REMOTE_TUNNEL_PROCESS.terminate()

    except Exception:
        pass


time.sleep(
    1
)


# ============================================================
# Start FastAPI
# ============================================================

REMOTE_UVICORN_CONFIG = uvicorn.Config(
    app,
    host=REMOTE_API_HOST,
    port=REMOTE_API_PORT,
    log_level="warning",
)


REMOTE_UVICORN_SERVER = uvicorn.Server(
    REMOTE_UVICORN_CONFIG
)


REMOTE_UVICORN_THREAD = (
    threading.Thread(
        target=REMOTE_UVICORN_SERVER.run,
        daemon=True,
    )
)


REMOTE_UVICORN_THREAD.start()


for _ in range(
    60
):
    try:
        r = requests.get(
            (
                f"http://"
                f"{REMOTE_API_HOST}:"
                f"{REMOTE_API_PORT}"
                "/health"
            ),
            headers={
                "Authorization":
                    f"Bearer {REMOTE_API_TOKEN}"
            },
            timeout=2,
        )

        if r.ok:
            break

    except requests.RequestException:
        pass

    time.sleep(
        1
    )


else:
    raise RuntimeError(
        "Remote API server 啟動失敗。"
    )


# ============================================================
# Cloudflare tunnel
# ============================================================

cloudflared = Path(
    "/content/cloudflared"
)


if not cloudflared.is_file():

    subprocess.run(
        [
            "curl",
            "-L",
            "--fail",
            "--retry",
            "3",
            (
                "https://github.com/"
                "cloudflare/cloudflared/"
                "releases/latest/download/"
                "cloudflared-linux-amd64"
            ),
            "-o",
            str(
                cloudflared
            ),
        ],
        check=True,
    )

    cloudflared.chmod(
        0o755
    )


REMOTE_TUNNEL_PROCESS = (
    subprocess.Popen(
        [
            str(
                cloudflared
            ),
            "tunnel",
            "--url",
            (
                f"http://"
                f"{REMOTE_API_HOST}:"
                f"{REMOTE_API_PORT}"
            ),
            "--no-autoupdate",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
)


REMOTE_API_PUBLIC_URL = None


url_re = re.compile(
    (
        r"https://"
        r"[a-z0-9-]+"
        r"\.trycloudflare\.com"
    )
)


deadline = (
    time.time()
    + 45
)


while (
    time.time()
    < deadline
):

    line = (
        REMOTE_TUNNEL_PROCESS
        .stdout
        .readline()
    )

    if not line:

        if (
            REMOTE_TUNNEL_PROCESS.poll()
            is not None
        ):
            break

        time.sleep(
            0.2
        )

        continue

    match = url_re.search(
        line
    )

    if match:
        REMOTE_API_PUBLIC_URL = (
            match.group(
                0
            )
        )

        break


if not REMOTE_API_PUBLIC_URL:
    raise RuntimeError(
        "Cloudflare Quick Tunnel 啟動失敗；"
        "請重新執行 Cell 10。"
    )


# ============================================================
# Show settings
# ============================================================

print(
    "\n"
    + "=" * 72
)

print(
    "MiniMax H3 Remote API v1.4 已啟動"
)

print(
    "=" * 72
)

print(
    "API URL:",
    REMOTE_API_PUBLIC_URL,
)

print(
    "API TOKEN:",
    REMOTE_API_TOKEN,
)


print(
    "\n目前 H3 shift:"
)

print(
    (
        "enabled"
        if globals().get(
            "H3_SHIFT_ENABLED",
            False,
        )
        else "disabled"
    ),
    "/ Video",
    globals().get(
        "H3_SHIFT_VIDEO",
        6.0,
    ),
    "/ Audio",
    globals().get(
        "H3_SHIFT_AUDIO",
        3.0,
    ),
)


print(
    "\nWindows PowerShell 請貼上："
)


print(
    (
        '$env:H3_API_URL="'
        f'{REMOTE_API_PUBLIC_URL}'
        '"'
    )
)


print(
    (
        '$env:H3_API_TOKEN="'
        f'{REMOTE_API_TOKEN}'
        '"'
    )
)


print(
    "\n可用能力："
    "Ref2V Picture+Video+Audio / "
    "ModelSamplingMiniMaxH3 shift / "
    "generate / "
    "inline VOSR2 / "
    "existing-video VOSR2 upscale / "
    "MP4+TXT"
)


print(
    "Windows 測試："
)

print(
    "python h3_client.py health"
)

print(
    "python h3_client.py models"
)

print(
    "=" * 72
)